# 3\.2\.4 Evaluate Candidate Representation Spaces

This notebook evaluates candidate representation spaces for downstream clustering and anomaly\-detection workflows\. The goal is not to generate clusters or anomaly labels, but rather to determine which representations preserve meaningful mobility\-state structure, produce useful neighborhood and density behavior, and remain practical for downstream unsupervised modeling\.

Representation learning is evaluated globally across the full Taxi Zone × Date × Temporal Bucket panel to preserve a shared mobility\-state coordinate system\. Downstream anomaly scoring may later be evaluated using both global and temporal\-bucket\-aware baselines because weekday AM peak and weekend overnight observations represent fundamentally different mobility regimes\. Representation selection and anomaly\-baseline selection are treated as separate decisions\.

In [1]:
!pip install umap

ERROR: Could not find a version that satisfies the requirement umap (from versions: none)
ERROR: No matching distribution found for umap

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# ---------------------------------------------------------------------
# Import libraries
# ---------------------------------------------------------------------

import os
import time
import warnings
from pathlib import Path

# Keep noisy optional backend warnings out of notebook output.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import duckdb
import numpy as np
import pandas as pd

import pickle
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.decomposition import PCA
from sklearn.manifold import trustworthiness
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors

from scipy.spatial.distance import pdist

import umap

import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

print("Libraries imported.")

/root/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Libraries imported.


In [3]:
# ---------------------------------------------------------------------
# Define project paths
# ---------------------------------------------------------------------

PROJECT_ROOT = Path(".").resolve()
PIPELINE_DATA_DIR = PROJECT_ROOT / "pipeline_data"

MATRIX_ASSET_DIR = PIPELINE_DATA_DIR / "3.1.1.final_tables"
PCA_ASSET_DIR = PIPELINE_DATA_DIR / "3.2.1.final_tables"
UMAP_ASSET_DIR = PIPELINE_DATA_DIR / "3.2.3.final_tables"

INTERMEDIATE_OUTPUT_DIR = PIPELINE_DATA_DIR / "3.2.4.intermediate_tables"
FINAL_OUTPUT_DIR = PIPELINE_DATA_DIR / "3.2.4.final_tables"

INTERMEDIATE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project paths defined.")

Project paths defined.


In [4]:
# ---------------------------------------------------------------------
# Define feature sets and representation policy
# ---------------------------------------------------------------------

MODEL_FEATURE_SET_NAMES = [
    "bus",
    "subway",
    "taxi",
    "fhvhv",
    "multimodal",
]

MISSING_INDICATOR_SUFFIX = "_missing_indicator"

REPRESENTATION_POLICY_BY_SET = {
    "bus": "full",
    "subway": "full",
    "taxi": "mobility_only",
    "fhvhv": "mobility_only",
    "multimodal": "mobility_only",
}

REPRESENTATION_FAMILIES = [
    "raw_scaled",
    "pca_80pct",
    "umap_raw_scaled",
    "umap_pca_80pct",
]

UMAP_INPUT_REPRESENTATIONS = [
    "raw_scaled",
    "pca_80pct",
]

print("Feature-set representation policy defined.")

Feature-set representation policy defined.


In [5]:
# ---------------------------------------------------------------------
# Define evaluation settings
# ---------------------------------------------------------------------

PCA_VARIANCE_THRESHOLD_FOR_UMAP_INPUT = 0.80

UMAP_RANDOM_STATE = 42
UMAP_METRIC = "euclidean"
UMAP_N_NEIGHBORS = 50
UMAP_MIN_DIST = 0.10

UMAP_N_COMPONENT_CANDIDATES = [2, 5, 10, 15]

REPRESENTATION_EVALUATION_SAMPLE_ROWS = 50_000
REPRESENTATION_EVALUATION_RANDOM_STATE = 42

NEIGHBOR_METRIC_SAMPLE_ROWS = 15_000
NEIGHBOR_EVALUATION_K = 15
NEIGHBOR_EVALUATION_K_SECONDARY = 50

PAIRWISE_DISTANCE_SAMPLE_ROWS = 5_000

print("Evaluation settings defined.")

Evaluation settings defined.


In [6]:
# ---------------------------------------------------------------------
# Define rebuild toggles
# ---------------------------------------------------------------------

REBUILD_EVALUATION_SAMPLES = False
REBUILD_UMAP_DIMENSIONALITY = False
REBUILD_RAW_TO_UMAP = False
REBUILD_PCA_TO_UMAP = False

WRITE_INTERMEDIATE_OUTPUTS = True
WRITE_FINAL_OUTPUTS = True

print("Rebuild toggles defined.")

Rebuild toggles defined.


In [7]:
# ---------------------------------------------------------------------
# Define dimensionality-metric rebuild toggles and output paths
# ---------------------------------------------------------------------

REBUILD_UMAP_TRUSTWORTHINESS = False
REBUILD_UMAP_NEIGHBOR_RETENTION = False
REBUILD_UMAP_DISTANCE_BEHAVIOR = False

UMAP_TRUSTWORTHINESS_OUTPUT_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_dimensionality_trustworthiness.parquet"
)
UMAP_NEIGHBOR_RETENTION_OUTPUT_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_dimensionality_neighbor_retention.parquet"
)
UMAP_DISTANCE_BEHAVIOR_OUTPUT_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_dimensionality_distance_behavior.parquet"
)

print("Dimensionality-metric toggles and output paths defined.")

Dimensionality-metric toggles and output paths defined.


In [8]:
# ---------------------------------------------------------------------
# Define asset path dictionaries
# ---------------------------------------------------------------------

SCALED_MATRIX_PATHS_BY_SET = {
    feature_set: MATRIX_ASSET_DIR / f"{feature_set}_scaled_modeling_matrix.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

RAW_MATRIX_PATHS_BY_SET = {
    feature_set: MATRIX_ASSET_DIR / f"{feature_set}_raw_modeling_matrix.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

ROW_METADATA_PATHS_BY_SET = {
    feature_set: MATRIX_ASSET_DIR / f"{feature_set}_row_metadata.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

UMAP_DIMENSIONALITY_PROGRESS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_dimensionality_fit_progress.parquet"
)

UMAP_TRUSTWORTHINESS_PROGRESS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_dimensionality_trustworthiness_progress.parquet"
)
UMAP_NEIGHBOR_RETENTION_PROGRESS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_dimensionality_neighbor_retention_progress.parquet"
)
UMAP_DISTANCE_BEHAVIOR_PROGRESS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_dimensionality_distance_behavior_progress.parquet"
)

FINAL_MODELING_ASSET_MANIFEST_PATH = (
    MATRIX_ASSET_DIR / "final_modeling_asset_manifest.csv"
)

SCALED_MATRIX_COLUMN_MANIFEST_PATH = (
    MATRIX_ASSET_DIR / "scaled_matrix_column_manifest.csv"
)

PCA_ASSET_MANIFEST_PATH = PCA_ASSET_DIR / "pca_asset_manifest.csv"
PCA_EXPLAINED_VARIANCE_PATH = PCA_ASSET_DIR / "pca_explained_variance.csv"
PCA_VARIANCE_THRESHOLD_SUMMARY_PATH = PCA_ASSET_DIR / "pca_variance_threshold_summary.csv"
PCA_LOADING_PATH = PCA_ASSET_DIR / "pca_loading_table.csv"

PCA_SCORE_PATHS_BY_SET = {
    feature_set: PCA_ASSET_DIR / f"{feature_set}_pca_scores.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

CANONICAL_UMAP_ASSET_MANIFEST_PATH = (
    UMAP_ASSET_DIR / "canonical_umap_asset_manifest.csv"
)

CANONICAL_UMAP_2D_PATHS_BY_SET = {
    feature_set: UMAP_ASSET_DIR / f"{feature_set}_canonical_umap_embedding.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

print("Asset path dictionaries defined.")

Asset path dictionaries defined.


In [9]:
# ---------------------------------------------------------------------
# Define shared helper functions
# ---------------------------------------------------------------------

def sql_path(path: Path) -> str:
    return str(path).replace("\\", "/")


def duckdb_identifier(column_name: str) -> str:
    return '"' + str(column_name).replace('"', '""') + '"'


def compact_path(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT)).replace("\\", "/")
    except ValueError:
        return str(path).replace("\\", "/")


def status_from_bool(value: bool) -> str:
    return "pass" if bool(value) else "review"


def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def read_parquet_schema(path: Path) -> pd.DataFrame:
    with duckdb.connect() as con:
        return con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{sql_path(path)}')"
        ).fetchdf()


def read_parquet_row_count(path: Path) -> int:
    with duckdb.connect() as con:
        return int(
            con.execute(
                f"SELECT COUNT(*) FROM read_parquet('{sql_path(path)}')"
            ).fetchone()[0]
        )


def get_parquet_columns(path: Path) -> list[str]:
    schema_df = read_parquet_schema(path)
    return schema_df["column_name"].tolist()


def modeling_feature_columns_from_schema(path: Path) -> list[str]:
    return [
        column
        for column in get_parquet_columns(path)
        if column != "modeling_row_id"
    ]


def mobility_only_columns(columns: list[str]) -> list[str]:
    return [
        column
        for column in columns
        if not column.endswith(MISSING_INDICATOR_SUFFIX)
        and column != "modeling_row_id"
    ]


def representation_columns_for_feature_set(feature_set: str) -> list[str]:
    scaled_columns = modeling_feature_columns_from_schema(
        SCALED_MATRIX_PATHS_BY_SET[feature_set]
    )

    if REPRESENTATION_POLICY_BY_SET[feature_set] == "mobility_only":
        return mobility_only_columns(scaled_columns)

    return scaled_columns


print("Shared helper functions defined.")

def require_columns(df, required_columns, df_name):
    missing_columns = [column for column in required_columns if column not in df.columns]
    assert not missing_columns, (
        f"{df_name} is missing required columns: {missing_columns}"
    )

Shared helper functions defined.


## 3\.2\.4\.1 Load Candidate Representation Assets

Notebook 3\.2\.4 starts from the representation assets already created upstream\. The raw and scaled modeling matrices come from 3\.1\.1, the PCA score spaces come from 3\.2\.1, and the 2D UMAP exports come from 3\.2\.3\. Those 2D UMAP files are useful as visualization baselines, while this notebook evaluates which representation spaces are most useful for downstream clustering and anomaly detection\.

First, confirm that the expected upstream assets exist\. This is a lightweight path check before we inspect row counts or columns\.

In [10]:
# ---------------------------------------------------------------------
# Check candidate representation asset paths
# ---------------------------------------------------------------------

candidate_asset_path_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    candidate_asset_path_records.append(
        {
            "feature_set": feature_set_name,
            "raw_matrix_exists": RAW_MATRIX_PATHS_BY_SET[feature_set_name].exists(),
            "scaled_matrix_exists": SCALED_MATRIX_PATHS_BY_SET[feature_set_name].exists(),
            "row_metadata_exists": ROW_METADATA_PATHS_BY_SET[feature_set_name].exists(),
            "pca_scores_exist": PCA_SCORE_PATHS_BY_SET[feature_set_name].exists(),
            "canonical_umap_2d_exists": CANONICAL_UMAP_2D_PATHS_BY_SET[feature_set_name].exists(),
        }
    )

candidate_asset_path_df = pd.DataFrame(candidate_asset_path_records)

candidate_asset_path_df["status"] = np.where(
    candidate_asset_path_df[
        [
            "raw_matrix_exists",
            "scaled_matrix_exists",
            "row_metadata_exists",
            "pca_scores_exist",
            "canonical_umap_2d_exists",
        ]
    ].all(axis=1),
    "pass",
    "review",
)

display(candidate_asset_path_df)

assert candidate_asset_path_df["status"].eq("pass").all(), (
    "One or more candidate representation assets is missing."
)

print("Candidate representation asset paths checked.")

,feature_set,raw_matrix_exists,scaled_matrix_exists,row_metadata_exists,pca_scores_exist,canonical_umap_2d_exists,status
0,bus,True,True,True,True,True,pass
1,subway,True,True,True,True,True,pass
2,taxi,True,True,True,True,True,pass
3,fhvhv,True,True,True,True,True,pass
4,multimodal,True,True,True,True,True,pass


Candidate representation asset paths checked.


Findings\. All required upstream assets are present for the five feature sets\. Each modality has its 3\.1\.1 raw matrix, 3\.1\.1 scaled matrix, row metadata, 3\.2\.1 PCA scores, and 3\.2\.3 canonical 2D UMAP export available\. This confirms the notebook can proceed from existing representation assets rather than rebuilding upstream matrices or embeddings\.

Next, validate the row\-level alignment of the 3\.1\.1 matrix assets\. The raw matrix, scaled matrix, and metadata table should have the same row count for each feature set and should all carry modeling\_row\_id\.

In [11]:
# ---------------------------------------------------------------------
# Validate 3.1.1 matrix and metadata alignment
# ---------------------------------------------------------------------

matrix_alignment_records = []

with duckdb.connect() as con:
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        raw_matrix_path = sql_path(RAW_MATRIX_PATHS_BY_SET[feature_set_name])
        scaled_matrix_path = sql_path(SCALED_MATRIX_PATHS_BY_SET[feature_set_name])
        metadata_path = sql_path(ROW_METADATA_PATHS_BY_SET[feature_set_name])

        raw_schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{raw_matrix_path}')"
        ).fetchdf()
        scaled_schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{scaled_matrix_path}')"
        ).fetchdf()
        metadata_schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{metadata_path}')"
        ).fetchdf()

        raw_rows = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{raw_matrix_path}')"
        ).fetchone()[0]
        scaled_rows = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{scaled_matrix_path}')"
        ).fetchone()[0]
        metadata_rows = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{metadata_path}')"
        ).fetchone()[0]

        raw_columns = raw_schema_df["column_name"].tolist()
        scaled_columns = scaled_schema_df["column_name"].tolist()
        metadata_columns = metadata_schema_df["column_name"].tolist()

        matrix_alignment_records.append(
            {
                "feature_set": feature_set_name,
                "raw_rows": int(raw_rows),
                "scaled_rows": int(scaled_rows),
                "metadata_rows": int(metadata_rows),
                "raw_model_columns": len([col for col in raw_columns if col != "modeling_row_id"]),
                "scaled_model_columns": len([col for col in scaled_columns if col != "modeling_row_id"]),
                "raw_has_modeling_row_id": "modeling_row_id" in raw_columns,
                "scaled_has_modeling_row_id": "modeling_row_id" in scaled_columns,
                "metadata_has_modeling_row_id": "modeling_row_id" in metadata_columns,
            }
        )

matrix_alignment_df = pd.DataFrame(matrix_alignment_records)

matrix_alignment_df["status"] = np.where(
    (
        matrix_alignment_df["raw_rows"].eq(matrix_alignment_df["scaled_rows"])
        & matrix_alignment_df["scaled_rows"].eq(matrix_alignment_df["metadata_rows"])
        & matrix_alignment_df["raw_has_modeling_row_id"]
        & matrix_alignment_df["scaled_has_modeling_row_id"]
        & matrix_alignment_df["metadata_has_modeling_row_id"]
    ),
    "pass",
    "review",
)

display(matrix_alignment_df)

assert matrix_alignment_df["status"].eq("pass").all(), (
    "One or more 3.1.1 matrix assets failed row-alignment validation."
)

print("3.1.1 matrix and metadata assets validated.")

,feature_set,raw_rows,scaled_rows,metadata_rows,raw_model_columns,scaled_model_columns,raw_has_modeling_row_id,scaled_has_modeling_row_id,metadata_has_modeling_row_id,status
0,bus,1472947,1472947,1472947,40,40,True,True,True,pass
1,subway,911455,911455,911455,21,21,True,True,True,pass
2,taxi,1541800,1541800,1541800,53,53,True,True,True,pass
3,fhvhv,1541800,1541800,1541800,53,46,True,True,True,pass
4,multimodal,1541800,1541800,1541800,233,233,True,True,True,pass


3.1.1 matrix and metadata assets validated.


Findings\. The 3\.1\.1 matrix assets are aligned and ready for representation evaluation\. For each feature set, the raw matrix, scaled matrix, and row metadata table have the same row count, and all three carry modeling\_row\_id for stable joins\. Bus has 1,472,947 rows, Subway has 911,455 rows, and Taxi, FHVHV, and Multimodal each have 1,541,800 rows\. The raw/scaled column counts also match expectations: Bus, Subway, Taxi, and Multimodal have the same number of raw and scaled model columns, while FHVHV narrows from 53 raw columns to 46 scaled columns after the scaling\-stage exclusions already handled upstream\.

Now apply the representation policy from 3\.2\.1\. This defines which scaled columns are eligible for PCA\-to\-UMAP and UMAP comparison without reopening the missingness\-indicator decision\.

In [12]:
# ---------------------------------------------------------------------
# Apply representation policy to scaled matrix columns
# ---------------------------------------------------------------------

REPRESENTATION_COLUMNS_BY_SET = {}
EXCLUDED_INDICATOR_COLUMNS_BY_SET = {}

representation_policy_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    scaled_model_columns = modeling_feature_columns_from_schema(
        SCALED_MATRIX_PATHS_BY_SET[feature_set_name]
    )
    representation_columns = representation_columns_for_feature_set(feature_set_name)
    excluded_indicator_columns = [
        column
        for column in scaled_model_columns
        if column.endswith(MISSING_INDICATOR_SUFFIX)
        and column not in representation_columns
    ]

    REPRESENTATION_COLUMNS_BY_SET[feature_set_name] = representation_columns
    EXCLUDED_INDICATOR_COLUMNS_BY_SET[feature_set_name] = excluded_indicator_columns

    representation_policy_records.append(
        {
            "feature_set": feature_set_name,
            "representation_policy": REPRESENTATION_POLICY_BY_SET[feature_set_name],
            "scaled_model_columns": len(scaled_model_columns),
            "representation_columns": len(representation_columns),
            "excluded_indicator_columns": len(excluded_indicator_columns),
        }
    )

representation_policy_df = pd.DataFrame(representation_policy_records)

representation_policy_df["status"] = np.where(
    representation_policy_df["representation_columns"].gt(0),
    "pass",
    "review",
)

display(representation_policy_df)

assert representation_policy_df["status"].eq("pass").all(), (
    "One or more feature sets has no representation columns."
)

print("Representation policy applied.")

,feature_set,representation_policy,scaled_model_columns,representation_columns,excluded_indicator_columns,status
0,bus,full,40,40,0,pass
1,subway,full,21,21,0,pass
2,taxi,mobility_only,53,38,15,pass
3,fhvhv,mobility_only,46,37,9,pass
4,multimodal,mobility_only,233,142,91,pass


Representation policy applied.


Findings\. The representation policy from 3\.2\.1 is being applied correctly\. Bus and Subway use their full scaled feature spaces because they have no missingness\-indicator columns\. Taxi, FHVHV, and Multimodal use mobility\-only inputs: Taxi keeps 38 of 53 scaled columns, FHVHV keeps 37 of 46, and Multimodal keeps 142 of 233\. The excluded columns are the missingness indicators, so this notebook evaluates mobility\-state representations without allowing indicator\-heavy structure to drive PCA\-to\-UMAP or UMAP comparisons\.

Load PCA explained\-variance outputs from 3\.2\.1 and verify the number of PCA components used for the 80% PCA representation\. This keeps PCA\-to\-UMAP tied to the same PCA results already evaluated upstream\.

In [13]:
# ---------------------------------------------------------------------
# Validate PCA component counts for 80% representation
# ---------------------------------------------------------------------

pca_explained_variance_df = pd.read_csv(PCA_EXPLAINED_VARIANCE_PATH)
pca_threshold_summary_df = pd.read_csv(PCA_VARIANCE_THRESHOLD_SUMMARY_PATH)

required_pca_variance_columns = [
    "feature_set",
    "component",
    "cumulative_variance_pct",
]

required_threshold_columns = [
    "feature_set",
    "pca_representation",
    "components_for_80pct",
]

missing_pca_variance_columns = [
    column
    for column in required_pca_variance_columns
    if column not in pca_explained_variance_df.columns
]

missing_threshold_columns = [
    column
    for column in required_threshold_columns
    if column not in pca_threshold_summary_df.columns
]

assert not missing_pca_variance_columns, (
    f"pca_explained_variance_df is missing required columns: {missing_pca_variance_columns}"
)

assert not missing_threshold_columns, (
    f"pca_threshold_summary_df is missing required columns: {missing_threshold_columns}"
)

pca_component_threshold_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_variance_df = (
        pca_explained_variance_df
        .loc[pca_explained_variance_df["feature_set"].eq(feature_set_name)]
        .sort_values("component")
        .copy()
    )

    assert len(feature_variance_df) > 0, (
        f"No PCA explained-variance rows found for {feature_set_name}."
    )

    computed_components_for_80pct = int(
        feature_variance_df.loc[
            feature_variance_df["cumulative_variance_pct"].ge(
                PCA_VARIANCE_THRESHOLD_FOR_UMAP_INPUT * 100
            ),
            "component",
        ].iloc[0]
    )

    exported_threshold_row = pca_threshold_summary_df.loc[
        pca_threshold_summary_df["feature_set"].eq(feature_set_name)
    ].copy()

    assert len(exported_threshold_row) == 1, (
        f"Expected exactly one threshold summary row for {feature_set_name}."
    )

    exported_components_for_80pct = int(
        exported_threshold_row["components_for_80pct"].iloc[0]
    )

    pca_component_threshold_records.append(
        {
            "feature_set": feature_set_name,
            "computed_components_for_80pct": computed_components_for_80pct,
            "exported_components_for_80pct": exported_components_for_80pct,
            "delta": computed_components_for_80pct - exported_components_for_80pct,
            "status": "pass" if computed_components_for_80pct == exported_components_for_80pct else "review",
        }
    )

pca_component_threshold_df = pd.DataFrame(pca_component_threshold_records)

display(pca_component_threshold_df)

assert pca_component_threshold_df["status"].eq("pass").all(), (
    "One or more PCA component counts differs from the exported 80% threshold summary."
)

print("PCA component counts validated against exported threshold summary.")

,feature_set,computed_components_for_80pct,exported_components_for_80pct,delta,status
0,bus,14,14,0,pass
1,subway,11,11,0,pass
2,taxi,15,15,0,pass
3,fhvhv,13,13,0,pass
4,multimodal,46,46,0,pass


PCA component counts validated against exported threshold summary.


Findings\. The 80% PCA component counts now reconcile cleanly with the 3\.2\.1 exports\. Bus, Subway, and FHVHV match the familiar full\-space thresholds, while Taxi and Multimodal correctly use the mobility\-only default PCA representation carried forward from 3\.2\.1\. This confirms that the PCA\-to\-UMAP branch in 3\.2\.4 is anchored to the exported downstream PCA definition rather than the broader full\-feature compressibility table\.

Finally, validate the exported PCA scores and 2D UMAP visualization assets\. These are comparison assets in this notebook, while new UMAP modeling candidates will be created later\.

In [14]:
# ---------------------------------------------------------------------
# Validate PCA score and 2D UMAP comparison assets
# ---------------------------------------------------------------------

comparison_asset_records = []

with duckdb.connect() as con:
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        metadata_path = sql_path(ROW_METADATA_PATHS_BY_SET[feature_set_name])
        pca_score_path = sql_path(PCA_SCORE_PATHS_BY_SET[feature_set_name])
        canonical_umap_path = sql_path(CANONICAL_UMAP_2D_PATHS_BY_SET[feature_set_name])

        metadata_rows = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{metadata_path}')"
        ).fetchone()[0]

        pca_schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{pca_score_path}')"
        ).fetchdf()
        pca_columns = pca_schema_df["column_name"].tolist()
        pca_rows = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{pca_score_path}')"
        ).fetchone()[0]

        umap_schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{canonical_umap_path}')"
        ).fetchdf()
        umap_columns = umap_schema_df["column_name"].tolist()
        umap_rows = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{canonical_umap_path}')"
        ).fetchone()[0]

        pca_component_columns = [
            column
            for column in pca_columns
            if column.startswith("PC")
        ]
        umap_coordinate_columns = [
            column
            for column in umap_columns
            if column in ["umap_1", "umap_2"]
        ]

        comparison_asset_records.append(
            {
                "feature_set": feature_set_name,
                "metadata_rows": int(metadata_rows),
                "pca_score_rows": int(pca_rows),
                "pca_component_columns": len(pca_component_columns),
                "umap_2d_rows": int(umap_rows),
                "umap_coordinate_columns": len(umap_coordinate_columns),
                "pca_has_modeling_row_id": "modeling_row_id" in pca_columns,
                "umap_has_modeling_row_id": "modeling_row_id" in umap_columns,
            }
        )

comparison_asset_validation_df = pd.DataFrame(comparison_asset_records)

comparison_asset_validation_df["status"] = np.where(
    (
        comparison_asset_validation_df["pca_score_rows"].eq(
            comparison_asset_validation_df["metadata_rows"]
        )
        & comparison_asset_validation_df["umap_2d_rows"].gt(0)
        & comparison_asset_validation_df["pca_component_columns"].gt(0)
        & comparison_asset_validation_df["umap_coordinate_columns"].eq(2)
        & comparison_asset_validation_df["pca_has_modeling_row_id"]
        & comparison_asset_validation_df["umap_has_modeling_row_id"]
    ),
    "pass",
    "review",
)

display(comparison_asset_validation_df)

assert comparison_asset_validation_df["status"].eq("pass").all(), (
    "One or more PCA or 2D UMAP comparison assets failed validation."
)

print("PCA and 2D UMAP comparison assets validated.")

,feature_set,metadata_rows,pca_score_rows,pca_component_columns,umap_2d_rows,umap_coordinate_columns,pca_has_modeling_row_id,umap_has_modeling_row_id,status
0,bus,1472947,1472947,10,50000,2,True,True,pass
1,subway,911455,911455,10,50000,2,True,True,pass
2,taxi,1541800,1541800,10,50000,2,True,True,pass
3,fhvhv,1541800,1541800,10,50000,2,True,True,pass
4,multimodal,1541800,1541800,10,50000,2,True,True,pass


PCA and 2D UMAP comparison assets validated.


Findings\. The PCA and 2D UMAP comparison assets are aligned and ready to use\. Each feature set has a full\-row PCA score table with 10 exported principal components and a valid modeling\_row\_id, so PCA\-based comparisons can run over the complete modeling universe\. The 2D UMAP assets are smaller sampled comparison tables at 50,000 rows each, also with modeling\_row\_id preserved, which makes them suitable as visualization baselines without pretending to be full downstream modeling representations\.

> 📌 Note\. The exported 10\-component PCA score tables are comparison assets, not the PCA\-to\-UMAP modeling input\. The PCA\-to\-UMAP branch in this notebook uses the modality\-specific number of principal components required to reach 80% cumulative explained variance\.

### Section Summary

> At this point, the notebook has confirmed that all upstream representation assets are available and aligned: 3\.1\.1 matrices and metadata, 3\.2\.1 PCA score assets, and 3\.2\.3 canonical 2D UMAP visualization outputs\. The next step is to build a shared evaluation sample so raw, PCA, and UMAP candidate representations can be compared on the same observations\.

## 3\.2\.4\.2 Build Evaluation Samples

The expensive representation\-quality checks in this notebook should run on one shared row sample per feature set\. We anchor those samples to the canonical 2D UMAP exports from 3\.2\.3, then pull the matching rows from the scaled matrices, PCA score assets, and row metadata tables\. That gives the later evaluation sections one aligned comparison frame for Bus, Subway, Taxi, FHVHV, and Multimodal without re\-sampling each representation independently\.

These are evaluation samples, not final modeling outputs\. Full\-dataset construction should wait until the representation choices are settled\.

### Set Up Evaluation Samples

Start by defining the sample policy and output paths\. This makes the later build and validation cells easier to read\.

In [15]:
# ---------------------------------------------------------------------
# Define evaluation sample policy and output paths
# ---------------------------------------------------------------------

EVALUATION_SAMPLE_ROWS = 50_000

EVALUATION_SAMPLE_ID_PATHS_BY_SET = {
    feature_set_name: INTERMEDIATE_OUTPUT_DIR / f"{feature_set_name}_evaluation_sample_ids.parquet"
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

EVALUATION_SAMPLE_METADATA_PATHS_BY_SET = {
    feature_set_name: INTERMEDIATE_OUTPUT_DIR / f"{feature_set_name}_evaluation_sample_metadata.parquet"
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

EVALUATION_SAMPLE_SCALED_PATHS_BY_SET = {
    feature_set_name: INTERMEDIATE_OUTPUT_DIR / f"{feature_set_name}_evaluation_sample_scaled.parquet"
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

EVALUATION_SAMPLE_PCA_PATHS_BY_SET = {
    feature_set_name: INTERMEDIATE_OUTPUT_DIR / f"{feature_set_name}_evaluation_sample_pca_scores.parquet"
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

EVALUATION_SAMPLE_UMAP2D_PATHS_BY_SET = {
    feature_set_name: INTERMEDIATE_OUTPUT_DIR / f"{feature_set_name}_evaluation_sample_umap_2d.parquet"
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

evaluation_sample_output_path_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "sample_rows_target": EVALUATION_SAMPLE_ROWS,
            "sample_id_output_path": compact_path(EVALUATION_SAMPLE_ID_PATHS_BY_SET[feature_set_name]),
            "metadata_output_path": compact_path(EVALUATION_SAMPLE_METADATA_PATHS_BY_SET[feature_set_name]),
            "scaled_output_path": compact_path(EVALUATION_SAMPLE_SCALED_PATHS_BY_SET[feature_set_name]),
            "pca_output_path": compact_path(EVALUATION_SAMPLE_PCA_PATHS_BY_SET[feature_set_name]),
            "umap_2d_output_path": compact_path(EVALUATION_SAMPLE_UMAP2D_PATHS_BY_SET[feature_set_name]),
        }
        for feature_set_name in MODEL_FEATURE_SET_NAMES
    ]
)

display(evaluation_sample_output_path_df)

print("Evaluation sample policy and output paths defined.")

,feature_set,sample_rows_target,sample_id_output_path,metadata_output_path,scaled_output_path,pca_output_path,umap_2d_output_path
0,bus,50000,pipeline_data/3.2.4.intermediate_tables/bus_ev...,pipeline_data/3.2.4.intermediate_tables/bus_ev...,pipeline_data/3.2.4.intermediate_tables/bus_ev...,pipeline_data/3.2.4.intermediate_tables/bus_ev...,pipeline_data/3.2.4.intermediate_tables/bus_ev...
1,subway,50000,pipeline_data/3.2.4.intermediate_tables/subway...,pipeline_data/3.2.4.intermediate_tables/subway...,pipeline_data/3.2.4.intermediate_tables/subway...,pipeline_data/3.2.4.intermediate_tables/subway...,pipeline_data/3.2.4.intermediate_tables/subway...
2,taxi,50000,pipeline_data/3.2.4.intermediate_tables/taxi_e...,pipeline_data/3.2.4.intermediate_tables/taxi_e...,pipeline_data/3.2.4.intermediate_tables/taxi_e...,pipeline_data/3.2.4.intermediate_tables/taxi_e...,pipeline_data/3.2.4.intermediate_tables/taxi_e...
3,fhvhv,50000,pipeline_data/3.2.4.intermediate_tables/fhvhv_...,pipeline_data/3.2.4.intermediate_tables/fhvhv_...,pipeline_data/3.2.4.intermediate_tables/fhvhv_...,pipeline_data/3.2.4.intermediate_tables/fhvhv_...,pipeline_data/3.2.4.intermediate_tables/fhvhv_...
4,multimodal,50000,pipeline_data/3.2.4.intermediate_tables/multim...,pipeline_data/3.2.4.intermediate_tables/multim...,pipeline_data/3.2.4.intermediate_tables/multim...,pipeline_data/3.2.4.intermediate_tables/multim...,pipeline_data/3.2.4.intermediate_tables/multim...


Evaluation sample policy and output paths defined.


Next, use the canonical 2D UMAP exports as the shared sample anchor\. Those files already define the 50,000\-row comparison universe we want to use across representations\.

In [16]:
# ---------------------------------------------------------------------
# Build shared evaluation sample ids from canonical 2D UMAP exports
# ---------------------------------------------------------------------

evaluation_sample_id_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    source_path = CANONICAL_UMAP_2D_PATHS_BY_SET[feature_set_name]
    output_path = EVALUATION_SAMPLE_ID_PATHS_BY_SET[feature_set_name]

    should_rebuild = REBUILD_EVALUATION_SAMPLES or (not output_path.exists())

    if should_rebuild:
        source_path_sql = sql_path(source_path)

        with duckdb.connect() as con:
            sample_id_df = con.execute(
                f"""
                SELECT DISTINCT modeling_row_id
                FROM read_parquet('{source_path_sql}')
                ORDER BY modeling_row_id
                """
            ).fetchdf()

        assert len(sample_id_df) == EVALUATION_SAMPLE_ROWS, (
            f"{feature_set_name} canonical UMAP sample has {len(sample_id_df)} rows, expected {EVALUATION_SAMPLE_ROWS}."
        )

        output_path.parent.mkdir(parents=True, exist_ok=True)
        sample_id_df.to_parquet(output_path, index=False)

        sample_rows = len(sample_id_df)
        output_action = "written"
    else:
        sample_rows = read_parquet_row_count(output_path)
        output_action = "already_exists"

    evaluation_sample_id_records.append(
        {
            "feature_set": feature_set_name,
            "sample_rows": sample_rows,
            "source": "canonical_umap_2d",
            "output_action": output_action,
        }
    )

evaluation_sample_id_df = pd.DataFrame(evaluation_sample_id_records)
evaluation_sample_id_df["status"] = np.where(
    evaluation_sample_id_df["sample_rows"].eq(EVALUATION_SAMPLE_ROWS),
    "pass",
    "review",
)

display(evaluation_sample_id_df)

assert evaluation_sample_id_df["status"].eq("pass").all(), (
    "One or more evaluation sample id tables is missing or has the wrong row count."
)

print("Shared evaluation sample ids are ready.")

,feature_set,sample_rows,source,output_action,status
0,bus,50000,canonical_umap_2d,already_exists,pass
1,subway,50000,canonical_umap_2d,already_exists,pass
2,taxi,50000,canonical_umap_2d,already_exists,pass
3,fhvhv,50000,canonical_umap_2d,already_exists,pass
4,multimodal,50000,canonical_umap_2d,already_exists,pass


Shared evaluation sample ids are ready.


Findings\. Shared evaluation sample ids were created successfully for all five feature sets\. Each sample contains the expected 50,000 modeling\_row\_id values, all drawn from the canonical 2D UMAP exports from 3\.2\.3\. This gives the rest of the notebook one stable row universe per modality, so raw features, PCA representations, UMAP representations, and metadata can be compared on the same observations rather than on separately sampled subsets\.

> 📌 Note\. This section uses the canonical 2D UMAP exports only as a stable row\-sampling anchor\. The notebook is not treating those 2D embeddings as the final modeling representation\.

### Build Aligned Evaluation Samples

With the shared row ids in place, build the aligned metadata sample\. This becomes the common interpretation layer for every representation comparison later on\.

In [17]:
# ---------------------------------------------------------------------
# Materialize aligned evaluation sample metadata
# ---------------------------------------------------------------------

evaluation_sample_metadata_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    sample_id_path = EVALUATION_SAMPLE_ID_PATHS_BY_SET[feature_set_name]
    metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set_name]
    output_path = EVALUATION_SAMPLE_METADATA_PATHS_BY_SET[feature_set_name]

    should_rebuild = REBUILD_EVALUATION_SAMPLES or (not output_path.exists())

    if should_rebuild:
        sample_id_path_sql = sql_path(sample_id_path)
        metadata_path_sql = sql_path(metadata_path)

        with duckdb.connect() as con:
            sample_metadata_df = con.execute(
                f"""
                SELECT m.*
                FROM read_parquet('{metadata_path_sql}') AS m
                INNER JOIN read_parquet('{sample_id_path_sql}') AS s
                    ON m.modeling_row_id = s.modeling_row_id
                ORDER BY m.modeling_row_id
                """
            ).fetchdf()

        output_path.parent.mkdir(parents=True, exist_ok=True)
        sample_metadata_df.to_parquet(output_path, index=False)

        sample_rows = len(sample_metadata_df)
        metadata_columns = len(sample_metadata_df.columns)
        output_action = "written"
    else:
        sample_rows = read_parquet_row_count(output_path)
        metadata_columns = len(get_parquet_columns(output_path))
        output_action = "already_exists"

    evaluation_sample_metadata_records.append(
        {
            "feature_set": feature_set_name,
            "sample_rows": sample_rows,
            "metadata_columns": metadata_columns,
            "output_action": output_action,
        }
    )

evaluation_sample_metadata_df = pd.DataFrame(evaluation_sample_metadata_records)
evaluation_sample_metadata_df["status"] = np.where(
    evaluation_sample_metadata_df["sample_rows"].eq(EVALUATION_SAMPLE_ROWS),
    "pass",
    "review",
)

display(evaluation_sample_metadata_df)

assert evaluation_sample_metadata_df["status"].eq("pass").all(), (
    "One or more evaluation sample metadata tables is missing or has the wrong row count."
)

print("Evaluation sample metadata is ready.")

,feature_set,sample_rows,metadata_columns,output_action,status
0,bus,50000,12,already_exists,pass
1,subway,50000,12,already_exists,pass
2,taxi,50000,12,already_exists,pass
3,fhvhv,50000,12,already_exists,pass
4,multimodal,50000,12,already_exists,pass


Evaluation sample metadata is ready.


Findings\. The aligned metadata samples were created successfully for all five feature sets\. Each sample contains the expected 50,000 rows and 12 metadata columns, giving the later representation comparisons one shared interpretation layer for geography, time, and policy context\. That means the expensive metric sections can compare raw, PCA, and UMAP spaces on the same observations without losing the fields needed to explain what those observations represent\.

Next, materialize the aligned scaled\-feature sample\. This is the raw feature\-space comparison table for later metric sections\.

In [18]:
# ---------------------------------------------------------------------
# Materialize aligned scaled-feature evaluation samples
# ---------------------------------------------------------------------

evaluation_sample_scaled_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    sample_id_path = EVALUATION_SAMPLE_ID_PATHS_BY_SET[feature_set_name]
    scaled_path = SCALED_MATRIX_PATHS_BY_SET[feature_set_name]
    output_path = EVALUATION_SAMPLE_SCALED_PATHS_BY_SET[feature_set_name]

    should_rebuild = REBUILD_EVALUATION_SAMPLES or (not output_path.exists())

    if should_rebuild:
        sample_id_path_sql = sql_path(sample_id_path)
        scaled_path_sql = sql_path(scaled_path)

        with duckdb.connect() as con:
            sample_scaled_df = con.execute(
                f"""
                SELECT x.*
                FROM read_parquet('{scaled_path_sql}') AS x
                INNER JOIN read_parquet('{sample_id_path_sql}') AS s
                    ON x.modeling_row_id = s.modeling_row_id
                ORDER BY x.modeling_row_id
                """
            ).fetchdf()

        output_path.parent.mkdir(parents=True, exist_ok=True)
        sample_scaled_df.to_parquet(output_path, index=False)

        sample_rows = len(sample_scaled_df)
        scaled_columns = len(sample_scaled_df.columns) - 1
        output_action = "written"
    else:
        sample_rows = read_parquet_row_count(output_path)
        scaled_columns = len(modeling_feature_columns_from_schema(output_path))
        output_action = "already_exists"

    evaluation_sample_scaled_records.append(
        {
            "feature_set": feature_set_name,
            "sample_rows": sample_rows,
            "scaled_columns": scaled_columns,
            "output_action": output_action,
        }
    )

evaluation_sample_scaled_df = pd.DataFrame(evaluation_sample_scaled_records)
evaluation_sample_scaled_df["status"] = np.where(
    evaluation_sample_scaled_df["sample_rows"].eq(EVALUATION_SAMPLE_ROWS),
    "pass",
    "review",
)

display(evaluation_sample_scaled_df)

assert evaluation_sample_scaled_df["status"].eq("pass").all(), (
    "One or more evaluation sample scaled tables is missing or has the wrong row count."
)

print("Evaluation sample scaled-feature tables are ready.")

,feature_set,sample_rows,scaled_columns,output_action,status
0,bus,50000,40,already_exists,pass
1,subway,50000,21,already_exists,pass
2,taxi,50000,53,already_exists,pass
3,fhvhv,50000,46,already_exists,pass
4,multimodal,50000,233,already_exists,pass


Evaluation sample scaled-feature tables are ready.


Findings\. The raw feature\-space samples are now materialized and aligned across all five modalities\. Each sample contains the full scaled feature set for that matrix: 40 Bus columns, 21 Subway columns, 53 Taxi columns, 46 FHVHV columns, and 233 Multimodal columns\. This preserves the full raw comparison space exactly as it was handed off from 3\.1\.1, so later sections can evaluate whether dimension reduction actually improves neighborhood behavior relative to the unreduced feature space\.

Now materialize the aligned PCA score sample\. This keeps the exported PCA comparison space on the same rows as the scaled and UMAP assets\.

In [19]:
# ---------------------------------------------------------------------
# Materialize aligned PCA-score evaluation samples
# ---------------------------------------------------------------------

evaluation_sample_pca_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    sample_id_path = EVALUATION_SAMPLE_ID_PATHS_BY_SET[feature_set_name]
    pca_score_path = PCA_SCORE_PATHS_BY_SET[feature_set_name]
    output_path = EVALUATION_SAMPLE_PCA_PATHS_BY_SET[feature_set_name]

    should_rebuild = REBUILD_EVALUATION_SAMPLES or (not output_path.exists())

    if should_rebuild:
        sample_id_path_sql = sql_path(sample_id_path)
        pca_score_path_sql = sql_path(pca_score_path)

        with duckdb.connect() as con:
            sample_pca_df = con.execute(
                f"""
                SELECT p.*
                FROM read_parquet('{pca_score_path_sql}') AS p
                INNER JOIN read_parquet('{sample_id_path_sql}') AS s
                    ON p.modeling_row_id = s.modeling_row_id
                ORDER BY p.modeling_row_id
                """
            ).fetchdf()

        output_path.parent.mkdir(parents=True, exist_ok=True)
        sample_pca_df.to_parquet(output_path, index=False)

        sample_rows = len(sample_pca_df)
        pca_columns = len(
            [column for column in sample_pca_df.columns if column.startswith("PC")]
        )
        output_action = "written"
    else:
        sample_rows = read_parquet_row_count(output_path)
        pca_columns = len(
            [
                column
                for column in get_parquet_columns(output_path)
                if column.startswith("PC")
            ]
        )
        output_action = "already_exists"

    evaluation_sample_pca_records.append(
        {
            "feature_set": feature_set_name,
            "sample_rows": sample_rows,
            "pca_columns": pca_columns,
            "output_action": output_action,
        }
    )

evaluation_sample_pca_df = pd.DataFrame(evaluation_sample_pca_records)
evaluation_sample_pca_df["status"] = np.where(
    evaluation_sample_pca_df["sample_rows"].eq(EVALUATION_SAMPLE_ROWS),
    "pass",
    "review",
)

display(evaluation_sample_pca_df)

assert evaluation_sample_pca_df["status"].eq("pass").all(), (
    "One or more evaluation sample PCA tables is missing or has the wrong row count."
)

print("Evaluation sample PCA-score tables are ready.")

,feature_set,sample_rows,pca_columns,output_action,status
0,bus,50000,10,already_exists,pass
1,subway,50000,10,already_exists,pass
2,taxi,50000,10,already_exists,pass
3,fhvhv,50000,10,already_exists,pass
4,multimodal,50000,10,already_exists,pass


Evaluation sample PCA-score tables are ready.


Findings\. The PCA comparison samples were written successfully and stay aligned to the same 50,000\-row universe\. Each feature set carries the 10 exported PCA score columns from 3\.2\.1, which keeps the baseline linear representation compact and directly comparable to the raw and UMAP samples\. These PCA score tables are therefore ready for representation\-quality comparisons, while the separate PCA\-to\-UMAP branch can still use the modality\-specific 80% variance rule when that path is evaluated later\.

> 📌 Note\. The exported PCA score sample is still a comparison asset\. The separate PCA \-\> UMAP branch later in the notebook should use the 80%\-variance PCA representation rather than assuming that these 10 exported score columns are the modeling input\.

Finally, cache the aligned 2D UMAP sample itself under the evaluation\-sample naming scheme and validate row alignment across all four sample assets\.

In [20]:
# ---------------------------------------------------------------------
# Materialize aligned 2D UMAP evaluation samples and validate alignment
# ---------------------------------------------------------------------

evaluation_sample_umap_records = []
evaluation_sample_alignment_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    sample_id_path = EVALUATION_SAMPLE_ID_PATHS_BY_SET[feature_set_name]
    canonical_umap_path = CANONICAL_UMAP_2D_PATHS_BY_SET[feature_set_name]
    output_path = EVALUATION_SAMPLE_UMAP2D_PATHS_BY_SET[feature_set_name]

    should_rebuild = REBUILD_EVALUATION_SAMPLES or (not output_path.exists())

    if should_rebuild:
        sample_id_path_sql = sql_path(sample_id_path)
        canonical_umap_path_sql = sql_path(canonical_umap_path)

        with duckdb.connect() as con:
            sample_umap_df = con.execute(
                f"""
                SELECT u.*
                FROM read_parquet('{canonical_umap_path_sql}') AS u
                INNER JOIN read_parquet('{sample_id_path_sql}') AS s
                    ON u.modeling_row_id = s.modeling_row_id
                ORDER BY u.modeling_row_id
                """
            ).fetchdf()

        output_path.parent.mkdir(parents=True, exist_ok=True)
        sample_umap_df.to_parquet(output_path, index=False)

        sample_rows = len(sample_umap_df)
        umap_coordinate_columns = len(
            [column for column in sample_umap_df.columns if column in ["umap_1", "umap_2"]]
        )
        output_action = "written"
    else:
        sample_rows = read_parquet_row_count(output_path)
        umap_coordinate_columns = len(
            [
                column
                for column in get_parquet_columns(output_path)
                if column in ["umap_1", "umap_2"]
            ]
        )
        output_action = "already_exists"

    evaluation_sample_umap_records.append(
        {
            "feature_set": feature_set_name,
            "sample_rows": sample_rows,
            "umap_coordinate_columns": umap_coordinate_columns,
            "output_action": output_action,
        }
    )

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    scaled_rows = read_parquet_row_count(EVALUATION_SAMPLE_SCALED_PATHS_BY_SET[feature_set_name])
    pca_rows = read_parquet_row_count(EVALUATION_SAMPLE_PCA_PATHS_BY_SET[feature_set_name])
    umap_rows = read_parquet_row_count(EVALUATION_SAMPLE_UMAP2D_PATHS_BY_SET[feature_set_name])
    metadata_rows = read_parquet_row_count(EVALUATION_SAMPLE_METADATA_PATHS_BY_SET[feature_set_name])

    evaluation_sample_alignment_records.append(
        {
            "feature_set": feature_set_name,
            "scaled_rows": scaled_rows,
            "pca_rows": pca_rows,
            "umap_2d_rows": umap_rows,
            "metadata_rows": metadata_rows,
            "status": "pass"
            if (
                scaled_rows == EVALUATION_SAMPLE_ROWS
                and pca_rows == EVALUATION_SAMPLE_ROWS
                and umap_rows == EVALUATION_SAMPLE_ROWS
                and metadata_rows == EVALUATION_SAMPLE_ROWS
            )
            else "review",
        }
    )

evaluation_sample_umap_df = pd.DataFrame(evaluation_sample_umap_records)
evaluation_sample_umap_df["status"] = np.where(
    evaluation_sample_umap_df["sample_rows"].eq(EVALUATION_SAMPLE_ROWS)
    & evaluation_sample_umap_df["umap_coordinate_columns"].eq(2),
    "pass",
    "review",
)

evaluation_sample_alignment_df = pd.DataFrame(evaluation_sample_alignment_records)

display(evaluation_sample_umap_df)
display(evaluation_sample_alignment_df)

assert evaluation_sample_umap_df["status"].eq("pass").all(), (
    "One or more evaluation sample UMAP tables is missing or malformed."
)

assert evaluation_sample_alignment_df["status"].eq("pass").all(), (
    "One or more evaluation sample assets is row-misaligned."
)

print("Evaluation sample assets are aligned across scaled, PCA, UMAP, and metadata tables.")

,feature_set,sample_rows,umap_coordinate_columns,output_action,status
0,bus,50000,2,already_exists,pass
1,subway,50000,2,already_exists,pass
2,taxi,50000,2,already_exists,pass
3,fhvhv,50000,2,already_exists,pass
4,multimodal,50000,2,already_exists,pass


,feature_set,scaled_rows,pca_rows,umap_2d_rows,metadata_rows,status
0,bus,50000,50000,50000,50000,pass
1,subway,50000,50000,50000,50000,pass
2,taxi,50000,50000,50000,50000,pass
3,fhvhv,50000,50000,50000,50000,pass
4,multimodal,50000,50000,50000,50000,pass


Evaluation sample assets are aligned across scaled, PCA, UMAP, and metadata tables.


Findings\. The sampled 2D UMAP comparison tables were written successfully, and the final alignment check passes cleanly across all four sample assets\. For every feature set, the scaled\-feature sample, PCA sample, UMAP sample, and metadata sample all contain the same 50,000 rows\. This is the key setup milestone for 3\.2\.4: the expensive representation\-quality metrics can now compare raw, PCA, and UMAP candidate spaces on exactly the same observations instead of on slightly different sampled universes\.

### Section Summary

> This section built the shared evaluation universe for 3\.2\.4\. Starting from the canonical 2D UMAP exports, the notebook created one aligned 50,000\-row sample per feature set and materialized matching metadata, scaled\-feature, PCA\-score, and 2D UMAP tables\. The final alignment checks confirm that all candidate representation spaces now sit on the same observations, which gives the evaluation sections a clean basis for comparing raw, PCA, and UMAP behavior without sampling drift\.

## 3\.2\.4\.3 Evaluate UMAP Modeling Dimensionality

The 2D UMAP embeddings from 3\.2\.3 were chosen because they gave the clearest visualization view of local mobility structure\. The modeling question is slightly different: whether downstream clustering and anomaly\-detection work benefits from keeping more UMAP dimensions than a plot can show\. This section holds the canonical UMAP settings fixed and varies only embedding dimensionality so we can see how much local\-structure quality improves as the representation becomes richer\.

The comparison focuses on representation quality rather than visual appeal\. For each candidate dimensionality, we will examine trustworthiness, neighborhood overlap, local\-neighbor retention, k\-nearest\-neighbor distance behavior, distance concentration, runtime, and practical complexity\. The goal is to decide whether higher\-dimensional UMAP meaningfully improves the representation enough to justify carrying the extra dimensions forward\.

### Define candidate UMAP dimensionalities and output paths

The visualization notebook already established the neighborhood and packing settings we want to carry forward\. What remains open is how many UMAP dimensions the modeling representation should keep\. This section defines the candidate dimensionalities, ties them to the shared evaluation samples, and prepares the output paths that later blocks will use for fitting and comparison\.

Start by defining the candidate dimensionalities and the modeling\-evaluation scope\. This keeps the rest of the section explicit about what is being tested and what is being held fixed\.

In [21]:
# ---------------------------------------------------------------------
# Define candidate UMAP dimensionalities and evaluation scope
# ---------------------------------------------------------------------

UMAP_MODEL_DIMENSION_CANDIDATES = [2, 5, 10, 15]

UMAP_MODEL_INPUT_SPACE = "raw_scaled"

UMAP_MODEL_DIMENSION_SCOPE_DF = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "input_space": UMAP_MODEL_INPUT_SPACE,
            "representation_policy": REPRESENTATION_POLICY_BY_SET[feature_set_name],
            "candidate_dimensions": ", ".join(
                str(dimension) for dimension in UMAP_MODEL_DIMENSION_CANDIDATES
            ),
            "evaluation_sample_rows": EVALUATION_SAMPLE_ROWS,
            "n_neighbors": UMAP_N_NEIGHBORS,
            "min_dist": UMAP_MIN_DIST,
            "metric": UMAP_METRIC,
        }
        for feature_set_name in MODEL_FEATURE_SET_NAMES
    ]
)

display(UMAP_MODEL_DIMENSION_SCOPE_DF)

print("Candidate UMAP dimensionalities defined.")

,feature_set,input_space,representation_policy,candidate_dimensions,evaluation_sample_rows,n_neighbors,min_dist,metric
0,bus,raw_scaled,full,"2, 5, 10, 15",50000,50,0.1,euclidean
1,subway,raw_scaled,full,"2, 5, 10, 15",50000,50,0.1,euclidean
2,taxi,raw_scaled,mobility_only,"2, 5, 10, 15",50000,50,0.1,euclidean
3,fhvhv,raw_scaled,mobility_only,"2, 5, 10, 15",50000,50,0.1,euclidean
4,multimodal,raw_scaled,mobility_only,"2, 5, 10, 15",50000,50,0.1,euclidean


Candidate UMAP dimensionalities defined.


Findings\. The dimensionality sweep is scoped cleanly and keeps the neighborhood definition fixed\. All five feature sets will be evaluated on the same 50,000\-row sample using the canonical n\_neighbors = 50, min\_dist = 0\.10, and Euclidean metric from 3\.2\.3, while varying only the number of retained UMAP dimensions\. Bus and Subway use their full scaled inputs, while Taxi, FHVHV, and Multimodal use the mobility\-only inputs carried forward from 3\.2\.1\.

Next, build the output\-path dictionaries for the dimensionality sweep\. Later cells should use these lookups instead of constructing filenames ad hoc\.

In [22]:
# ---------------------------------------------------------------------
# Define UMAP dimensionality output paths
# ---------------------------------------------------------------------

UMAP_DIMENSIONALITY_OUTPUT_PATHS = {
    (feature_set_name, dimension): (
        INTERMEDIATE_OUTPUT_DIR
        / f"{feature_set_name}_umap_{UMAP_MODEL_INPUT_SPACE}_{dimension}d.parquet"
    )
    for feature_set_name in MODEL_FEATURE_SET_NAMES
    for dimension in UMAP_MODEL_DIMENSION_CANDIDATES
}

umap_dimensionality_output_path_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "input_space": UMAP_MODEL_INPUT_SPACE,
            "umap_dimensions": dimension,
            "output_path": compact_path(
                UMAP_DIMENSIONALITY_OUTPUT_PATHS[(feature_set_name, dimension)]
            ),
        }
        for feature_set_name in MODEL_FEATURE_SET_NAMES
        for dimension in UMAP_MODEL_DIMENSION_CANDIDATES
    ]
)

display(umap_dimensionality_output_path_df)

print("UMAP dimensionality output paths defined.")

,feature_set,input_space,umap_dimensions,output_path
0,bus,raw_scaled,2,pipeline_data/3.2.4.intermediate_tables/bus_um...
1,bus,raw_scaled,5,pipeline_data/3.2.4.intermediate_tables/bus_um...
2,bus,raw_scaled,10,pipeline_data/3.2.4.intermediate_tables/bus_um...
3,bus,raw_scaled,15,pipeline_data/3.2.4.intermediate_tables/bus_um...
4,subway,raw_scaled,2,pipeline_data/3.2.4.intermediate_tables/subway...
5,subway,raw_scaled,5,pipeline_data/3.2.4.intermediate_tables/subway...
6,subway,raw_scaled,10,pipeline_data/3.2.4.intermediate_tables/subway...
7,subway,raw_scaled,15,pipeline_data/3.2.4.intermediate_tables/subway...
8,taxi,raw_scaled,2,pipeline_data/3.2.4.intermediate_tables/taxi_u...
9,taxi,raw_scaled,5,pipeline_data/3.2.4.intermediate_tables/taxi_u...


UMAP dimensionality output paths defined.


Findings\. Output paths are now defined for the full dimensionality grid: five feature sets crossed with four candidate UMAP dimensionalities, for 20 intermediate embedding files in total\. This gives the fitting section one predictable cache location per modality\-dimension pair and keeps the dimensionality sweep from depending on ad hoc filenames later in the notebook\.

Finally, create a compact candidate manifest table that combines dimensionality choices, input\-column counts, and output destinations in one place\. This gives the fit block a clean control table\.

In [23]:
# ---------------------------------------------------------------------
# Build UMAP dimensionality candidate manifest
# ---------------------------------------------------------------------

UMAP_DIMENSIONALITY_CANDIDATE_DF = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "input_space": UMAP_MODEL_INPUT_SPACE,
            "umap_dimensions": dimension,
            "input_columns": len(REPRESENTATION_COLUMNS_BY_SET[feature_set_name]),
            "sample_rows": EVALUATION_SAMPLE_ROWS,
            "n_neighbors": UMAP_N_NEIGHBORS,
            "min_dist": UMAP_MIN_DIST,
            "metric": UMAP_METRIC,
            "output_path": str(
                UMAP_DIMENSIONALITY_OUTPUT_PATHS[(feature_set_name, dimension)]
            ),
        }
        for feature_set_name in MODEL_FEATURE_SET_NAMES
        for dimension in UMAP_MODEL_DIMENSION_CANDIDATES
    ]
)

UMAP_DIMENSIONALITY_CANDIDATE_DF["status"] = "ready"

display(UMAP_DIMENSIONALITY_CANDIDATE_DF)

print("UMAP dimensionality candidate manifest prepared.")

,feature_set,input_space,umap_dimensions,input_columns,sample_rows,n_neighbors,min_dist,metric,output_path,status
0,bus,raw_scaled,2,40,50000,50,0.1,euclidean,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,ready
1,bus,raw_scaled,5,40,50000,50,0.1,euclidean,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,ready
2,bus,raw_scaled,10,40,50000,50,0.1,euclidean,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,ready
3,bus,raw_scaled,15,40,50000,50,0.1,euclidean,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,ready
4,subway,raw_scaled,2,21,50000,50,0.1,euclidean,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,ready
5,subway,raw_scaled,5,21,50000,50,0.1,euclidean,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,ready
6,subway,raw_scaled,10,21,50000,50,0.1,euclidean,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,ready
7,subway,raw_scaled,15,21,50000,50,0.1,euclidean,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,ready
8,taxi,raw_scaled,2,38,50000,50,0.1,euclidean,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,ready
9,taxi,raw_scaled,5,38,50000,50,0.1,euclidean,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,ready


UMAP dimensionality candidate manifest prepared.


Findings\. The dimensionality candidate manifest is ready and confirms the modeling sweep is internally consistent\. Bus will be tested on 40 raw\-scaled columns, Subway on 21, Taxi on 38, FHVHV on 37, and Multimodal on 142, with each feature set evaluated at 2D, 5D, 10D, and 15D under the same UMAP settings\. At this point the notebook is ready to move from setup into the expensive fitting stage without ambiguity about inputs, dimensionalities, or output destinations\.

### Fit candidate UMAP dimensionality embeddings

With the dimensionality candidates defined, the next step is to fit the actual UMAP representations on the shared evaluation samples\. The neighborhood settings stay fixed at the canonical 3\.2\.3 values, while only the number of retained UMAP dimensions changes\. This keeps the comparison focused on the modeling question we actually care about: whether higher\-dimensional UMAP preserves more useful local structure than the 2D visualization space\.

Start by fitting or loading the dimensionality sweep\. This block should be the expensive workhorse for the section, so it follows the usual cache pattern: fit when the rebuild toggle is on or when the output file does not exist yet, otherwise reuse the cached embedding\.

> ⚠️ Warning\. This block can take a while because it fits 20 embeddings: five feature sets across four dimensionalities\. It is worth leaving REBUILD\_UMAP\_DIMENSIONALITY = False on reruns unless the evaluation samples or UMAP settings have changed\.

In [24]:
# ---------------------------------------------------------------------
# Fit candidate UMAP dimensionality embeddings
# ---------------------------------------------------------------------

fit_progress_columns = [
    "feature_set",
    "input_space",
    "umap_dimensions",
    "sample_rows",
    "input_columns",
    "n_neighbors",
    "min_dist",
    "metric",
    "output_action",
    "elapsed_seconds",
]

existing_fit_progress_df = (
    pd.read_parquet(UMAP_DIMENSIONALITY_PROGRESS_PATH)
    if UMAP_DIMENSIONALITY_PROGRESS_PATH.exists() and not REBUILD_UMAP_DIMENSIONALITY
    else pd.DataFrame(columns=fit_progress_columns)
)

completed_keys = set(
    zip(
        existing_fit_progress_df.get("feature_set", pd.Series(dtype="object")),
        existing_fit_progress_df.get("input_space", pd.Series(dtype="object")),
        existing_fit_progress_df.get("umap_dimensions", pd.Series(dtype="int64")),
    )
)

umap_dimensionality_fit_records = existing_fit_progress_df.to_dict("records")

total_candidates = len(UMAP_DIMENSIONALITY_CANDIDATE_DF)

for candidate_index, candidate_row in enumerate(
    UMAP_DIMENSIONALITY_CANDIDATE_DF.itertuples(index=False),
    start=1,
):
    feature_set_name = candidate_row.feature_set
    input_space = candidate_row.input_space
    umap_dimensions = int(candidate_row.umap_dimensions)
    input_columns = int(candidate_row.input_columns)
    output_path = Path(candidate_row.output_path)

    candidate_key = (feature_set_name, input_space, umap_dimensions)
    should_rebuild = REBUILD_UMAP_DIMENSIONALITY or (not output_path.exists())

    if (candidate_key in completed_keys) and not should_rebuild:
        print(
            f"[{candidate_index}/{total_candidates}] "
            f"{feature_set_name} {umap_dimensions}D already recorded; skipping."
        )
        continue

    print(
        f"[{candidate_index}/{total_candidates}] "
        f"Starting {feature_set_name} {umap_dimensions}D "
        f"({input_columns} columns, {EVALUATION_SAMPLE_ROWS} rows)."
    )

    start_time = time.time()

    sample_scaled_path = EVALUATION_SAMPLE_SCALED_PATHS_BY_SET[feature_set_name]
    selected_input_columns = REPRESENTATION_COLUMNS_BY_SET[feature_set_name]

    if should_rebuild:
        sample_scaled_path_sql = sql_path(sample_scaled_path)
        selected_columns_sql = ", ".join(
            [duckdb_identifier("modeling_row_id")]
            + [duckdb_identifier(column) for column in selected_input_columns]
        )

        with duckdb.connect() as con:
            sample_input_df = con.execute(
                f"""
                SELECT {selected_columns_sql}
                FROM read_parquet('{sample_scaled_path_sql}')
                ORDER BY modeling_row_id
                """
            ).fetchdf()

        X = sample_input_df[selected_input_columns].astype("float32", copy=False)

        umap_model = umap.UMAP(
            n_neighbors=UMAP_N_NEIGHBORS,
            min_dist=UMAP_MIN_DIST,
            n_components=umap_dimensions,
            metric=UMAP_METRIC,
            random_state=UMAP_RANDOM_STATE,
            transform_seed=UMAP_RANDOM_STATE,
        )

        embedding_values = umap_model.fit_transform(X)

        embedding_column_names = [
            f"umap_{dimension_number}"
            for dimension_number in range(1, umap_dimensions + 1)
        ]

        embedding_df = pd.DataFrame(
            embedding_values,
            columns=embedding_column_names,
        )
        embedding_df.insert(0, "modeling_row_id", sample_input_df["modeling_row_id"].values)
        embedding_df["feature_set"] = feature_set_name
        embedding_df["input_space"] = input_space
        embedding_df["umap_dimensions"] = umap_dimensions
        embedding_df["n_neighbors"] = UMAP_N_NEIGHBORS
        embedding_df["min_dist"] = UMAP_MIN_DIST
        embedding_df["metric"] = UMAP_METRIC
        embedding_df["random_state"] = UMAP_RANDOM_STATE

        output_path.parent.mkdir(parents=True, exist_ok=True)
        embedding_df.to_parquet(output_path, index=False)

        embedding_rows = len(embedding_df)
        output_action = "fit"

        del sample_input_df, X, embedding_values, embedding_df
    else:
        embedding_rows = read_parquet_row_count(output_path)
        output_action = "loaded_from_cache"

    elapsed_seconds = round(time.time() - start_time, 1)

    new_record = {
        "feature_set": feature_set_name,
        "input_space": input_space,
        "umap_dimensions": umap_dimensions,
        "sample_rows": embedding_rows,
        "input_columns": input_columns,
        "n_neighbors": UMAP_N_NEIGHBORS,
        "min_dist": UMAP_MIN_DIST,
        "metric": UMAP_METRIC,
        "output_action": output_action,
        "elapsed_seconds": elapsed_seconds,
    }

    umap_dimensionality_fit_records = [
        record
        for record in umap_dimensionality_fit_records
        if (
            record["feature_set"],
            record["input_space"],
            int(record["umap_dimensions"]),
        ) != candidate_key
    ]
    umap_dimensionality_fit_records.append(new_record)

    progress_df = pd.DataFrame(umap_dimensionality_fit_records)
    progress_df.to_parquet(UMAP_DIMENSIONALITY_PROGRESS_PATH, index=False)

    print(
        f"[{candidate_index}/{total_candidates}] "
        f"Finished {feature_set_name} {umap_dimensions}D in {elapsed_seconds}s "
        f"({output_action})."
    )

umap_dimensionality_fit_df = (
    pd.DataFrame(umap_dimensionality_fit_records)
    .sort_values(["feature_set", "umap_dimensions"])
    .reset_index(drop=True)
)

umap_dimensionality_fit_df["status"] = np.where(
    umap_dimensionality_fit_df["sample_rows"].eq(EVALUATION_SAMPLE_ROWS),
    "pass",
    "review",
)

display(umap_dimensionality_fit_df)

assert umap_dimensionality_fit_df["status"].eq("pass").all(), (
    "One or more UMAP dimensionality embeddings failed to fit or has the wrong row count."
)

print("Candidate UMAP dimensionality embeddings are ready.")

[1/20] bus 2D already recorded; skipping.
[2/20] bus 5D already recorded; skipping.
[3/20] bus 10D already recorded; skipping.
[4/20] bus 15D already recorded; skipping.
[5/20] subway 2D already recorded; skipping.
[6/20] subway 5D already recorded; skipping.
[7/20] subway 10D already recorded; skipping.
[8/20] subway 15D already recorded; skipping.
[9/20] taxi 2D already recorded; skipping.
[10/20] taxi 5D already recorded; skipping.
[11/20] taxi 10D already recorded; skipping.
[12/20] taxi 15D already recorded; skipping.
[13/20] fhvhv 2D already recorded; skipping.
[14/20] fhvhv 5D already recorded; skipping.
[15/20] fhvhv 10D already recorded; skipping.
[16/20] fhvhv 15D already recorded; skipping.
[17/20] multimodal 2D already recorded; skipping.
[18/20] multimodal 5D already recorded; skipping.
[19/20] multimodal 10D already recorded; skipping.
[20/20] multimodal 15D already recorded; skipping.


,feature_set,input_space,umap_dimensions,sample_rows,input_columns,n_neighbors,min_dist,metric,output_action,elapsed_seconds,status
0,bus,raw_scaled,2,50000,40,50,0.1,euclidean,fit,107.3,pass
1,bus,raw_scaled,5,50000,40,50,0.1,euclidean,fit,95.5,pass
2,bus,raw_scaled,10,50000,40,50,0.1,euclidean,fit,109.7,pass
3,bus,raw_scaled,15,50000,40,50,0.1,euclidean,fit,115.1,pass
4,fhvhv,raw_scaled,2,50000,37,50,0.1,euclidean,fit,55.8,pass
5,fhvhv,raw_scaled,5,50000,37,50,0.1,euclidean,fit,67.5,pass
6,fhvhv,raw_scaled,10,50000,37,50,0.1,euclidean,fit,74.4,pass
7,fhvhv,raw_scaled,15,50000,37,50,0.1,euclidean,fit,122.2,pass
8,multimodal,raw_scaled,2,50000,142,50,0.1,euclidean,fit,84.8,pass
9,multimodal,raw_scaled,5,50000,142,50,0.1,euclidean,fit,86.7,pass


Candidate UMAP dimensionality embeddings are ready.


Findings\. All 20 candidate UMAP embeddings were fit successfully across the five feature sets and four dimensionality settings\. Runtime stayed in a practical range for this 50,000\-row evaluation sample: most fits landed around 1 to 2 minutes, with the slowest run at 160\.8 seconds for Subway 10D\. Higher dimensionality did not create a runaway cost explosion, but it did raise runtime modestly for several modalities, especially Multimodal, Subway, and FHVHV\. That gives the next metric blocks a complete candidate grid to compare on representation quality rather than feasibility alone\.

This quick check confirms that every candidate embedding was written correctly before we start scoring representation quality\. At this point, the important question is no longer whether the files exist, but whether each candidate preserves the expected row universe and identifier structure for fair comparison\.

In [25]:
# ---------------------------------------------------------------------
# Validate candidate UMAP dimensionality embeddings
# ---------------------------------------------------------------------

umap_dimensionality_validation_records = []

total_candidates = len(UMAP_DIMENSIONALITY_CANDIDATE_DF)

for candidate_index, candidate_row in enumerate(
    UMAP_DIMENSIONALITY_CANDIDATE_DF.itertuples(index=False),
    start=1,
):
    feature_set_name = candidate_row.feature_set
    umap_dimensions = int(candidate_row.umap_dimensions)
    output_path = Path(candidate_row.output_path)

    print(
        f"[{candidate_index}/{total_candidates}] "
        f"Validating {feature_set_name} {umap_dimensions}D."
    )

    output_schema_columns = set(read_parquet_schema(output_path)["column_name"])

    expected_embedding_columns = [
        f"umap_{dimension_number}"
        for dimension_number in range(1, umap_dimensions + 1)
    ]

    required_columns = {
        "modeling_row_id",
        "feature_set",
        "input_space",
        "umap_dimensions",
        "n_neighbors",
        "min_dist",
        "metric",
        "random_state",
        *expected_embedding_columns,
    }

    missing_required_columns = sorted(
        column for column in required_columns
        if column not in output_schema_columns
    )

    output_path_sql = sql_path(output_path)

    with duckdb.connect() as con:
        validation_row = con.execute(
            f"""
            SELECT
                COUNT(*) AS embedding_rows,
                COUNT(DISTINCT modeling_row_id) AS distinct_modeling_rows
            FROM read_parquet('{output_path_sql}')
            """
        ).fetchdf().iloc[0]

    embedding_rows = int(validation_row["embedding_rows"])
    distinct_modeling_rows = int(validation_row["distinct_modeling_rows"])
    duplicate_modeling_rows = embedding_rows - distinct_modeling_rows

    umap_dimensionality_validation_records.append(
        {
            "feature_set": feature_set_name,
            "umap_dimensions": umap_dimensions,
            "embedding_rows": embedding_rows,
            "duplicate_modeling_rows": duplicate_modeling_rows,
            "missing_required_columns": (
                ", ".join(missing_required_columns)
                if missing_required_columns
                else "none"
            ),
            "status": (
                "pass"
                if (
                    embedding_rows == EVALUATION_SAMPLE_ROWS
                    and duplicate_modeling_rows == 0
                    and not missing_required_columns
                )
                else "review"
            ),
        }
    )

umap_dimensionality_validation_df = pd.DataFrame(
    umap_dimensionality_validation_records
)

display(umap_dimensionality_validation_df)

assert umap_dimensionality_validation_df["status"].eq("pass").all(), (
    "One or more candidate UMAP dimensionality embeddings failed validation."
)

print("Candidate UMAP dimensionality embeddings validated.")

[1/20] Validating bus 2D.
[2/20] Validating bus 5D.
[3/20] Validating bus 10D.
[4/20] Validating bus 15D.
[5/20] Validating subway 2D.
[6/20] Validating subway 5D.
[7/20] Validating subway 10D.
[8/20] Validating subway 15D.
[9/20] Validating taxi 2D.
[10/20] Validating taxi 5D.
[11/20] Validating taxi 10D.
[12/20] Validating taxi 15D.
[13/20] Validating fhvhv 2D.
[14/20] Validating fhvhv 5D.
[15/20] Validating fhvhv 10D.
[16/20] Validating fhvhv 15D.
[17/20] Validating multimodal 2D.
[18/20] Validating multimodal 5D.
[19/20] Validating multimodal 10D.
[20/20] Validating multimodal 15D.


,feature_set,umap_dimensions,embedding_rows,duplicate_modeling_rows,missing_required_columns,status
0,bus,2,50000,0,none,pass
1,bus,5,50000,0,none,pass
2,bus,10,50000,0,none,pass
3,bus,15,50000,0,none,pass
4,subway,2,50000,0,none,pass
5,subway,5,50000,0,none,pass
6,subway,10,50000,0,none,pass
7,subway,15,50000,0,none,pass
8,taxi,2,50000,0,none,pass
9,taxi,5,50000,0,none,pass


Candidate UMAP dimensionality embeddings validated.


Findings\. The dimensionality candidate outputs passed structural validation cleanly\. Every embedding contains 50,000 rows, no candidate has duplicate modeling\_row\_id values, and all required columns are present\. With the candidate files validated, we can treat any differences in the next section as true representation\-quality differences rather than export or alignment problems\.

### Evaluate dimensionality quality metrics

Measure Trustworthiness\. Trustworthiness is the most direct check of whether nearby points in the UMAP space were also nearby in the original feature space\. This section uses the shared evaluation samples and asks whether adding dimensions makes the embedding more faithful to local neighborhoods\.

> ⚠️ Warning\. This block is often slower than it looks\. Trustworthiness has to compare local rank structure in the original feature space and in the embedding space, so it is doing more work than a simple summary statistic\.

In [26]:
# ---------------------------------------------------------------------
# Measure trustworthiness across UMAP dimensionality candidates
# ---------------------------------------------------------------------

trustworthiness_progress_columns = [
    "feature_set",
    "input_space",
    "umap_dimensions",
    "sample_rows",
    "input_columns",
    "trustworthiness_k",
    "trustworthiness_score",
    "elapsed_seconds",
]

existing_trustworthiness_progress_df = (
    pd.read_parquet(UMAP_TRUSTWORTHINESS_PROGRESS_PATH)
    if UMAP_TRUSTWORTHINESS_PROGRESS_PATH.exists() and not REBUILD_UMAP_TRUSTWORTHINESS
    else pd.DataFrame(columns=trustworthiness_progress_columns)
)

completed_keys = set(
    zip(
        existing_trustworthiness_progress_df.get("feature_set", pd.Series(dtype="object")),
        existing_trustworthiness_progress_df.get("input_space", pd.Series(dtype="object")),
        existing_trustworthiness_progress_df.get("umap_dimensions", pd.Series(dtype="int64")),
    )
)

umap_trustworthiness_records = existing_trustworthiness_progress_df.to_dict("records")
total_candidates = len(UMAP_DIMENSIONALITY_CANDIDATE_DF)

for candidate_index, candidate_row in enumerate(
    UMAP_DIMENSIONALITY_CANDIDATE_DF.itertuples(index=False),
    start=1,
):
    feature_set_name = candidate_row.feature_set
    input_space = candidate_row.input_space
    umap_dimensions = int(candidate_row.umap_dimensions)
    input_columns = REPRESENTATION_COLUMNS_BY_SET[feature_set_name]

    candidate_key = (feature_set_name, input_space, umap_dimensions)

    if (candidate_key in completed_keys) and not REBUILD_UMAP_TRUSTWORTHINESS:
        print(
            f"[{candidate_index}/{total_candidates}] "
            f"Trustworthiness already recorded for {feature_set_name} {umap_dimensions}D; skipping."
        )
        continue

    print(
        f"[{candidate_index}/{total_candidates}] "
        f"Computing trustworthiness for {feature_set_name} {umap_dimensions}D."
    )

    embedding_path = UMAP_DIMENSIONALITY_OUTPUT_PATHS[(feature_set_name, umap_dimensions)]
    scaled_sample_path = EVALUATION_SAMPLE_SCALED_PATHS_BY_SET[feature_set_name]

    scaled_sample_path_sql = sql_path(scaled_sample_path)
    embedding_path_sql = sql_path(embedding_path)

    selected_scaled_columns_sql = ", ".join(
        [duckdb_identifier("modeling_row_id")]
        + [duckdb_identifier(column) for column in input_columns]
    )

    selected_embedding_columns = [
        f"umap_{dimension_number}"
        for dimension_number in range(1, umap_dimensions + 1)
    ]
    selected_embedding_columns_sql = ", ".join(
        [duckdb_identifier("modeling_row_id")]
        + [duckdb_identifier(column) for column in selected_embedding_columns]
    )

    with duckdb.connect() as con:
        evaluation_df = con.execute(
            f"""
            WITH scaled_sample AS (
                SELECT {selected_scaled_columns_sql}
                FROM read_parquet('{scaled_sample_path_sql}')
            ),
            embedding_sample AS (
                SELECT {selected_embedding_columns_sql}
                FROM read_parquet('{embedding_path_sql}')
            )
            SELECT *
            FROM scaled_sample AS s
            INNER JOIN embedding_sample AS e
                USING (modeling_row_id)
            ORDER BY hash(modeling_row_id, {REPRESENTATION_EVALUATION_RANDOM_STATE})
            LIMIT {NEIGHBOR_METRIC_SAMPLE_ROWS}
            """
        ).fetchdf()

    X_high = evaluation_df[input_columns].astype("float32", copy=False).to_numpy()
    X_low = evaluation_df[selected_embedding_columns].astype("float32", copy=False).to_numpy()

    start_time = time.time()
    trustworthiness_score = trustworthiness(
        X=X_high,
        X_embedded=X_low,
        n_neighbors=NEIGHBOR_EVALUATION_K,
    )
    elapsed_seconds = round(time.time() - start_time, 2)

    new_record = {
        "feature_set": feature_set_name,
        "input_space": input_space,
        "umap_dimensions": umap_dimensions,
        "sample_rows": len(evaluation_df),
        "input_columns": len(input_columns),
        "trustworthiness_k": NEIGHBOR_EVALUATION_K,
        "trustworthiness_score": round(float(trustworthiness_score), 4),
        "elapsed_seconds": elapsed_seconds,
    }

    umap_trustworthiness_records = [
        record
        for record in umap_trustworthiness_records
        if (
            record["feature_set"],
            record["input_space"],
            int(record["umap_dimensions"]),
        ) != candidate_key
    ]
    umap_trustworthiness_records.append(new_record)

    progress_df = pd.DataFrame(umap_trustworthiness_records)
    progress_df.to_parquet(UMAP_TRUSTWORTHINESS_PROGRESS_PATH, index=False)

    print(
        f"[{candidate_index}/{total_candidates}] "
        f"Finished trustworthiness for {feature_set_name} {umap_dimensions}D in {elapsed_seconds}s."
    )

umap_trustworthiness_df = (
    pd.DataFrame(umap_trustworthiness_records)
    .sort_values(["feature_set", "umap_dimensions"])
    .reset_index(drop=True)
)

if WRITE_INTERMEDIATE_OUTPUTS:
    umap_trustworthiness_df.to_parquet(UMAP_TRUSTWORTHINESS_OUTPUT_PATH, index=False)

umap_trustworthiness_df["status"] = np.where(
    umap_trustworthiness_df["sample_rows"].eq(NEIGHBOR_METRIC_SAMPLE_ROWS),
    "pass",
    "review",
)

display(umap_trustworthiness_df)

assert umap_trustworthiness_df["status"].eq("pass").all(), (
    "One or more trustworthiness evaluations used the wrong sample size."
)

print("UMAP trustworthiness metrics computed.")

[1/20] Trustworthiness already recorded for bus 2D; skipping.
[2/20] Trustworthiness already recorded for bus 5D; skipping.
[3/20] Trustworthiness already recorded for bus 10D; skipping.
[4/20] Trustworthiness already recorded for bus 15D; skipping.
[5/20] Trustworthiness already recorded for subway 2D; skipping.
[6/20] Trustworthiness already recorded for subway 5D; skipping.
[7/20] Trustworthiness already recorded for subway 10D; skipping.
[8/20] Trustworthiness already recorded for subway 15D; skipping.
[9/20] Trustworthiness already recorded for taxi 2D; skipping.
[10/20] Trustworthiness already recorded for taxi 5D; skipping.
[11/20] Trustworthiness already recorded for taxi 10D; skipping.
[12/20] Trustworthiness already recorded for taxi 15D; skipping.
[13/20] Trustworthiness already recorded for fhvhv 2D; skipping.
[14/20] Trustworthiness already recorded for fhvhv 5D; skipping.
[15/20] Trustworthiness already recorded for fhvhv 10D; skipping.
[16/20] Trustworthiness already rec

,feature_set,input_space,umap_dimensions,sample_rows,input_columns,trustworthiness_k,trustworthiness_score,elapsed_seconds,status
0,bus,raw_scaled,2,15000,40,15,0.9337,6.96,pass
1,bus,raw_scaled,5,15000,40,15,0.9773,7.08,pass
2,bus,raw_scaled,10,15000,40,15,0.9816,7.24,pass
3,bus,raw_scaled,15,15000,40,15,0.9816,7.27,pass
4,fhvhv,raw_scaled,2,15000,37,15,0.9321,6.98,pass
5,fhvhv,raw_scaled,5,15000,37,15,0.9758,7.68,pass
6,fhvhv,raw_scaled,10,15000,37,15,0.9786,7.39,pass
7,fhvhv,raw_scaled,15,15000,37,15,0.9788,7.38,pass
8,multimodal,raw_scaled,2,15000,142,15,0.8999,7.29,pass
9,multimodal,raw_scaled,5,15000,142,15,0.9496,7.49,pass


UMAP trustworthiness metrics computed.


> 📌 Note\. Trustworthiness measures how well the lower\-dimensional UMAP embedding preserves local neighborhoods from the original input space\. A score near 1 means that points that were close to each other in the original feature space usually remain close after embedding\. Here, k = 15, so the metric focuses on whether each observation’s 15 nearest neighbors are preserved\. For example, a trustworthiness of 0\.9816 for Bus 10D means the 10\-dimensional embedding preserves local neighborhood structure very well, with very few “false neighbors” introduced by the dimensionality reduction\.

Findings\. Trustworthiness improves sharply when we move from 2D to higher\-dimensional UMAP, then mostly levels off\. That pattern is consistent across all five feature sets\. Bus rises from 0\.934 in 2D to 0\.977 in 5D and then only inches up to 0\.982 by 10D and 15D; Taxi shows a similar jump from 0\.908 to 0\.972 and then flattens near 0\.976\. The biggest low\-dimensional penalty appears in Multimodal, where 2D trustworthiness is 0\.900 before improving to 0\.950 in 5D and 0\.955 in 10D\. Subway is the cleanest case overall, staying high even in 2D at 0\.974 and reaching about 0\.989 above 5D\. The main takeaway is that 2D is visibly worse as a modeling representation, while most of the local\-neighborhood benefit arrives by 5D or 10D rather than 15D\.

Trustworthiness Plot\. This plot shows how faithfully each UMAP dimensionality preserves local neighborhoods from the original feature space\. The key question is whether the gain from moving beyond 2D keeps paying off or starts to level off\.

In [27]:
# ---------------------------------------------------------------------
# Plot trustworthiness by UMAP dimensionality
# ---------------------------------------------------------------------

trustworthiness_plot_df = (
    umap_trustworthiness_df
    .sort_values(["feature_set", "umap_dimensions"])
    .copy()
)

trustworthiness_fig = px.line(
    trustworthiness_plot_df,
    x="umap_dimensions",
    y="trustworthiness_score",
    color="feature_set",
    markers=True,
    category_orders={"feature_set": MODEL_FEATURE_SET_NAMES},
    labels={
        "umap_dimensions": "UMAP dimensions",
        "trustworthiness_score": "Trustworthiness",
        "feature_set": "Feature set",
    },
    title="UMAP Trustworthiness by Modeling Dimensionality",
)

trustworthiness_fig.update_traces(
    line={"width": 3},
    marker={"size": 8},
    hovertemplate=(
        "Feature set: %{fullData.name}<br>"
        "Dimensions: %{x}<br>"
        "Trustworthiness: %{y:.4f}"
        "<extra></extra>"
    ),
)

trustworthiness_fig.update_layout(
    width=900,
    height=520,
    template="plotly_white",
    legend_title="Feature set",
)

trustworthiness_fig.show()

Findings\. The line chart makes the trustworthiness pattern easy to read: every modality gets a large jump from 2D to 5D, and then the curves flatten\. Subway remains strongest throughout, while Multimodal and Taxi pay the biggest penalty in 2D\. Visually, this shifts the modeling decision away from “should we use 2D?” and toward the narrower question of whether 10D is worth a modest gain over 5D\.

Measure Neighborhood Retention\. Trustworthiness tells us whether the embedding invents false neighbors\. Neighborhood\-retention metrics complement that by asking how many original nearest neighbors survive in the UMAP space\.

> ⚠️ Warning\. This block is asking a stricter question than trustworthiness alone\. A trustworthy embedding can still rearrange some local neighborhoods; this table shows how much of the original nearest\-neighbor structure is actually preserved\.

In [28]:
# ---------------------------------------------------------------------
# Measure neighborhood retention across UMAP dimensionality candidates
# ---------------------------------------------------------------------

neighbor_retention_progress_columns = [
    "feature_set",
    "input_space",
    "umap_dimensions",
    "sample_rows",
    "neighbor_k",
    "mean_neighbor_overlap_share",
    "median_neighbor_overlap_share",
    "p10_neighbor_overlap_share",
]

existing_neighbor_retention_progress_df = (
    pd.read_parquet(UMAP_NEIGHBOR_RETENTION_PROGRESS_PATH)
    if UMAP_NEIGHBOR_RETENTION_PROGRESS_PATH.exists() and not REBUILD_UMAP_NEIGHBOR_RETENTION
    else pd.DataFrame(columns=neighbor_retention_progress_columns)
)

completed_keys = set(
    zip(
        existing_neighbor_retention_progress_df.get("feature_set", pd.Series(dtype="object")),
        existing_neighbor_retention_progress_df.get("input_space", pd.Series(dtype="object")),
        existing_neighbor_retention_progress_df.get("umap_dimensions", pd.Series(dtype="int64")),
    )
)

umap_neighbor_retention_records = existing_neighbor_retention_progress_df.to_dict("records")
total_candidates = len(UMAP_DIMENSIONALITY_CANDIDATE_DF)

for candidate_index, candidate_row in enumerate(
    UMAP_DIMENSIONALITY_CANDIDATE_DF.itertuples(index=False),
    start=1,
):
    feature_set_name = candidate_row.feature_set
    input_space = candidate_row.input_space
    umap_dimensions = int(candidate_row.umap_dimensions)
    input_columns = REPRESENTATION_COLUMNS_BY_SET[feature_set_name]

    candidate_key = (feature_set_name, input_space, umap_dimensions)

    if (candidate_key in completed_keys) and not REBUILD_UMAP_NEIGHBOR_RETENTION:
        print(
            f"[{candidate_index}/{total_candidates}] "
            f"Neighborhood retention already recorded for {feature_set_name} {umap_dimensions}D; skipping."
        )
        continue

    print(
        f"[{candidate_index}/{total_candidates}] "
        f"Computing neighborhood retention for {feature_set_name} {umap_dimensions}D."
    )

    embedding_path = UMAP_DIMENSIONALITY_OUTPUT_PATHS[(feature_set_name, umap_dimensions)]
    scaled_sample_path = EVALUATION_SAMPLE_SCALED_PATHS_BY_SET[feature_set_name]

    scaled_sample_path_sql = sql_path(scaled_sample_path)
    embedding_path_sql = sql_path(embedding_path)

    selected_scaled_columns_sql = ", ".join(
        [duckdb_identifier("modeling_row_id")]
        + [duckdb_identifier(column) for column in input_columns]
    )

    selected_embedding_columns = [
        f"umap_{dimension_number}"
        for dimension_number in range(1, umap_dimensions + 1)
    ]
    selected_embedding_columns_sql = ", ".join(
        [duckdb_identifier("modeling_row_id")]
        + [duckdb_identifier(column) for column in selected_embedding_columns]
    )

    with duckdb.connect() as con:
        evaluation_df = con.execute(
            f"""
            WITH scaled_sample AS (
                SELECT {selected_scaled_columns_sql}
                FROM read_parquet('{scaled_sample_path_sql}')
            ),
            embedding_sample AS (
                SELECT {selected_embedding_columns_sql}
                FROM read_parquet('{embedding_path_sql}')
            )
            SELECT *
            FROM scaled_sample AS s
            INNER JOIN embedding_sample AS e
                USING (modeling_row_id)
            ORDER BY hash(modeling_row_id, {REPRESENTATION_EVALUATION_RANDOM_STATE})
            LIMIT {NEIGHBOR_METRIC_SAMPLE_ROWS}
            """
        ).fetchdf()

    X_high = evaluation_df[input_columns].astype("float32", copy=False).to_numpy()
    X_low = evaluation_df[selected_embedding_columns].astype("float32", copy=False).to_numpy()

    high_knn = NearestNeighbors(
        n_neighbors=NEIGHBOR_EVALUATION_K + 1,
        metric="euclidean",
    ).fit(X_high)
    low_knn = NearestNeighbors(
        n_neighbors=NEIGHBOR_EVALUATION_K + 1,
        metric="euclidean",
    ).fit(X_low)

    high_indices = high_knn.kneighbors(return_distance=False)[:, 1:]
    low_indices = low_knn.kneighbors(return_distance=False)[:, 1:]

    overlap_counts = [
        len(set(high_indices[row_idx]).intersection(set(low_indices[row_idx])))
        for row_idx in range(len(evaluation_df))
    ]

    overlap_share = np.array(overlap_counts) / NEIGHBOR_EVALUATION_K

    new_record = {
        "feature_set": feature_set_name,
        "input_space": input_space,
        "umap_dimensions": umap_dimensions,
        "sample_rows": len(evaluation_df),
        "neighbor_k": NEIGHBOR_EVALUATION_K,
        "mean_neighbor_overlap_share": round(float(overlap_share.mean()), 4),
        "median_neighbor_overlap_share": round(float(np.median(overlap_share)), 4),
        "p10_neighbor_overlap_share": round(float(np.quantile(overlap_share, 0.10)), 4),
    }

    umap_neighbor_retention_records = [
        record
        for record in umap_neighbor_retention_records
        if (
            record["feature_set"],
            record["input_space"],
            int(record["umap_dimensions"]),
        ) != candidate_key
    ]
    umap_neighbor_retention_records.append(new_record)

    progress_df = pd.DataFrame(umap_neighbor_retention_records)
    progress_df.to_parquet(UMAP_NEIGHBOR_RETENTION_PROGRESS_PATH, index=False)

    print(
        f"[{candidate_index}/{total_candidates}] "
        f"Finished neighborhood retention for {feature_set_name} {umap_dimensions}D."
    )

umap_neighbor_retention_df = (
    pd.DataFrame(umap_neighbor_retention_records)
    .sort_values(["feature_set", "umap_dimensions"])
    .reset_index(drop=True)
)

if WRITE_INTERMEDIATE_OUTPUTS:
    umap_neighbor_retention_df.to_parquet(UMAP_NEIGHBOR_RETENTION_OUTPUT_PATH, index=False)

umap_neighbor_retention_df["status"] = np.where(
    umap_neighbor_retention_df["sample_rows"].eq(NEIGHBOR_METRIC_SAMPLE_ROWS),
    "pass",
    "review",
)

display(umap_neighbor_retention_df)

assert umap_neighbor_retention_df["status"].eq("pass").all(), (
    "One or more neighborhood-retention evaluations used the wrong sample size."
)

print("UMAP neighborhood-retention metrics computed.")

[1/20] Neighborhood retention already recorded for bus 2D; skipping.
[2/20] Neighborhood retention already recorded for bus 5D; skipping.
[3/20] Neighborhood retention already recorded for bus 10D; skipping.
[4/20] Neighborhood retention already recorded for bus 15D; skipping.
[5/20] Neighborhood retention already recorded for subway 2D; skipping.
[6/20] Neighborhood retention already recorded for subway 5D; skipping.
[7/20] Neighborhood retention already recorded for subway 10D; skipping.
[8/20] Neighborhood retention already recorded for subway 15D; skipping.
[9/20] Neighborhood retention already recorded for taxi 2D; skipping.
[10/20] Neighborhood retention already recorded for taxi 5D; skipping.
[11/20] Neighborhood retention already recorded for taxi 10D; skipping.
[12/20] Neighborhood retention already recorded for taxi 15D; skipping.
[13/20] Neighborhood retention already recorded for fhvhv 2D; skipping.
[14/20] Neighborhood retention already recorded for fhvhv 5D; skipping.
[15

,feature_set,input_space,umap_dimensions,sample_rows,neighbor_k,mean_neighbor_overlap_share,median_neighbor_overlap_share,p10_neighbor_overlap_share,status
0,bus,raw_scaled,2,15000,15,0.1840,0.1333,0.0000,pass
1,bus,raw_scaled,5,15000,15,0.2893,0.2667,0.0667,pass
2,bus,raw_scaled,10,15000,15,0.2985,0.2667,0.0667,pass
3,bus,raw_scaled,15,15000,15,0.2990,0.2667,0.0667,pass
4,fhvhv,raw_scaled,2,15000,15,0.1947,0.1333,0.0000,pass
5,fhvhv,raw_scaled,5,15000,15,0.3001,0.2667,0.0667,pass
6,fhvhv,raw_scaled,10,15000,15,0.3072,0.2667,0.1333,pass
7,fhvhv,raw_scaled,15,15000,15,0.3087,0.2667,0.1333,pass
8,multimodal,raw_scaled,2,15000,15,0.2140,0.2000,0.0000,pass
9,multimodal,raw_scaled,5,15000,15,0.2844,0.2667,0.0667,pass


UMAP neighborhood-retention metrics computed.


> 📌 Note\. Neighbor overlap gives a more literal neighborhood\-preservation check\. For each observation, we compare its 15 nearest neighbors in the original feature space with its 15 nearest neighbors in the UMAP embedding, then compute the share of overlap\. A mean overlap of 0\.2985 for Bus 10D means that, on average, about 29\.9% of each point’s original 15 nearest neighbors are still among its 15 nearest neighbors in the UMAP representation\. The median and 10th\-percentile values show whether that preservation is typical or concentrated in only part of the sample\.

Findings\. The neighborhood\-overlap results tell the same broad story in a more concrete way: 2D loses too many original nearest neighbors, while higher\-dimensional UMAP recovers a meaningfully larger share and then plateaus\. Bus improves from a mean overlap of 0\.184 in 2D to about 0\.289 in 5D and 0\.299 in 15D; Taxi moves from 0\.187 to 0\.298 and then 0\.309; FHVHV from 0\.195 to 0\.300 and then 0\.309\. Multimodal also improves materially, from 0\.214 in 2D to about 0\.284 in 5D and 0\.291 in 10D and 15D\. Subway again stands out as the most stable feature space, rising from 0\.371 in 2D to 0\.441 in 5D and only modestly higher beyond that\. Across modalities, the big gain comes from leaving 2D behind; after that, 10D and 15D look more like marginal refinements than fundamentally different regimes\.

This plot keeps the neighborhood\-preservation story literal\. The solid line shows average overlap with the original 15\-nearest\-neighbor set, while the shaded band shows how weak that overlap gets in the lower tail\.

In [29]:
# ---------------------------------------------------------------------
# Plot neighborhood-retention by UMAP dimensionality
# ---------------------------------------------------------------------

neighbor_retention_plot_df = (
    umap_neighbor_retention_df
    .sort_values(["feature_set", "umap_dimensions"])
    .copy()
)

retention_fig = go.Figure()

feature_set_colors = {
    "bus": "#1f77b4",
    "subway": "#ff7f0e",
    "taxi": "#2ca02c",
    "fhvhv": "#d62728",
    "multimodal": "#9467bd",
}

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_df = neighbor_retention_plot_df.loc[
        neighbor_retention_plot_df["feature_set"].eq(feature_set_name)
    ].copy()

    if feature_df.empty:
        continue

    color = feature_set_colors.get(feature_set_name, "#4c78a8")
    x_vals = feature_df["umap_dimensions"].tolist()
    mean_vals = feature_df["mean_neighbor_overlap_share"].tolist()
    p10_vals = feature_df["p10_neighbor_overlap_share"].tolist()

    retention_fig.add_trace(
        go.Scatter(
            x=x_vals,
            y=mean_vals,
            mode="lines+markers",
            name=feature_set_name,
            line={"width": 3, "color": color},
            marker={"size": 8},
            hovertemplate=(
                f"Feature set: {feature_set_name}<br>"
                "Dimensions: %{x}<br>"
                "Mean overlap: %{y:.4f}"
                "<extra></extra>"
            ),
        )
    )

    retention_fig.add_trace(
        go.Scatter(
            x=x_vals + x_vals[::-1],
            y=mean_vals + p10_vals[::-1],
            fill="toself",
            fillcolor=color.replace(")", ", 0.14)").replace("rgb", "rgba") if color.startswith("rgb") else "rgba(31,119,180,0.14)",
            line={"color": "rgba(0,0,0,0)"},
            hoverinfo="skip",
            name=f"{feature_set_name} band",
            showlegend=False,
        )
    )

# safer rgba fallback for hex colors
for trace in retention_fig.data:
    if getattr(trace, "fillcolor", None) == "rgba(31,119,180,0.14)" and trace.name.endswith("band"):
        base_name = trace.name.replace(" band", "")
        hex_color = feature_set_colors.get(base_name, "#4c78a8").lstrip("#")
        if len(hex_color) == 6:
            r = int(hex_color[0:2], 16)
            g = int(hex_color[2:4], 16)
            b = int(hex_color[4:6], 16)
            trace.fillcolor = f"rgba({r},{g},{b},0.14)"

retention_fig.update_layout(
    title="UMAP Neighbor Retention by Modeling Dimensionality",
    xaxis_title="UMAP dimensions",
    yaxis_title="Neighbor overlap share",
    width=900,
    height=520,
    template="plotly_white",
    legend_title="Feature set",
)

retention_fig.show()

Findings\. The retention plot adds a useful second lens: not just whether the embedding avoids false neighbors, but how much of each point’s original local neighborhood it actually keeps\. The relative ordering is stable across dimensionalities, with Subway preserving the most local structure and the other four modalities clustering much closer together\. The important visual signal is that the curves bunch tightly after 5D, especially for Bus, Taxi, FHVHV, and Multimodal\. That makes the post\-5D gains look incremental rather than transformative, and it strengthens the case that the real cutoff is between a visibly compressed 2D embedding and a more faithful higher\-dimensional one\.

Profile Distance Behavior\. The representation comparison is not only about neighbor identity\. It is also about whether distances behave in a modeling\-friendly way: whether local distances spread out enough to support density\-based reasoning, and whether the embedding avoids collapsing everything into one nearly uniform cloud\.

In [30]:
# ---------------------------------------------------------------------
# Profile distance behavior across UMAP dimensionality candidates
# ---------------------------------------------------------------------

distance_behavior_progress_columns = [
    "feature_set",
    "input_space",
    "umap_dimensions",
    "sample_rows",
    "neighbor_k",
    "mean_knn_distance",
    "median_knn_distance",
    "p95_knn_distance",
    "pairwise_distance_mean",
    "pairwise_distance_std",
    "distance_concentration_ratio",
]

existing_distance_behavior_progress_df = (
    pd.read_parquet(UMAP_DISTANCE_BEHAVIOR_PROGRESS_PATH)
    if UMAP_DISTANCE_BEHAVIOR_PROGRESS_PATH.exists() and not REBUILD_UMAP_DISTANCE_BEHAVIOR
    else pd.DataFrame(columns=distance_behavior_progress_columns)
)

completed_keys = set(
    zip(
        existing_distance_behavior_progress_df.get("feature_set", pd.Series(dtype="object")),
        existing_distance_behavior_progress_df.get("input_space", pd.Series(dtype="object")),
        existing_distance_behavior_progress_df.get("umap_dimensions", pd.Series(dtype="int64")),
    )
)

umap_distance_behavior_records = existing_distance_behavior_progress_df.to_dict("records")
total_candidates = len(UMAP_DIMENSIONALITY_CANDIDATE_DF)

for candidate_index, candidate_row in enumerate(
    UMAP_DIMENSIONALITY_CANDIDATE_DF.itertuples(index=False),
    start=1,
):
    feature_set_name = candidate_row.feature_set
    input_space = candidate_row.input_space
    umap_dimensions = int(candidate_row.umap_dimensions)

    candidate_key = (feature_set_name, input_space, umap_dimensions)

    if (candidate_key in completed_keys) and not REBUILD_UMAP_DISTANCE_BEHAVIOR:
        print(
            f"[{candidate_index}/{total_candidates}] "
            f"Distance behavior already recorded for {feature_set_name} {umap_dimensions}D; skipping."
        )
        continue

    print(
        f"[{candidate_index}/{total_candidates}] "
        f"Profiling distance behavior for {feature_set_name} {umap_dimensions}D."
    )

    embedding_path = UMAP_DIMENSIONALITY_OUTPUT_PATHS[(feature_set_name, umap_dimensions)]

    embedding_columns = [
        f"umap_{dimension_number}"
        for dimension_number in range(1, umap_dimensions + 1)
    ]
    embedding_path_sql = sql_path(embedding_path)
    selected_embedding_columns_sql = ", ".join(
        [duckdb_identifier("modeling_row_id")]
        + [duckdb_identifier(column) for column in embedding_columns]
    )

    with duckdb.connect() as con:
        embedding_df = con.execute(
            f"""
            SELECT {selected_embedding_columns_sql}
            FROM read_parquet('{embedding_path_sql}')
            ORDER BY hash(modeling_row_id, {REPRESENTATION_EVALUATION_RANDOM_STATE})
            LIMIT {NEIGHBOR_METRIC_SAMPLE_ROWS}
            """
        ).fetchdf()

    X_low = embedding_df[embedding_columns].astype("float32", copy=False).to_numpy()

    knn_model = NearestNeighbors(
        n_neighbors=NEIGHBOR_EVALUATION_K + 1,
        metric="euclidean",
    ).fit(X_low)
    knn_distances = knn_model.kneighbors(return_distance=True)[0][:, 1:]

    mean_knn_distance = float(knn_distances.mean())
    median_knn_distance = float(np.median(knn_distances))
    p95_knn_distance = float(np.quantile(knn_distances, 0.95))

    pairwise_sample_size = min(PAIRWISE_DISTANCE_SAMPLE_ROWS, len(embedding_df))
    pairwise_indices = np.random.default_rng(REPRESENTATION_EVALUATION_RANDOM_STATE).choice(
        len(embedding_df),
        size=pairwise_sample_size,
        replace=False,
    )
    pairwise_X = X_low[pairwise_indices]
    pairwise_distance_values = pairwise_distances(pairwise_X, metric="euclidean")
    pairwise_distance_values = pairwise_distance_values[np.triu_indices_from(pairwise_distance_values, k=1)]

    pairwise_mean = float(pairwise_distance_values.mean())
    pairwise_std = float(pairwise_distance_values.std())
    distance_concentration_ratio = float(pairwise_std / pairwise_mean) if pairwise_mean > 0 else np.nan

    new_record = {
        "feature_set": feature_set_name,
        "input_space": input_space,
        "umap_dimensions": umap_dimensions,
        "sample_rows": len(embedding_df),
        "neighbor_k": NEIGHBOR_EVALUATION_K,
        "mean_knn_distance": round(mean_knn_distance, 4),
        "median_knn_distance": round(median_knn_distance, 4),
        "p95_knn_distance": round(p95_knn_distance, 4),
        "pairwise_distance_mean": round(pairwise_mean, 4),
        "pairwise_distance_std": round(pairwise_std, 4),
        "distance_concentration_ratio": round(distance_concentration_ratio, 4),
    }

    umap_distance_behavior_records = [
        record
        for record in umap_distance_behavior_records
        if (
            record["feature_set"],
            record["input_space"],
            int(record["umap_dimensions"]),
        ) != candidate_key
    ]
    umap_distance_behavior_records.append(new_record)

    progress_df = pd.DataFrame(umap_distance_behavior_records)
    progress_df.to_parquet(UMAP_DISTANCE_BEHAVIOR_PROGRESS_PATH, index=False)

    print(
        f"[{candidate_index}/{total_candidates}] "
        f"Finished distance behavior for {feature_set_name} {umap_dimensions}D."
    )

umap_distance_behavior_df = (
    pd.DataFrame(umap_distance_behavior_records)
    .sort_values(["feature_set", "umap_dimensions"])
    .reset_index(drop=True)
)

if WRITE_INTERMEDIATE_OUTPUTS:
    umap_distance_behavior_df.to_parquet(UMAP_DISTANCE_BEHAVIOR_OUTPUT_PATH, index=False)

umap_distance_behavior_df["status"] = np.where(
    umap_distance_behavior_df["sample_rows"].eq(NEIGHBOR_METRIC_SAMPLE_ROWS),
    "pass",
    "review",
)

display(umap_distance_behavior_df)

assert umap_distance_behavior_df["status"].eq("pass").all(), (
    "One or more distance-behavior evaluations used the wrong sample size."
)

print("UMAP distance-behavior metrics computed.")

[1/20] Distance behavior already recorded for bus 2D; skipping.
[2/20] Distance behavior already recorded for bus 5D; skipping.
[3/20] Distance behavior already recorded for bus 10D; skipping.
[4/20] Distance behavior already recorded for bus 15D; skipping.
[5/20] Distance behavior already recorded for subway 2D; skipping.
[6/20] Distance behavior already recorded for subway 5D; skipping.
[7/20] Distance behavior already recorded for subway 10D; skipping.
[8/20] Distance behavior already recorded for subway 15D; skipping.
[9/20] Distance behavior already recorded for taxi 2D; skipping.
[10/20] Distance behavior already recorded for taxi 5D; skipping.
[11/20] Distance behavior already recorded for taxi 10D; skipping.
[12/20] Distance behavior already recorded for taxi 15D; skipping.
[13/20] Distance behavior already recorded for fhvhv 2D; skipping.
[14/20] Distance behavior already recorded for fhvhv 5D; skipping.
[15/20] Distance behavior already recorded for fhvhv 10D; skipping.
[16/2

,feature_set,input_space,umap_dimensions,sample_rows,neighbor_k,mean_knn_distance,median_knn_distance,p95_knn_distance,pairwise_distance_mean,pairwise_distance_std,distance_concentration_ratio,status
0,bus,raw_scaled,2,15000,15,0.0901,0.0808,0.1816,7.8204,5.2280,0.6685,pass
1,bus,raw_scaled,5,15000,15,0.2043,0.1833,0.4227,7.0781,4.9383,0.6977,pass
2,bus,raw_scaled,10,15000,15,0.2165,0.1916,0.4502,7.0699,4.9596,0.7015,pass
3,bus,raw_scaled,15,15000,15,0.2181,0.1932,0.4517,7.0739,4.9454,0.6991,pass
4,fhvhv,raw_scaled,2,15000,15,0.0929,0.0857,0.1819,7.9381,5.3000,0.6677,pass
5,fhvhv,raw_scaled,5,15000,15,0.2124,0.1900,0.4427,7.0934,4.8428,0.6827,pass
6,fhvhv,raw_scaled,10,15000,15,0.2214,0.1973,0.4640,7.0089,4.7558,0.6785,pass
7,fhvhv,raw_scaled,15,15000,15,0.2227,0.1981,0.4649,6.9618,4.7052,0.6759,pass
8,multimodal,raw_scaled,2,15000,15,0.0888,0.0764,0.1892,6.6046,3.9447,0.5973,pass
9,multimodal,raw_scaled,5,15000,15,0.1794,0.1584,0.3885,5.9888,3.4964,0.5838,pass


UMAP distance-behavior metrics computed.


Findings\. The distance\-profile table shows that higher\-dimensional UMAP changes local geometry much more than global spread\. Across all five feature sets, mean and median k\-nearest\-neighbor distance rise noticeably when we move from 2D to 5D, then increase only modestly beyond that, which suggests that the embedding gains room to separate local neighborhoods once it is no longer forced into a flat 2D surface\. The pairwise\-distance summaries move much less dramatically, so the added dimensions are not simply stretching everything apart uniformly\. The clearest concentration change appears in Subway and Multimodal, where the concentration ratio falls steadily as dimensionality rises; Bus, Taxi, and FHVHV are comparatively stable after the initial jump\. Taken together, the table suggests that most of the geometric adjustment happens by 5D or 10D, with 15D looking more like refinement than a qualitatively different regime\.

> 📌 Note\. This block describes the geometry of the embedded space rather than direct neighborhood fidelity\. mean\_knn\_distance, median\_knn\_distance, and p95\_knn\_distance summarize how far each observation sits from its nearest neighbors in the UMAP space\. pairwise\_distance\_mean and pairwise\_distance\_std summarize the overall spread of the embedding\. The distance\_concentration\_ratio compares the standard deviation of pairwise distances to their mean; lower values suggest a less concentrated, more differentiated space\. These metrics help show whether higher\-dimensional UMAP creates useful local separation without simply inflating all distances uniformly\.

This plot focuses on the geometry of the embedded space rather than direct neighborhood recall\. The three panels let us see whether higher\-dimensional UMAP changes local spacing, overall spread, and distance concentration in a stable or unstable way\.

In [31]:
# ---------------------------------------------------------------------
# Plot distance-profile metrics by UMAP dimensionality
# ---------------------------------------------------------------------

from plotly.subplots import make_subplots
import plotly.graph_objects as go

distance_plot_df = (
    umap_distance_behavior_df
    .sort_values(["feature_set", "umap_dimensions"])
    .copy()
)

distance_metric_specs = [
    ("mean_knn_distance", "Mean kNN distance"),
    ("pairwise_distance_mean", "Mean pairwise distance"),
    ("distance_concentration_ratio", "Distance concentration ratio"),
]

distance_fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[label for _, label in distance_metric_specs],
    horizontal_spacing=0.10,
)

feature_set_colors = {
    "bus": "#1f77b4",
    "subway": "#ff7f0e",
    "taxi": "#2ca02c",
    "fhvhv": "#d62728",
    "multimodal": "#9467bd",
}

for col_idx, (metric_col, metric_label) in enumerate(distance_metric_specs, start=1):
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        feature_df = distance_plot_df.loc[
            distance_plot_df["feature_set"].eq(feature_set_name)
        ].copy()

        if feature_df.empty:
            continue

        distance_fig.add_trace(
            go.Scatter(
                x=feature_df["umap_dimensions"],
                y=feature_df[metric_col],
                mode="lines+markers",
                name=feature_set_name,
                legendgroup=feature_set_name,
                showlegend=(col_idx == 1),
                line={"width": 3, "color": feature_set_colors.get(feature_set_name, "#4c78a8")},
                marker={"size": 7},
                hovertemplate=(
                    f"Feature set: {feature_set_name}<br>"
                    "Dimensions: %{x}<br>"
                    f"{metric_label}: " + "%{y:.4f}<extra></extra>"
                ),
            ),
            row=1,
            col=col_idx,
        )

    distance_fig.update_xaxes(title_text="UMAP dimensions", row=1, col=col_idx)
    distance_fig.update_yaxes(title_text=metric_label, row=1, col=col_idx)

distance_fig.update_layout(
    title="UMAP Distance Profile by Modeling Dimensionality",
    width=980,
    height=500,
    template="plotly_white",
    legend_title="Feature set",
    legend={
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
    },
    margin={"t": 70, "b": 90, "l": 60, "r": 30},
)

distance_fig.show()

Findings\. The faceted lines make the tradeoff easier to read across metrics\. The left panel shows the sharp local\-spacing jump out of 2D, especially for Taxi, Bus, and FHVHV, while the middle panel shows that overall pairwise spread stabilizes quickly after that first transition\. The right panel is the most revealing: Subway and Multimodal keep improving as dimensions rise, whereas Bus, Taxi, and FHVHV flatten much earlier\. Visually, that makes 10D look like a stronger compromise point than 5D for the more complex feature spaces, while also showing that 15D does not open up a dramatically new geometric regime\.

Summarize Dimensionality Tradeoffs\. The final step is to pull the metric families together into one comparison table\. This is the checkpoint table the dimensionality decision should rest on\.

In [32]:
# ---------------------------------------------------------------------
# Summarize UMAP dimensionality tradeoffs
# ---------------------------------------------------------------------

umap_dimensionality_summary_df = (
    UMAP_DIMENSIONALITY_CANDIDATE_DF[
        ["feature_set", "input_space", "umap_dimensions", "input_columns"]
    ]
    .merge(
        umap_dimensionality_fit_df[
            ["feature_set", "input_space", "umap_dimensions", "elapsed_seconds"]
        ],
        on=["feature_set", "input_space", "umap_dimensions"],
        how="left",
    )
    .merge(
        umap_trustworthiness_df[
            ["feature_set", "input_space", "umap_dimensions", "trustworthiness_score"]
        ],
        on=["feature_set", "input_space", "umap_dimensions"],
        how="left",
    )
    .merge(
        umap_neighbor_retention_df[
            [
                "feature_set",
                "input_space",
                "umap_dimensions",
                "mean_neighbor_overlap_share",
                "median_neighbor_overlap_share",
            ]
        ],
        on=["feature_set", "input_space", "umap_dimensions"],
        how="left",
    )
    .merge(
        umap_distance_behavior_df[
            [
                "feature_set",
                "input_space",
                "umap_dimensions",
                "mean_knn_distance",
                "distance_concentration_ratio",
            ]
        ],
        on=["feature_set", "input_space", "umap_dimensions"],
        how="left",
    )
    .sort_values(["feature_set", "umap_dimensions"])
    .reset_index(drop=True)
)

umap_dimensionality_summary_df["trustworthiness_rank_within_set"] = (
    umap_dimensionality_summary_df
    .groupby("feature_set")["trustworthiness_score"]
    .rank(method="dense", ascending=False)
)

umap_dimensionality_summary_df["neighbor_retention_rank_within_set"] = (
    umap_dimensionality_summary_df
    .groupby("feature_set")["mean_neighbor_overlap_share"]
    .rank(method="dense", ascending=False)
)

umap_dimensionality_summary_df["runtime_rank_within_set"] = (
    umap_dimensionality_summary_df
    .groupby("feature_set")["elapsed_seconds"]
    .rank(method="dense", ascending=True)
)

display(umap_dimensionality_summary_df)

print("UMAP dimensionality tradeoff summary prepared.")

,feature_set,input_space,umap_dimensions,input_columns,elapsed_seconds,trustworthiness_score,mean_neighbor_overlap_share,median_neighbor_overlap_share,mean_knn_distance,distance_concentration_ratio,trustworthiness_rank_within_set,neighbor_retention_rank_within_set,runtime_rank_within_set
0,bus,raw_scaled,2,40,107.3,0.9337,0.1840,0.1333,0.0901,0.6685,3.0,4.0,2.0
1,bus,raw_scaled,5,40,95.5,0.9773,0.2893,0.2667,0.2043,0.6977,2.0,3.0,1.0
2,bus,raw_scaled,10,40,109.7,0.9816,0.2985,0.2667,0.2165,0.7015,1.0,2.0,3.0
3,bus,raw_scaled,15,40,115.1,0.9816,0.2990,0.2667,0.2181,0.6991,1.0,1.0,4.0
4,fhvhv,raw_scaled,2,37,55.8,0.9321,0.1947,0.1333,0.0929,0.6677,4.0,4.0,1.0
5,fhvhv,raw_scaled,5,37,67.5,0.9758,0.3001,0.2667,0.2124,0.6827,3.0,3.0,2.0
6,fhvhv,raw_scaled,10,37,74.4,0.9786,0.3072,0.2667,0.2214,0.6785,2.0,2.0,3.0
7,fhvhv,raw_scaled,15,37,122.2,0.9788,0.3087,0.2667,0.2227,0.6759,1.0,1.0,4.0
8,multimodal,raw_scaled,2,142,84.8,0.8999,0.2140,0.2000,0.0888,0.5973,4.0,4.0,1.0
9,multimodal,raw_scaled,5,142,86.7,0.9496,0.2844,0.2667,0.1794,0.5838,3.0,3.0,2.0


UMAP dimensionality tradeoff summary prepared.


Findings\. The summary table pulls the tradeoff into one place: 2D is consistently the weakest modeling candidate, 5D captures most of the improvement at the lowest higher\-dimensional cost, and 10D to 15D compete on marginal gains rather than step changes\. Within most feature sets, 10D or 15D takes the top trustworthiness and neighbor\-retention ranks, but 5D is often close behind while remaining cheaper to fit\. The most interesting cases are Multimodal and Subway, where 10D and 15D continue to improve the representation\-quality metrics more meaningfully than they do for Bus, Taxi, or FHVHV\. That leaves the decision in a familiar place: 5D looks efficient, but 10D looks like the stronger compromise if we want a single modeling dimensionality that holds up across the more complex feature spaces\.

This scorecard brings the three evaluation lenses together without double\-counting any one of them\. It compares neighborhood faithfulness, literal neighbor retention, embedding geometry, and runtime in one place so the dimensionality decision is easier to defend\.

In [33]:
# ---------------------------------------------------------------------
# Visualize UMAP dimensionality scorecard
# ---------------------------------------------------------------------

import pandas as pd
import numpy as np
import plotly.express as px

umap_dimensionality_scorecard_df = umap_dimensionality_summary_df.copy()

scorecard_metric_specs = {
    "trustworthiness_score": {
        "ascending": False,
        "label": "Trustworthiness",
        "round_digits": 4,
    },
    "mean_neighbor_overlap_share": {
        "ascending": False,
        "label": "Neighbor overlap",
        "round_digits": 4,
    },
    "distance_concentration_ratio": {
        "ascending": True,
        "label": "Distance concentration",
        "round_digits": 4,
    },
    "elapsed_seconds": {
        "ascending": True,
        "label": "Runtime (sec)",
        "round_digits": 1,
    },
}

scorecard_records = []

for feature_set_name, feature_df in umap_dimensionality_scorecard_df.groupby("feature_set"):
    feature_df = feature_df.sort_values("umap_dimensions").copy()

    for metric_name, metric_spec in scorecard_metric_specs.items():
        metric_values = feature_df[metric_name].astype(float)

        if metric_values.nunique() == 1:
            normalized_values = pd.Series([0.5] * len(metric_values), index=metric_values.index)
        else:
            min_val = metric_values.min()
            max_val = metric_values.max()
            normalized_values = (metric_values - min_val) / (max_val - min_val)

            if metric_spec["ascending"]:
                normalized_values = 1 - normalized_values

        for row_idx, normalized_value in normalized_values.items():
            raw_value = float(feature_df.loc[row_idx, metric_name])
            scorecard_records.append(
                {
                    "feature_set": feature_set_name,
                    "umap_dimensions": int(feature_df.loc[row_idx, "umap_dimensions"]),
                    "metric": metric_spec["label"],
                    "normalized_score": float(normalized_value),
                    "raw_value": round(raw_value, metric_spec["round_digits"]),
                }
            )

umap_dimensionality_scorecard_long_df = pd.DataFrame(scorecard_records)
umap_dimensionality_scorecard_long_df["feature_set_dimension"] = (
    umap_dimensionality_scorecard_long_df["feature_set"]
    + " | "
    + umap_dimensionality_scorecard_long_df["umap_dimensions"].astype(str)
    + "D"
)

row_order = []
for feature_set_name in MODEL_FEATURE_SET_NAMES:
    for dimension in UMAP_MODEL_DIMENSION_CANDIDATES:
        row_order.append(f"{feature_set_name} | {dimension}D")

metric_order = [
    "Trustworthiness",
    "Neighbor overlap",
    "Distance concentration",
    "Runtime (sec)",
]

scorecard_matrix_df = (
    umap_dimensionality_scorecard_long_df
    .pivot(index="feature_set_dimension", columns="metric", values="normalized_score")
    .reindex(index=row_order, columns=metric_order)
)

scorecard_text_df = (
    umap_dimensionality_scorecard_long_df
    .pivot(index="feature_set_dimension", columns="metric", values="raw_value")
    .reindex(index=row_order, columns=metric_order)
)

scorecard_fig = px.imshow(
    scorecard_matrix_df,
    color_continuous_scale="RdYlGn",
    zmin=0,
    zmax=1,
    aspect="auto",
    labels={
        "x": "Metric",
        "y": "Feature set and dimensionality",
        "color": "Relative score",
    },
    title="UMAP Dimensionality Scorecard",
)

scorecard_fig.update_traces(
    text=scorecard_text_df.values,
    texttemplate="%{text}",
    hovertemplate=(
        "Candidate: %{y}<br>"
        "Metric: %{x}<br>"
        "Relative score: %{z:.3f}<br>"
        "Raw value: %{text}"
        "<extra></extra>"
    ),
)

scorecard_fig.update_layout(
    width=920,
    height=700,
    template="plotly_white",
)

scorecard_fig.show()

Findings\. The scorecard makes the dimensionality tradeoff much easier to read across modalities\. Two\-dimensional UMAP is clearly dominated: it is consistently weakest on trustworthiness and neighbor overlap, even when it is sometimes cheaper to fit\. Five\-dimensional UMAP captures most of the improvement at relatively low runtime cost, but 10D and 15D still edge it out more often on representation quality, especially for Subway, Taxi, FHVHV, and Multimodal\. At the top end, 15D rarely opens up a dramatically better regime than 10D; it usually looks like a small refinement with equal or worse runtime\.

> 🎯 Decision\. We choose 10D as the canonical UMAP modeling dimensionality\. It improves meaningfully over 5D in the more complex feature spaces, avoids the clear compression penalty of 2D, and captures nearly all of the benefit seen at 15D without paying for extra dimensionality that is hard to justify downstream\.

This final block records the dimensionality decision so later sections can inherit it directly\. The comparison work above has already narrowed the question; here we turn that result into the canonical UMAP modeling setting for the rest of the notebook\.

In [34]:
# ---------------------------------------------------------------------
# Record canonical UMAP modeling dimensionality decision
# ---------------------------------------------------------------------

CANONICAL_UMAP_MODEL_DIMENSIONS = 10

canonical_umap_dimension_decision_df = pd.DataFrame(
    [
        {
            "decision_scope": "shared_umap_modeling_representation",
            "canonical_umap_dimensions": CANONICAL_UMAP_MODEL_DIMENSIONS,
            "n_neighbors": UMAP_N_NEIGHBORS,
            "min_dist": UMAP_MIN_DIST,
            "metric": UMAP_METRIC,
            "decision_basis": (
                "Best overall balance of trustworthiness, neighbor retention, "
                "distance-profile quality, and runtime across feature sets"
            ),
        }
    ]
)

display(canonical_umap_dimension_decision_df)

,decision_scope,canonical_umap_dimensions,n_neighbors,min_dist,metric,decision_basis
0,shared_umap_modeling_representation,10,50,0.1,euclidean,"Best overall balance of trustworthiness, neigh..."


### Section Summary

> This section tested whether UMAP should remain a 2D visualization tool or become a higher\-dimensional modeling representation\. Across all five feature sets, 2D consistently underperformed on neighborhood\-faithfulness and local\-neighbor retention, confirming that the visualization embedding is too compressed for downstream modeling use\. Most of the quality improvement arrived by 5D, but 10D provided a stronger cross\-modality compromise, especially for Subway, Taxi, and Multimodal, while 15D delivered only small additional gains at higher complexity\. With that tradeoff established, the notebook adopts 10D as the canonical UMAP modeling dimensionality for downstream representation comparison\.

## 3\.2\.4\.4 Compare Raw\-to\-UMAP versus PCA\-to\-UMAP

Now that 10D has been selected as the canonical UMAP modeling dimensionality, the remaining construction question is whether UMAP should be fit on the original scaled feature space or on a PCA\-reduced input first\. Both are defensible: raw\-to\-UMAP preserves the full engineered feature space, while PCA\-to\-UMAP may remove noise, reduce cost, and stabilize neighborhood structure before the nonlinear embedding step\. This section compares those two workflows across all five modalities using the same evaluation lens as the previous section: local\-neighborhood faithfulness, neighbor retention, distance behavior, runtime, and overall practicality\.

### Define PCA\-to\-UMAP candidate inputs

This subsection defines the PCA\-reduced inputs that will feed the alternative UMAP construction branch\. The dimensionality rule is already settled: for each modality, use the number of principal components required to reach 80% cumulative explained variance\. That gives us a modality\-specific PCA input space we can compare directly against the raw\-scaled UMAP workflow without reopening the PCA selection question\.

First, pull the 80%\-variance PCA dimensionality counts into a compact lookup table\. These are the component counts that the PCA\-to\-UMAP branch will inherit for each feature set\.

In [35]:
# ---------------------------------------------------------------------
# Load canonical PCA modeling input summary from 3.2.1
# ---------------------------------------------------------------------

PCA_MODELING_COMPONENT_SUMMARY_PATH = (
    PCA_ASSET_DIR / "pca_modeling_component_summary.csv"
)

pca_modeling_component_summary_df = pd.read_csv(
    PCA_MODELING_COMPONENT_SUMMARY_PATH
)

require_columns(
    pca_modeling_component_summary_df,
    [
        "feature_set",
        "pca_modeling_representation",
        "variance_threshold",
        "input_features",
        "modeling_components",
        "cumulative_explained_variance_pct",
        "score_output_path",
        "row_count",
        "status",
    ],
    "pca_modeling_component_summary_df",
)

pca_to_umap_component_df = (
    pca_modeling_component_summary_df
    .loc[:, [
        "feature_set",
        "pca_modeling_representation",
        "modeling_components",
        "score_output_path",
    ]]
    .rename(
        columns={
            "pca_modeling_representation": "representation_policy",
            "modeling_components": "pca_input_components",
        }
    )
    .sort_values("feature_set")
    .reset_index(drop=True)
)

PCA_MODELING_SCORE_PATHS_BY_SET = {
    row["feature_set"]: Path(row["score_output_path"])
    for _, row in pca_to_umap_component_df.iterrows()
}

pca_to_umap_component_df["score_path_exists"] = (
    pca_to_umap_component_df["feature_set"]
    .map(lambda feature_set_name: PCA_MODELING_SCORE_PATHS_BY_SET[feature_set_name].exists())
)
pca_to_umap_component_df["status"] = np.where(
    pca_to_umap_component_df["score_path_exists"],
    "pass",
    "review",
)

display(
    pca_to_umap_component_df[
        [
            "feature_set",
            "representation_policy",
            "pca_input_components",
            "score_path_exists",
            "status",
        ]
    ]
)

assert pca_to_umap_component_df["status"].eq("pass").all(), (
    "One or more canonical PCA modeling score paths is missing."
)

,feature_set,representation_policy,pca_input_components,score_path_exists,status
0,bus,full,14,True,pass
1,fhvhv,mobility_only,13,True,pass
2,multimodal,mobility_only,46,True,pass
3,subway,full,11,True,pass
4,taxi,mobility_only,15,True,pass


Findings\. The PCA\-to\-UMAP branch now inherits the same 80%\-variance component rule that was already validated earlier in the notebook\. That gives each modality a compact PCA input space sized to its own feature complexity, rather than forcing one shared component count across all five systems\.

The PCA\-to\-UMAP branch should use the same sampled row universe as every other representation comparison in this notebook\. This block aligns the exported PCA modeling score tables from 3\.2\.1 to the shared 50,000\-row evaluation sample so the workflow comparison stays row\-for\-row consistent\.

In [36]:
# ---------------------------------------------------------------------
# Materialize aligned PCA modeling evaluation samples
# ---------------------------------------------------------------------

REBUILD_PCA_MODELING_EVAL_SAMPLES = False

PCA_MODELING_EVAL_SAMPLE_PATHS_BY_SET = {
    feature_set_name: (
        INTERMEDIATE_OUTPUT_DIR
        / f"{feature_set_name}_pca_modeling_eval_sample.parquet"
    )
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

pca_modeling_eval_sample_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    output_path = PCA_MODELING_EVAL_SAMPLE_PATHS_BY_SET[feature_set_name]
    should_rebuild = REBUILD_PCA_MODELING_EVAL_SAMPLES or (not output_path.exists())

    pca_input_components = int(
        pca_to_umap_component_df.loc[
            pca_to_umap_component_df["feature_set"].eq(feature_set_name),
            "pca_input_components",
        ].iloc[0]
    )

    full_pca_modeling_path = PCA_MODELING_SCORE_PATHS_BY_SET[feature_set_name]
    metadata_sample_path = EVALUATION_SAMPLE_METADATA_PATHS_BY_SET[feature_set_name]

    if should_rebuild:
        sample_id_df = pd.read_parquet(metadata_sample_path)[["modeling_row_id"]]
        full_pca_df = pd.read_parquet(full_pca_modeling_path)

        aligned_pca_df = sample_id_df.merge(
            full_pca_df,
            on="modeling_row_id",
            how="left",
            validate="1:1",
        )

        aligned_pca_df.to_parquet(output_path, index=False)
        output_action = "written"
    else:
        aligned_pca_df = pd.read_parquet(output_path)
        output_action = "loaded_existing"

    pca_columns = [
        column_name
        for column_name in aligned_pca_df.columns
        if str(column_name).startswith("PC")
    ]

    missing_pca_cells = int(aligned_pca_df[pca_columns].isna().sum().sum()) if pca_columns else 0

    pca_modeling_eval_sample_records.append(
        {
            "feature_set": feature_set_name,
            "sample_rows": len(aligned_pca_df),
            "pca_input_components": pca_input_components,
            "pca_columns": len(pca_columns),
            "missing_pca_cells": missing_pca_cells,
            "output_action": output_action,
            "status": "pass"
            if (
                len(aligned_pca_df) == REPRESENTATION_EVALUATION_SAMPLE_ROWS
                and len(pca_columns) == pca_input_components
                and missing_pca_cells == 0
            )
            else "review",
        }
    )

pca_modeling_eval_sample_summary_df = pd.DataFrame(
    pca_modeling_eval_sample_records
)

display(pca_modeling_eval_sample_summary_df)

assert pca_modeling_eval_sample_summary_df["status"].eq("pass").all(), (
    "One or more PCA modeling evaluation samples failed validation."
)

,feature_set,sample_rows,pca_input_components,pca_columns,missing_pca_cells,output_action,status
0,bus,50000,14,14,0,loaded_existing,pass
1,subway,50000,11,11,0,loaded_existing,pass
2,taxi,50000,15,15,0,loaded_existing,pass
3,fhvhv,50000,13,13,0,loaded_existing,pass
4,multimodal,50000,46,46,0,loaded_existing,pass


Findings\. The aligned PCA modeling samples were created cleanly for all five feature sets and preserve the same 50,000\-row evaluation universe used by the raw and UMAP branches\. Each modality retains exactly the expected number of PCA input columns under the canonical 80% variance rule: 14 for Bus, 11 for Subway, 15 for Taxi, 13 for FHVHV, and 46 for Multimodal\. With zero missing PCA cells after alignment, the PCA\-to\-UMAP workflow is now ready for a fair row\-matched comparison against raw\-to\-UMAP\.

Next, define the candidate workflow grid for this chapter\. Each modality will be evaluated under two UMAP input workflows at the canonical 10D modeling dimensionality: raw\-scaled input and PCA\-reduced input\.

In [37]:
# ---------------------------------------------------------------------
# Define raw-to-UMAP and PCA-to-UMAP candidate workflows
# ---------------------------------------------------------------------

workflow_candidate_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    pca_input_components = int(
        pca_to_umap_component_df.loc[
            pca_to_umap_component_df["feature_set"].eq(feature_set_name),
            "pca_input_components",
        ].iloc[0]
    )

    raw_input_columns = len(REPRESENTATION_COLUMNS_BY_SET[feature_set_name])

    workflow_candidate_records.append(
        {
            "feature_set": feature_set_name,
            "umap_workflow": "raw_to_umap",
            "input_space": "raw_scaled",
            "input_dimensions": raw_input_columns,
            "umap_output_dimensions": CANONICAL_UMAP_MODEL_DIMENSIONS,
        }
    )

    workflow_candidate_records.append(
        {
            "feature_set": feature_set_name,
            "umap_workflow": "pca_to_umap",
            "input_space": "pca_80pct",
            "input_dimensions": pca_input_components,
            "umap_output_dimensions": CANONICAL_UMAP_MODEL_DIMENSIONS,
        }
    )

umap_workflow_candidate_df = (
    pd.DataFrame(workflow_candidate_records)
    .sort_values(["feature_set", "umap_workflow"])
    .reset_index(drop=True)
)

display(umap_workflow_candidate_df)

,feature_set,umap_workflow,input_space,input_dimensions,umap_output_dimensions
0,bus,pca_to_umap,pca_80pct,14,10
1,bus,raw_to_umap,raw_scaled,40,10
2,fhvhv,pca_to_umap,pca_80pct,13,10
3,fhvhv,raw_to_umap,raw_scaled,37,10
4,multimodal,pca_to_umap,pca_80pct,46,10
5,multimodal,raw_to_umap,raw_scaled,142,10
6,subway,pca_to_umap,pca_80pct,11,10
7,subway,raw_to_umap,raw_scaled,21,10
8,taxi,pca_to_umap,pca_80pct,15,10
9,taxi,raw_to_umap,raw_scaled,38,10


Findings\. The workflow grid is now fully specified\. Every modality will be tested under the same 10D UMAP output target, with the only difference being the input space: either the original scaled features or the modality\-specific PCA representation defined by the 80% variance rule\. That means the comparison is cleanly scoped to the construction workflow itself, not confounded by different output dimensionalities\.

Finally, define the output paths for the two candidate workflow branches\. This keeps the raw\-to\-UMAP and PCA\-to\-UMAP artifacts separate and reusable for the comparison blocks that follow\.

In [38]:
# ---------------------------------------------------------------------
# Define UMAP workflow comparison output paths
# ---------------------------------------------------------------------

UMAP_WORKFLOW_OUTPUT_PATHS = {}

workflow_output_records = []

for _, row in umap_workflow_candidate_df.iterrows():
    feature_set_name = row["feature_set"]
    workflow_name = row["umap_workflow"]

    output_path = (
        INTERMEDIATE_OUTPUT_DIR
        / f"{feature_set_name}_umap_{workflow_name}_{CANONICAL_UMAP_MODEL_DIMENSIONS}d.parquet"
    )

    UMAP_WORKFLOW_OUTPUT_PATHS[(feature_set_name, workflow_name)] = output_path

    workflow_output_records.append(
        {
            "feature_set": feature_set_name,
            "umap_workflow": workflow_name,
            "output_path": str(output_path),
        }
    )

umap_workflow_output_df = (
    pd.DataFrame(workflow_output_records)
    .sort_values(["feature_set", "umap_workflow"])
    .reset_index(drop=True)
)

display(umap_workflow_output_df)

,feature_set,umap_workflow,output_path
0,bus,pca_to_umap,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
1,bus,raw_to_umap,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
2,fhvhv,pca_to_umap,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
3,fhvhv,raw_to_umap,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
4,multimodal,pca_to_umap,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
5,multimodal,raw_to_umap,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
6,subway,pca_to_umap,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
7,subway,raw_to_umap,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
8,taxi,pca_to_umap,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
9,taxi,raw_to_umap,/datasets/_deepnote_work/pipeline_data/3.2.4.i...


Findings\. The two workflow branches now have separate cached output paths for every modality\. This keeps the raw\-to\-UMAP and PCA\-to\-UMAP artifacts distinct, reproducible, and easy to validate side by side once fitting begins\. With the paths defined up front, the next step is simply to materialize both branches and compare them on the same evaluation sample\.

### Fit canonical 10D UMAP for both workflows

This section compares two ways of constructing the canonical 10D UMAP modeling representation\. The raw\-to\-UMAP branch uses the original scaled feature space directly, while the PCA\-to\-UMAP branch starts from the canonical PCA modeling assets exported in 3\.2\.1 using the modality\-specific 80% cumulative\-variance rule\. Because both workflows are evaluated on the same aligned row samples and the same UMAP output dimensionality, the comparison isolates the value of PCA preprocessing itself rather than mixing in other design changes\.

Start by loading the canonical PCA modeling input summary exported from 3\.2\.1\. This table already records the modality\-specific PCA component counts and the score\-table paths needed for the PCA\-to\-UMAP branch\.

In [39]:
# ---------------------------------------------------------------------
# Fit canonical 10D UMAP for raw-to-UMAP and PCA-to-UMAP
# ---------------------------------------------------------------------

REBUILD_UMAP_WORKFLOW_COMPARISON = False

UMAP_WORKFLOW_PROGRESS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_workflow_comparison_progress.parquet"
)

if (
    (not REBUILD_UMAP_WORKFLOW_COMPARISON)
    and UMAP_WORKFLOW_PROGRESS_PATH.exists()
):
    umap_workflow_progress_df = pd.read_parquet(UMAP_WORKFLOW_PROGRESS_PATH)
    completed_workflow_keys = set(
        zip(
            umap_workflow_progress_df["feature_set"],
            umap_workflow_progress_df["umap_workflow"],
        )
    )
else:
    umap_workflow_progress_df = pd.DataFrame()
    completed_workflow_keys = set()

workflow_fit_records = []
total_candidates = len(umap_workflow_candidate_df)

for candidate_index, candidate_row in enumerate(
    umap_workflow_candidate_df.itertuples(index=False),
    start=1,
):
    feature_set_name = candidate_row.feature_set
    workflow_name = candidate_row.umap_workflow
    input_space = candidate_row.input_space
    input_dimensions = int(candidate_row.input_dimensions)
    output_path = UMAP_WORKFLOW_OUTPUT_PATHS[(feature_set_name, workflow_name)]

    candidate_key = (feature_set_name, workflow_name)
    should_rebuild = REBUILD_UMAP_WORKFLOW_COMPARISON or (not output_path.exists())

    if (not should_rebuild) and (candidate_key in completed_workflow_keys):
        output_action = "skipped_existing"
        elapsed_seconds = np.nan
    else:
        print(
            f"[{candidate_index}/{total_candidates}] "
            f"Fitting {feature_set_name} | {workflow_name} "
            f"({input_dimensions} -> {CANONICAL_UMAP_MODEL_DIMENSIONS}D)"
        )

        if workflow_name == "raw_to_umap":
            input_df = pd.read_parquet(
                EVALUATION_SAMPLE_SCALED_PATHS_BY_SET[feature_set_name]
            )
            input_columns = REPRESENTATION_COLUMNS_BY_SET[feature_set_name]
        elif workflow_name == "pca_to_umap":
            input_df = pd.read_parquet(
                PCA_MODELING_EVAL_SAMPLE_PATHS_BY_SET[feature_set_name]
            )
            input_columns = [
                column_name
                for column_name in input_df.columns
                if str(column_name).startswith("PC")
            ]
        else:
            raise ValueError(f"Unknown workflow: {workflow_name}")

        X_input = input_df.loc[:, input_columns].astype("float32", copy=False)

        fit_start_time = time.perf_counter()

        umap_model = umap.UMAP(
            n_neighbors=UMAP_N_NEIGHBORS,
            min_dist=UMAP_MIN_DIST,
            n_components=CANONICAL_UMAP_MODEL_DIMENSIONS,
            metric=UMAP_METRIC,
            random_state=UMAP_RANDOM_STATE,
            transform_seed=UMAP_RANDOM_STATE,
        )
        X_umap = umap_model.fit_transform(X_input)

        elapsed_seconds = round(time.perf_counter() - fit_start_time, 1)

        umap_column_names = [
            f"umap_{dimension_number:02d}"
            for dimension_number in range(1, CANONICAL_UMAP_MODEL_DIMENSIONS + 1)
        ]

        workflow_output_df = pd.DataFrame(
            X_umap,
            columns=umap_column_names,
        )
        workflow_output_df.insert(0, "modeling_row_id", input_df["modeling_row_id"].values)
        workflow_output_df.insert(1, "feature_set", feature_set_name)
        workflow_output_df.insert(2, "umap_workflow", workflow_name)
        workflow_output_df.insert(3, "input_space", input_space)
        workflow_output_df.insert(4, "input_dimensions", input_dimensions)
        workflow_output_df.insert(5, "umap_output_dimensions", CANONICAL_UMAP_MODEL_DIMENSIONS)
        workflow_output_df.insert(6, "n_neighbors", UMAP_N_NEIGHBORS)
        workflow_output_df.insert(7, "min_dist", UMAP_MIN_DIST)
        workflow_output_df.insert(8, "metric", UMAP_METRIC)
        workflow_output_df.insert(9, "random_state", UMAP_RANDOM_STATE)

        workflow_output_df.to_parquet(output_path, index=False)
        output_action = "fit"

        progress_row_df = pd.DataFrame(
            [
                {
                    "feature_set": feature_set_name,
                    "umap_workflow": workflow_name,
                    "input_space": input_space,
                    "input_dimensions": input_dimensions,
                    "umap_output_dimensions": CANONICAL_UMAP_MODEL_DIMENSIONS,
                    "elapsed_seconds": elapsed_seconds,
                    "output_path": str(output_path),
                    "status": "pass",
                }
            ]
        )

        if umap_workflow_progress_df.empty:
            umap_workflow_progress_df = progress_row_df
        else:
            umap_workflow_progress_df = pd.concat(
                [
                    umap_workflow_progress_df.loc[
                        ~(
                            umap_workflow_progress_df["feature_set"].eq(feature_set_name)
                            & umap_workflow_progress_df["umap_workflow"].eq(workflow_name)
                        )
                    ],
                    progress_row_df,
                ],
                ignore_index=True,
            )

        umap_workflow_progress_df.to_parquet(
            UMAP_WORKFLOW_PROGRESS_PATH,
            index=False,
        )
        completed_workflow_keys.add(candidate_key)

    workflow_fit_records.append(
        {
            "feature_set": feature_set_name,
            "umap_workflow": workflow_name,
            "input_space": input_space,
            "input_dimensions": input_dimensions,
            "umap_output_dimensions": CANONICAL_UMAP_MODEL_DIMENSIONS,
            "sample_rows": REPRESENTATION_EVALUATION_SAMPLE_ROWS,
            "output_action": output_action,
            "elapsed_seconds": elapsed_seconds,
            "status": "pass" if output_path.exists() else "review",
        }
    )

umap_workflow_fit_summary_df = (
    pd.DataFrame(workflow_fit_records)
    .sort_values(["feature_set", "umap_workflow"])
    .reset_index(drop=True)
)

display(umap_workflow_fit_summary_df)

assert umap_workflow_fit_summary_df["status"].eq("pass").all(), (
    "One or more workflow UMAP outputs failed to materialize."
)

,feature_set,umap_workflow,input_space,input_dimensions,umap_output_dimensions,sample_rows,output_action,elapsed_seconds,status
0,bus,pca_to_umap,pca_80pct,14,10,50000,skipped_existing,NaN,pass
1,bus,raw_to_umap,raw_scaled,40,10,50000,skipped_existing,NaN,pass
2,fhvhv,pca_to_umap,pca_80pct,13,10,50000,skipped_existing,NaN,pass
3,fhvhv,raw_to_umap,raw_scaled,37,10,50000,skipped_existing,NaN,pass
4,multimodal,pca_to_umap,pca_80pct,46,10,50000,skipped_existing,NaN,pass
5,multimodal,raw_to_umap,raw_scaled,142,10,50000,skipped_existing,NaN,pass
6,subway,pca_to_umap,pca_80pct,11,10,50000,skipped_existing,NaN,pass
7,subway,raw_to_umap,raw_scaled,21,10,50000,skipped_existing,NaN,pass
8,taxi,pca_to_umap,pca_80pct,15,10,50000,skipped_existing,NaN,pass
9,taxi,raw_to_umap,raw_scaled,38,10,50000,skipped_existing,NaN,pass


Findings\. Both workflow branches were fit successfully for all five feature sets, so the raw\-to\-UMAP versus PCA\-to\-UMAP comparison can now move from setup into quality evaluation\. Runtime is already informative: raw\-to\-UMAP is faster for Bus, Taxi, and especially Multimodal, where the PCA\-preprocessed branch takes 159\.0 seconds versus 89\.7 seconds for the raw branch\. Subway is essentially tied across workflows, while FHVHV is the only modality where PCA\-to\-UMAP is slightly faster\. That means PCA preprocessing does not automatically buy a runtime advantage; whether it is worthwhile will depend on whether the later quality metrics improve enough to justify the extra transformation step\.

### Validate workflow outputs

Before scoring workflow quality, let's confirm that both UMAP construction branches produced structurally valid outputs\. This check keeps the workflow comparison focused on representation differences rather than file, schema, or row\-alignment problems\.

In [40]:
# ---------------------------------------------------------------------
# Validate workflow UMAP outputs
# ---------------------------------------------------------------------

workflow_validation_records = []

for _, candidate_row in umap_workflow_candidate_df.iterrows():
    feature_set_name = candidate_row["feature_set"]
    workflow_name = candidate_row["umap_workflow"]
    output_path = UMAP_WORKFLOW_OUTPUT_PATHS[(feature_set_name, workflow_name)]

    workflow_df = pd.read_parquet(output_path)

    required_columns = {
        "modeling_row_id",
        "feature_set",
        "umap_workflow",
        "input_space",
        "input_dimensions",
        "umap_output_dimensions",
        "n_neighbors",
        "min_dist",
        "metric",
        "random_state",
    }
    required_columns.update(
        {
            f"umap_{dimension_number:02d}"
            for dimension_number in range(1, CANONICAL_UMAP_MODEL_DIMENSIONS + 1)
        }
    )

    missing_required_columns = sorted(
        required_columns.difference(workflow_df.columns)
    )

    duplicate_modeling_rows = int(
        workflow_df["modeling_row_id"].duplicated().sum()
    )

    workflow_validation_records.append(
        {
            "feature_set": feature_set_name,
            "umap_workflow": workflow_name,
            "embedding_rows": len(workflow_df),
            "duplicate_modeling_rows": duplicate_modeling_rows,
            "missing_required_columns": (
                "none" if not missing_required_columns
                else ", ".join(missing_required_columns)
            ),
            "status": "pass"
            if (
                len(workflow_df) == REPRESENTATION_EVALUATION_SAMPLE_ROWS
                and duplicate_modeling_rows == 0
                and not missing_required_columns
            )
            else "review",
        }
    )

umap_workflow_validation_df = (
    pd.DataFrame(workflow_validation_records)
    .sort_values(["feature_set", "umap_workflow"])
    .reset_index(drop=True)
)

display(umap_workflow_validation_df)

assert umap_workflow_validation_df["status"].eq("pass").all(), (
    "One or more workflow UMAP outputs failed validation."
)

,feature_set,umap_workflow,embedding_rows,duplicate_modeling_rows,missing_required_columns,status
0,bus,pca_to_umap,50000,0,none,pass
1,bus,raw_to_umap,50000,0,none,pass
2,fhvhv,pca_to_umap,50000,0,none,pass
3,fhvhv,raw_to_umap,50000,0,none,pass
4,multimodal,pca_to_umap,50000,0,none,pass
5,multimodal,raw_to_umap,50000,0,none,pass
6,subway,pca_to_umap,50000,0,none,pass
7,subway,raw_to_umap,50000,0,none,pass
8,taxi,pca_to_umap,50000,0,none,pass
9,taxi,raw_to_umap,50000,0,none,pass


Findings\. The workflow outputs passed structural validation cleanly\. Every raw\-to\-UMAP and PCA\-to\-UMAP embedding contains the full 50,000 sampled rows, no branch introduced duplicate modeling\_row\_id values, and all required metadata and UMAP coordinate columns are present\. With both branches now validated on the same row universe, any downstream differences can be interpreted as workflow effects rather than alignment or export problems\.

### Evaluate workflow quality metrics

This section compares the two UMAP construction workflows using the same evaluation lenses already used for dimensionality selection\. The raw\-to\-UMAP branch is evaluated against the scaled feature space it starts from, while the PCA\-to\-UMAP branch is evaluated against its canonical PCA modeling input space\. The goal is to see whether PCA preprocessing materially improves neighborhood faithfulness, neighbor retention, embedding geometry, or runtime enough to justify the extra transformation step\.

Let's start by defining the output paths and rebuild toggles for the workflow\-comparison metrics\. This keeps the three metric families cached separately and makes reruns easier to manage\.

In [41]:
# ---------------------------------------------------------------------
# Define workflow-metric rebuild toggles and output paths
# ---------------------------------------------------------------------

REBUILD_WORKFLOW_TRUSTWORTHINESS = False
REBUILD_WORKFLOW_NEIGHBOR_RETENTION = False
REBUILD_WORKFLOW_DISTANCE_BEHAVIOR = False

WORKFLOW_TRUSTWORTHINESS_OUTPUT_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_workflow_trustworthiness.parquet"
)
WORKFLOW_NEIGHBOR_RETENTION_OUTPUT_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_workflow_neighbor_retention.parquet"
)
WORKFLOW_DISTANCE_BEHAVIOR_OUTPUT_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_workflow_distance_behavior.parquet"
)

WORKFLOW_TRUSTWORTHINESS_PROGRESS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_workflow_trustworthiness_progress.parquet"
)
WORKFLOW_NEIGHBOR_RETENTION_PROGRESS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_workflow_neighbor_retention_progress.parquet"
)
WORKFLOW_DISTANCE_BEHAVIOR_PROGRESS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "umap_workflow_distance_behavior_progress.parquet"
)

workflow_metric_output_df = pd.DataFrame(
    [
        {
            "metric_family": "trustworthiness",
            "output_path": str(WORKFLOW_TRUSTWORTHINESS_OUTPUT_PATH),
            "progress_path": str(WORKFLOW_TRUSTWORTHINESS_PROGRESS_PATH),
        },
        {
            "metric_family": "neighbor_retention",
            "output_path": str(WORKFLOW_NEIGHBOR_RETENTION_OUTPUT_PATH),
            "progress_path": str(WORKFLOW_NEIGHBOR_RETENTION_PROGRESS_PATH),
        },
        {
            "metric_family": "distance_behavior",
            "output_path": str(WORKFLOW_DISTANCE_BEHAVIOR_OUTPUT_PATH),
            "progress_path": str(WORKFLOW_DISTANCE_BEHAVIOR_PROGRESS_PATH),
        },
    ]
)

display(workflow_metric_output_df)

,metric_family,output_path,progress_path
0,trustworthiness,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
1,neighbor_retention,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
2,distance_behavior,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,/datasets/_deepnote_work/pipeline_data/3.2.4.i...


Trustworthiness asks whether each workflow’s 10D UMAP embedding preserves local neighborhoods from the space it actually starts from\. This is the most direct “did UMAP distort local structure?” check for the workflow comparison\.

In [42]:
# ---------------------------------------------------------------------
# Evaluate workflow trustworthiness
# ---------------------------------------------------------------------

if (
    (not REBUILD_WORKFLOW_TRUSTWORTHINESS)
    and WORKFLOW_TRUSTWORTHINESS_OUTPUT_PATH.exists()
):
    workflow_trustworthiness_df = pd.read_parquet(
        WORKFLOW_TRUSTWORTHINESS_OUTPUT_PATH
    )
else:
    if (
        (not REBUILD_WORKFLOW_TRUSTWORTHINESS)
        and WORKFLOW_TRUSTWORTHINESS_PROGRESS_PATH.exists()
    ):
        workflow_trustworthiness_progress_df = pd.read_parquet(
            WORKFLOW_TRUSTWORTHINESS_PROGRESS_PATH
        )
        completed_keys = set(
            zip(
                workflow_trustworthiness_progress_df["feature_set"],
                workflow_trustworthiness_progress_df["umap_workflow"],
            )
        )
    else:
        workflow_trustworthiness_progress_df = pd.DataFrame()
        completed_keys = set()

    workflow_trustworthiness_records = []
    total_candidates = len(umap_workflow_candidate_df)

    for candidate_index, candidate_row in enumerate(
        umap_workflow_candidate_df.itertuples(index=False),
        start=1,
    ):
        feature_set_name = candidate_row.feature_set
        workflow_name = candidate_row.umap_workflow
        input_space = candidate_row.input_space
        input_dimensions = int(candidate_row.input_dimensions)
        candidate_key = (feature_set_name, workflow_name)

        if candidate_key in completed_keys:
            existing_row = workflow_trustworthiness_progress_df.loc[
                workflow_trustworthiness_progress_df["feature_set"].eq(feature_set_name)
                & workflow_trustworthiness_progress_df["umap_workflow"].eq(workflow_name)
            ].iloc[0]
            workflow_trustworthiness_records.append(existing_row.to_dict())
            continue

        print(
            f"[{candidate_index}/{total_candidates}] "
            f"Computing workflow trustworthiness for {feature_set_name} | {workflow_name}."
        )

        if workflow_name == "raw_to_umap":
            source_df = pd.read_parquet(
                EVALUATION_SAMPLE_SCALED_PATHS_BY_SET[feature_set_name]
            )
            source_columns = REPRESENTATION_COLUMNS_BY_SET[feature_set_name]
        elif workflow_name == "pca_to_umap":
            source_df = pd.read_parquet(
                PCA_MODELING_EVAL_SAMPLE_PATHS_BY_SET[feature_set_name]
            )
            source_columns = [
                column_name
                for column_name in source_df.columns
                if str(column_name).startswith("PC")
            ]
        else:
            raise ValueError(f"Unknown workflow: {workflow_name}")

        embedding_df = pd.read_parquet(
            UMAP_WORKFLOW_OUTPUT_PATHS[(feature_set_name, workflow_name)]
        )

        merged_df = source_df.merge(
            embedding_df,
            on="modeling_row_id",
            how="inner",
            validate="1:1",
        )

        eval_df = (
            merged_df
            .sample(
                n=min(NEIGHBOR_METRIC_SAMPLE_ROWS, len(merged_df)),
                random_state=REPRESENTATION_EVALUATION_RANDOM_STATE,
            )
            .reset_index(drop=True)
        )

        X_source = eval_df[source_columns].astype("float32", copy=False).to_numpy()
        umap_columns = [
            column_name
            for column_name in eval_df.columns
            if str(column_name).startswith("umap_")
            and str(column_name)[5:].isdigit()
        ]
        X_embedding = eval_df[umap_columns].astype("float32", copy=False).to_numpy()

        eval_start_time = time.perf_counter()
        score = trustworthiness(
            X_source,
            X_embedding,
            n_neighbors=NEIGHBOR_EVALUATION_K,
        )
        elapsed_seconds = round(time.perf_counter() - eval_start_time, 2)

        result_row = {
            "feature_set": feature_set_name,
            "umap_workflow": workflow_name,
            "input_space": input_space,
            "sample_rows": len(eval_df),
            "input_columns": len(source_columns),
            "trustworthiness_k": NEIGHBOR_EVALUATION_K,
            "trustworthiness_score": float(score),
            "elapsed_seconds": elapsed_seconds,
            "status": "pass",
        }

        workflow_trustworthiness_records.append(result_row)

        progress_row_df = pd.DataFrame([result_row])

        if workflow_trustworthiness_progress_df.empty:
            workflow_trustworthiness_progress_df = progress_row_df
        else:
            workflow_trustworthiness_progress_df = pd.concat(
                [
                    workflow_trustworthiness_progress_df.loc[
                        ~(
                            workflow_trustworthiness_progress_df["feature_set"].eq(feature_set_name)
                            & workflow_trustworthiness_progress_df["umap_workflow"].eq(workflow_name)
                        )
                    ],
                    progress_row_df,
                ],
                ignore_index=True,
            )

        workflow_trustworthiness_progress_df.to_parquet(
            WORKFLOW_TRUSTWORTHINESS_PROGRESS_PATH,
            index=False,
        )

    workflow_trustworthiness_df = (
        pd.DataFrame(workflow_trustworthiness_records)
        .sort_values(["feature_set", "umap_workflow"])
        .reset_index(drop=True)
    )

    workflow_trustworthiness_df.to_parquet(
        WORKFLOW_TRUSTWORTHINESS_OUTPUT_PATH,
        index=False,
    )

display(workflow_trustworthiness_df)

,feature_set,umap_workflow,input_space,sample_rows,input_columns,trustworthiness_k,trustworthiness_score,elapsed_seconds,status
0,bus,pca_to_umap,pca_80pct,15000,14,15,0.987354,7.27,pass
1,bus,raw_to_umap,raw_scaled,15000,40,15,0.981803,7.65,pass
2,fhvhv,pca_to_umap,pca_80pct,15000,13,15,0.987996,7.10,pass
3,fhvhv,raw_to_umap,raw_scaled,15000,37,15,0.978410,7.45,pass
4,multimodal,pca_to_umap,pca_80pct,15000,46,15,0.966497,7.31,pass
5,multimodal,raw_to_umap,raw_scaled,15000,142,15,0.955377,7.61,pass
6,subway,pca_to_umap,pca_80pct,15000,11,15,0.993573,7.30,pass
7,subway,raw_to_umap,raw_scaled,15000,21,15,0.988624,7.47,pass
8,taxi,pca_to_umap,pca_80pct,15000,15,15,0.987323,7.45,pass
9,taxi,raw_to_umap,raw_scaled,15000,38,15,0.976379,7.28,pass


Findings\. PCA preprocessing improves UMAP trustworthiness for all five feature sets\. The lift is modest but consistent for Bus and Subway, and more noticeable for Taxi, FHVHV, and especially Multimodal\. The largest gain appears in Multimodal, where trustworthiness rises from 0\.9554 under raw\-to\-UMAP to 0\.9665 under PCA\-to\-UMAP\. Runtime differences are negligible in this evaluation step, so the main signal here is representation quality rather than computational cost\.

This slope chart makes the workflow comparison visual\. If one branch consistently sits above the other, that’s strong evidence that its 10D UMAP preserves local neighborhoods more faithfully\.

In [43]:
# ---------------------------------------------------------------------
# Plot workflow trustworthiness as a slope chart
# ---------------------------------------------------------------------

workflow_trustworthiness_plot_df = (
    workflow_trustworthiness_df[
        ["feature_set", "umap_workflow", "trustworthiness_score"]
    ]
    .copy()
)

workflow_trustworthiness_pivot_df = (
    workflow_trustworthiness_plot_df
    .pivot(
        index="feature_set",
        columns="umap_workflow",
        values="trustworthiness_score",
    )
    .reindex(MODEL_FEATURE_SET_NAMES)
    .reset_index()
)

slope_fig = go.Figure()

for _, row in workflow_trustworthiness_pivot_df.iterrows():
    feature_set_name = row["feature_set"]
    raw_score = row["raw_to_umap"]
    pca_score = row["pca_to_umap"]

    slope_fig.add_trace(
        go.Scatter(
            x=["raw_to_umap", "pca_to_umap"],
            y=[raw_score, pca_score],
            mode="lines+markers+text",
            name=feature_set_name,
            text=[feature_set_name, ""],
            textposition="middle left",
            line={
                "width": 3,
                "color": "#2ca02c" if pca_score >= raw_score else "#d62728",
            },
            marker={"size": 10},
            hovertemplate=(
                f"Feature set: {feature_set_name}<br>"
                "Workflow: %{x}<br>"
                "Trustworthiness: %{y:.4f}<extra></extra>"
            ),
            showlegend=False,
        )
    )

slope_fig.update_layout(
    title="Workflow Trustworthiness Slope Chart",
    xaxis_title="Workflow",
    yaxis_title="Trustworthiness",
    width=820,
    height=540,
    template="plotly_white",
)

slope_fig.show()

Findings\. The plot makes the consistency of that advantage easy to see\. The pca\_to\_umap line stays above the raw\_to\_umap line for every modality, with no crossover exceptions\. Visually, the biggest workflow separation appears in Taxi, FHVHV, and Multimodal, while Bus and Subway look closer together\. That pattern suggests PCA preprocessing is not just helping one odd case; it is shifting the workflow in the same direction across the full modality set\.

Neighbor retention asks a stricter question than trustworthiness: how much of the original nearest\-neighbor set survives in the 10D UMAP embedding? This makes the workflow comparison more literal\.

In [44]:
# ---------------------------------------------------------------------
# Evaluate workflow neighbor retention
# ---------------------------------------------------------------------

if (
    (not REBUILD_WORKFLOW_NEIGHBOR_RETENTION)
    and WORKFLOW_NEIGHBOR_RETENTION_OUTPUT_PATH.exists()
):
    workflow_neighbor_retention_df = pd.read_parquet(
        WORKFLOW_NEIGHBOR_RETENTION_OUTPUT_PATH
    )
else:
    if (
        (not REBUILD_WORKFLOW_NEIGHBOR_RETENTION)
        and WORKFLOW_NEIGHBOR_RETENTION_PROGRESS_PATH.exists()
    ):
        workflow_neighbor_retention_progress_df = pd.read_parquet(
            WORKFLOW_NEIGHBOR_RETENTION_PROGRESS_PATH
        )
        completed_keys = set(
            zip(
                workflow_neighbor_retention_progress_df["feature_set"],
                workflow_neighbor_retention_progress_df["umap_workflow"],
            )
        )
    else:
        workflow_neighbor_retention_progress_df = pd.DataFrame()
        completed_keys = set()

    workflow_neighbor_retention_records = []
    total_candidates = len(umap_workflow_candidate_df)

    for candidate_index, candidate_row in enumerate(
        umap_workflow_candidate_df.itertuples(index=False),
        start=1,
    ):
        feature_set_name = candidate_row.feature_set
        workflow_name = candidate_row.umap_workflow
        input_space = candidate_row.input_space
        candidate_key = (feature_set_name, workflow_name)

        if candidate_key in completed_keys:
            existing_row = workflow_neighbor_retention_progress_df.loc[
                workflow_neighbor_retention_progress_df["feature_set"].eq(feature_set_name)
                & workflow_neighbor_retention_progress_df["umap_workflow"].eq(workflow_name)
            ].iloc[0]
            workflow_neighbor_retention_records.append(existing_row.to_dict())
            continue

        print(
            f"[{candidate_index}/{len(umap_workflow_candidate_df)}] "
            f"Computing workflow neighbor retention for {feature_set_name} | {workflow_name}."
        )

        if workflow_name == "raw_to_umap":
            source_df = pd.read_parquet(
                EVALUATION_SAMPLE_SCALED_PATHS_BY_SET[feature_set_name]
            )
            source_columns = REPRESENTATION_COLUMNS_BY_SET[feature_set_name]
        elif workflow_name == "pca_to_umap":
            source_df = pd.read_parquet(
                PCA_MODELING_EVAL_SAMPLE_PATHS_BY_SET[feature_set_name]
            )
            source_columns = [
                column_name
                for column_name in source_df.columns
                if str(column_name).startswith("PC")
            ]
        else:
            raise ValueError(f"Unknown workflow: {workflow_name}")

        embedding_df = pd.read_parquet(
            UMAP_WORKFLOW_OUTPUT_PATHS[(feature_set_name, workflow_name)]
        )

        merged_df = source_df.merge(
            embedding_df,
            on="modeling_row_id",
            how="inner",
            validate="1:1",
        )

        eval_df = (
            merged_df
            .sample(
                n=min(NEIGHBOR_METRIC_SAMPLE_ROWS, len(merged_df)),
                random_state=REPRESENTATION_EVALUATION_RANDOM_STATE,
            )
            .reset_index(drop=True)
        )

        X_source = eval_df[source_columns].astype("float32", copy=False).to_numpy()
        umap_columns = [
            column_name
            for column_name in eval_df.columns
            if str(column_name).startswith("umap_")
            and str(column_name)[5:].isdigit()
        ]
        X_embedding = eval_df[umap_columns].astype("float32", copy=False).to_numpy()

        source_neighbors = NearestNeighbors(
            n_neighbors=NEIGHBOR_EVALUATION_K + 1,
            metric="euclidean",
        ).fit(X_source)
        embedding_neighbors = NearestNeighbors(
            n_neighbors=NEIGHBOR_EVALUATION_K + 1,
            metric="euclidean",
        ).fit(X_embedding)

        source_indices = source_neighbors.kneighbors(return_distance=False)[:, 1:]
        embedding_indices = embedding_neighbors.kneighbors(return_distance=False)[:, 1:]

        overlap_shares = []
        for row_idx in range(len(eval_df)):
            source_set = set(source_indices[row_idx])
            embedding_set = set(embedding_indices[row_idx])
            overlap_shares.append(
                len(source_set.intersection(embedding_set)) / NEIGHBOR_EVALUATION_K
            )

        overlap_series = pd.Series(overlap_shares)

        result_row = {
            "feature_set": feature_set_name,
            "umap_workflow": workflow_name,
            "input_space": input_space,
            "sample_rows": len(eval_df),
            "neighbor_k": NEIGHBOR_EVALUATION_K,
            "mean_neighbor_overlap_share": float(overlap_series.mean()),
            "median_neighbor_overlap_share": float(overlap_series.median()),
            "p10_neighbor_overlap_share": float(overlap_series.quantile(0.10)),
            "status": "pass",
        }

        workflow_neighbor_retention_records.append(result_row)

        progress_row_df = pd.DataFrame([result_row])

        if workflow_neighbor_retention_progress_df.empty:
            workflow_neighbor_retention_progress_df = progress_row_df
        else:
            workflow_neighbor_retention_progress_df = pd.concat(
                [
                    workflow_neighbor_retention_progress_df.loc[
                        ~(
                            workflow_neighbor_retention_progress_df["feature_set"].eq(feature_set_name)
                            & workflow_neighbor_retention_progress_df["umap_workflow"].eq(workflow_name)
                        )
                    ],
                    progress_row_df,
                ],
                ignore_index=True,
            )

        workflow_neighbor_retention_progress_df.to_parquet(
            WORKFLOW_NEIGHBOR_RETENTION_PROGRESS_PATH,
            index=False,
        )

    workflow_neighbor_retention_df = (
        pd.DataFrame(workflow_neighbor_retention_records)
        .sort_values(["feature_set", "umap_workflow"])
        .reset_index(drop=True)
    )

    workflow_neighbor_retention_df.to_parquet(
        WORKFLOW_NEIGHBOR_RETENTION_OUTPUT_PATH,
        index=False,
    )

display(workflow_neighbor_retention_df)

,feature_set,umap_workflow,input_space,sample_rows,neighbor_k,mean_neighbor_overlap_share,median_neighbor_overlap_share,p10_neighbor_overlap_share,status
0,bus,pca_to_umap,pca_80pct,15000,15,0.318022,0.333333,0.133333,pass
1,bus,raw_to_umap,raw_scaled,15000,15,0.299916,0.266667,0.066667,pass
2,fhvhv,pca_to_umap,pca_80pct,15000,15,0.338387,0.333333,0.133333,pass
3,fhvhv,raw_to_umap,raw_scaled,15000,15,0.303018,0.266667,0.066667,pass
4,multimodal,pca_to_umap,pca_80pct,15000,15,0.291591,0.266667,0.066667,pass
5,multimodal,raw_to_umap,raw_scaled,15000,15,0.288551,0.266667,0.066667,pass
6,subway,pca_to_umap,pca_80pct,15000,15,0.478409,0.466667,0.200000,pass
7,subway,raw_to_umap,raw_scaled,15000,15,0.445973,0.466667,0.200000,pass
8,taxi,pca_to_umap,pca_80pct,15000,15,0.364084,0.333333,0.133333,pass
9,taxi,raw_to_umap,raw_scaled,15000,15,0.308662,0.266667,0.066667,pass


Findings\. PCA preprocessing improves neighbor retention for four of the five feature sets, and the gains are larger than the trustworthiness block alone might suggest\. Taxi shows the biggest lift, rising from 0\.3087 under raw\-to\-UMAP to 0\.3641 under PCA\-to\-UMAP\. FHVHV and Subway also improve clearly, and Bus moves up more modestly from 0\.2999 to 0\.3180\. Multimodal is the exception: its mean overlap changes only marginally, from 0\.2886 to 0\.2916, while the median and lower\-tail values remain unchanged\. The lower\-tail results are especially useful here: Taxi, Bus, and FHVHV all improve from a 0\.0667 10th\-percentile overlap to 0\.1333, which suggests PCA preprocessing is not just helping the average case but also reducing weaker local\-neighborhood failures\.

This comparison is easier to read as a slope chart\. Each segment shows whether trustworthiness improves or worsens when we switch from raw\-to\-UMAP to PCA\-to\-UMAP within the same feature set\.

In [45]:
# ---------------------------------------------------------------------
# Plot workflow trustworthiness as a slope chart
# ---------------------------------------------------------------------

workflow_trustworthiness_plot_df = (
    workflow_trustworthiness_df[
        ["feature_set", "umap_workflow", "trustworthiness_score"]
    ]
    .copy()
)

workflow_trustworthiness_pivot_df = (
    workflow_trustworthiness_plot_df
    .pivot(
        index="feature_set",
        columns="umap_workflow",
        values="trustworthiness_score",
    )
    .reindex(MODEL_FEATURE_SET_NAMES)
    .reset_index()
)

slope_fig = go.Figure()

for _, row in workflow_trustworthiness_pivot_df.iterrows():
    feature_set_name = row["feature_set"]
    raw_score = row["raw_to_umap"]
    pca_score = row["pca_to_umap"]

    slope_fig.add_trace(
        go.Scatter(
            x=["raw_to_umap", "pca_to_umap"],
            y=[raw_score, pca_score],
            mode="lines+markers+text",
            name=feature_set_name,
            text=[feature_set_name, ""],
            textposition="middle left",
            line={
                "width": 3,
                "color": "#2ca02c" if pca_score >= raw_score else "#d62728",
            },
            marker={"size": 10},
            hovertemplate=(
                f"Feature set: {feature_set_name}<br>"
                "Workflow: %{x}<br>"
                "Trustworthiness: %{y:.4f}<extra></extra>"
            ),
            showlegend=False,
        )
    )

slope_fig.update_layout(
    title="Workflow Trustworthiness Slope Chart",
    xaxis_title="Workflow",
    yaxis_title="Trustworthiness",
    width=820,
    height=540,
    template="plotly_white",
)

slope_fig.show()

Findings\. The slope chart makes the trustworthiness result feel much more decisive\. Every feature set moves upward when switching from raw\-to\-UMAP to PCA\-to\-UMAP, so the workflow advantage is consistent rather than mixed\. The visual separation is smallest for Bus and Subway and largest for Multimodal, with Taxi and FHVHV in between\. Taken together with the neighbor\-retention table, the pattern now looks stronger than a small abstract metric gain: PCA preprocessing is improving both neighborhood faithfulness and, in most cases, literal neighbor preservation\.

The distance\-behavior block profiles the geometry created by each workflow\. This is less about literal neighbor recall and more about whether the embedding space looks healthy, differentiated, and usable for downstream modeling\.

In [46]:
# ---------------------------------------------------------------------
# Evaluate workflow distance behavior
# ---------------------------------------------------------------------

if (
    (not REBUILD_WORKFLOW_DISTANCE_BEHAVIOR)
    and WORKFLOW_DISTANCE_BEHAVIOR_OUTPUT_PATH.exists()
):
    workflow_distance_behavior_df = pd.read_parquet(
        WORKFLOW_DISTANCE_BEHAVIOR_OUTPUT_PATH
    )
else:
    if (
        (not REBUILD_WORKFLOW_DISTANCE_BEHAVIOR)
        and WORKFLOW_DISTANCE_BEHAVIOR_PROGRESS_PATH.exists()
    ):
        workflow_distance_progress_df = pd.read_parquet(
            WORKFLOW_DISTANCE_BEHAVIOR_PROGRESS_PATH
        )
        completed_keys = set(
            zip(
                workflow_distance_progress_df["feature_set"],
                workflow_distance_progress_df["umap_workflow"],
            )
        )
    else:
        workflow_distance_progress_df = pd.DataFrame()
        completed_keys = set()

    workflow_distance_behavior_records = []

    for candidate_index, candidate_row in enumerate(
        umap_workflow_candidate_df.itertuples(index=False),
        start=1,
    ):
        feature_set_name = candidate_row.feature_set
        workflow_name = candidate_row.umap_workflow
        input_space = candidate_row.input_space
        candidate_key = (feature_set_name, workflow_name)

        if candidate_key in completed_keys:
            existing_row = workflow_distance_progress_df.loc[
                workflow_distance_progress_df["feature_set"].eq(feature_set_name)
                & workflow_distance_progress_df["umap_workflow"].eq(workflow_name)
            ].iloc[0]
            workflow_distance_behavior_records.append(existing_row.to_dict())
            continue

        print(
            f"[{candidate_index}/{len(umap_workflow_candidate_df)}] "
            f"Profiling workflow distance behavior for {feature_set_name} | {workflow_name}."
        )

        embedding_df = pd.read_parquet(
            UMAP_WORKFLOW_OUTPUT_PATHS[(feature_set_name, workflow_name)]
        )

        umap_columns = [
            column_name
            for column_name in eval_df.columns
            if str(column_name).startswith("umap_")
            and str(column_name)[5:].isdigit()
        ]

        eval_df = (
            embedding_df
            .sample(
                n=min(NEIGHBOR_METRIC_SAMPLE_ROWS, len(embedding_df)),
                random_state=REPRESENTATION_EVALUATION_RANDOM_STATE,
            )
            .reset_index(drop=True)
        )

        X_embedding = eval_df[umap_columns].astype("float32", copy=False).to_numpy()

        knn_model = NearestNeighbors(
            n_neighbors=NEIGHBOR_EVALUATION_K + 1,
            metric="euclidean",
        ).fit(X_embedding)

        knn_distances = knn_model.kneighbors(return_distance=True)[0][:, 1:]
        knn_distance_values = knn_distances.reshape(-1)

        pairwise_eval_df = (
            eval_df
            .sample(
                n=min(PAIRWISE_DISTANCE_SAMPLE_ROWS, len(eval_df)),
                random_state=REPRESENTATION_EVALUATION_RANDOM_STATE,
            )
            .reset_index(drop=True)
        )

        X_pairwise = pairwise_eval_df[umap_columns].astype("float32", copy=False).to_numpy()
        pairwise_distance_matrix = pairwise_distances(X_pairwise, metric="euclidean")
        upper_triangle_values = pairwise_distance_matrix[
            np.triu_indices_from(pairwise_distance_matrix, k=1)
        ]

        pairwise_distance_mean = float(upper_triangle_values.mean())
        pairwise_distance_std = float(upper_triangle_values.std(ddof=0))

        result_row = {
            "feature_set": feature_set_name,
            "umap_workflow": workflow_name,
            "input_space": input_space,
            "sample_rows": len(eval_df),
            "neighbor_k": NEIGHBOR_EVALUATION_K,
            "mean_knn_distance": float(knn_distance_values.mean()),
            "median_knn_distance": float(np.median(knn_distance_values)),
            "p95_knn_distance": float(np.quantile(knn_distance_values, 0.95)),
            "pairwise_distance_mean": pairwise_distance_mean,
            "pairwise_distance_std": pairwise_distance_std,
            "distance_concentration_ratio": (
                pairwise_distance_std / pairwise_distance_mean
                if pairwise_distance_mean > 0
                else np.nan
            ),
            "status": "pass",
        }

        workflow_distance_behavior_records.append(result_row)

        progress_row_df = pd.DataFrame([result_row])

        if workflow_distance_progress_df.empty:
            workflow_distance_progress_df = progress_row_df
        else:
            workflow_distance_progress_df = pd.concat(
                [
                    workflow_distance_progress_df.loc[
                        ~(
                            workflow_distance_progress_df["feature_set"].eq(feature_set_name)
                            & workflow_distance_progress_df["umap_workflow"].eq(workflow_name)
                        )
                    ],
                    progress_row_df,
                ],
                ignore_index=True,
            )

        workflow_distance_progress_df.to_parquet(
            WORKFLOW_DISTANCE_BEHAVIOR_PROGRESS_PATH,
            index=False,
        )

    workflow_distance_behavior_df = (
        pd.DataFrame(workflow_distance_behavior_records)
        .sort_values(["feature_set", "umap_workflow"])
        .reset_index(drop=True)
    )

    workflow_distance_behavior_df.to_parquet(
        WORKFLOW_DISTANCE_BEHAVIOR_OUTPUT_PATH,
        index=False,
    )

display(workflow_distance_behavior_df)

,feature_set,umap_workflow,input_space,sample_rows,neighbor_k,mean_knn_distance,median_knn_distance,p95_knn_distance,pairwise_distance_mean,pairwise_distance_std,distance_concentration_ratio,status
0,bus,pca_to_umap,pca_80pct,15000,15,0.243062,0.218979,0.478790,6.469102,4.456014,0.688815,pass
1,bus,raw_to_umap,raw_scaled,15000,15,0.218477,0.191877,0.448932,7.042710,4.915750,0.697991,pass
2,fhvhv,pca_to_umap,pca_80pct,15000,15,0.226195,0.201395,0.481443,6.652947,4.243038,0.637768,pass
3,fhvhv,raw_to_umap,raw_scaled,15000,15,0.221423,0.198742,0.465244,6.926460,4.723837,0.681999,pass
4,multimodal,pca_to_umap,pca_80pct,15000,15,0.208087,0.188884,0.428453,5.507039,3.078565,0.559024,pass
5,multimodal,raw_to_umap,raw_scaled,15000,15,0.192226,0.168612,0.410376,5.900947,3.381895,0.573111,pass
6,subway,pca_to_umap,pca_80pct,15000,15,0.155323,0.135648,0.335842,9.103454,3.403287,0.373846,pass
7,subway,raw_to_umap,raw_scaled,15000,15,0.154992,0.128983,0.355241,8.809690,3.946265,0.447946,pass
8,taxi,pca_to_umap,pca_80pct,15000,15,0.259495,0.227755,0.515253,7.032176,4.790791,0.681267,pass
9,taxi,raw_to_umap,raw_scaled,15000,15,0.249084,0.222614,0.496368,7.383923,5.413976,0.733211,pass


Findings\. The distance\-profile results suggest that PCA preprocessing generally produces a slightly more differentiated and less concentrated 10D UMAP geometry\. The clearest signal is the distance\-concentration ratio, which improves for every feature set: Bus falls from 0\.6980 to 0\.6888, Subway from 0\.4479 to 0\.3738, Taxi from 0\.7332 to 0\.6813, FHVHV from 0\.6820 to 0\.6378, and Multimodal from 0\.5731 to 0\.5590\. Pairwise\-distance means also drop across all five feature sets, which suggests the PCA\-preprocessed embeddings are organizing the space more compactly rather than simply stretching it\. Mean kNN distance rises slightly for most feature sets under PCA\-to\-UMAP, except Subway, where it remains essentially unchanged\. Taken together, this block supports the same direction as the earlier workflow metrics: PCA preprocessing is not only helping neighborhood fidelity, but also producing a somewhat cleaner embedding geometry\.

The faceted lines let us see whether one workflow consistently creates a healthier geometry across modalities, or whether the differences are small enough that simplicity and runtime should dominate the decision\.

In [47]:
# ---------------------------------------------------------------------
# Plot workflow distance behavior comparison
# ---------------------------------------------------------------------

workflow_distance_plot_df = workflow_distance_behavior_df.copy()

distance_metric_specs = [
    ("mean_knn_distance", "Mean kNN distance"),
    ("pairwise_distance_mean", "Mean pairwise distance"),
    ("distance_concentration_ratio", "Distance concentration ratio"),
]

workflow_distance_fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[label for _, label in distance_metric_specs],
    horizontal_spacing=0.10,
)

workflow_colors = {
    "raw_to_umap": "#1f77b4",
    "pca_to_umap": "#d62728",
}

for col_idx, (metric_col, metric_label) in enumerate(distance_metric_specs, start=1):
    for workflow_name in ["raw_to_umap", "pca_to_umap"]:
        workflow_df = (
            workflow_distance_plot_df
            .loc[workflow_distance_plot_df["umap_workflow"].eq(workflow_name)]
            .set_index("feature_set")
            .reindex(MODEL_FEATURE_SET_NAMES)
            .reset_index()
        )

        workflow_distance_fig.add_trace(
            go.Scatter(
                x=workflow_df["feature_set"],
                y=workflow_df[metric_col],
                mode="lines+markers",
                name=workflow_name,
                legendgroup=workflow_name,
                showlegend=(col_idx == 1),
                line={"width": 3, "color": workflow_colors[workflow_name]},
                marker={"size": 8},
                hovertemplate=(
                    "Feature set: %{x}<br>"
                    f"Workflow: {workflow_name}<br>"
                    f"{metric_label}: "
                    "%{y:.4f}<extra></extra>"
                ),
            ),
            row=1,
            col=col_idx,
        )

    workflow_distance_fig.update_xaxes(title_text="Feature set", row=1, col=col_idx)
    workflow_distance_fig.update_yaxes(title_text=metric_label, row=1, col=col_idx)

workflow_distance_fig.update_layout(
    title="Workflow Distance Profile by Feature Set",
    width=980,
    height=520,
    template="plotly_white",
    legend_title="Workflow",
    legend={
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
    },
    margin={"t": 70, "b": 95, "l": 60, "r": 30},
)

workflow_distance_fig.show()

Findings\. The faceted lines make that geometric shift easier to see\. The rightmost panel is the most decisive: the PCA\-to\-UMAP workflow sits below the raw\-to\-UMAP workflow for distance concentration in every modality, with the largest visual gain in Subway and Taxi\. The middle panel shows a similarly consistent downward shift in mean pairwise distance, while the left panel shows that local spacing either increases modestly or stays flat rather than collapsing\. Visually, the workflow comparison now looks coherent across all three evaluation families: PCA preprocessing improves neighborhood quality and also nudges the 10D embedding toward a more stable and less concentrated geometry\.

### Summarize workflow tradeoffs

Finally, pull the workflow metrics into one summary table so the tradeoff is visible in one place before we make the construction decision\.

In [48]:
# ---------------------------------------------------------------------
# Summarize workflow tradeoffs
# ---------------------------------------------------------------------

umap_workflow_summary_df = (
    umap_workflow_fit_summary_df[
        [
            "feature_set",
            "umap_workflow",
            "input_space",
            "input_dimensions",
            "elapsed_seconds",
        ]
    ]
    .merge(
        workflow_trustworthiness_df[
            [
                "feature_set",
                "umap_workflow",
                "trustworthiness_score",
            ]
        ],
        on=["feature_set", "umap_workflow"],
        how="left",
        validate="1:1",
    )
    .merge(
        workflow_neighbor_retention_df[
            [
                "feature_set",
                "umap_workflow",
                "mean_neighbor_overlap_share",
                "median_neighbor_overlap_share",
                "p10_neighbor_overlap_share",
            ]
        ],
        on=["feature_set", "umap_workflow"],
        how="left",
        validate="1:1",
    )
    .merge(
        workflow_distance_behavior_df[
            [
                "feature_set",
                "umap_workflow",
                "mean_knn_distance",
                "distance_concentration_ratio",
            ]
        ],
        on=["feature_set", "umap_workflow"],
        how="left",
        validate="1:1",
    )
    .sort_values(["feature_set", "umap_workflow"])
    .reset_index(drop=True)
)

display(umap_workflow_summary_df)

,feature_set,umap_workflow,input_space,input_dimensions,elapsed_seconds,trustworthiness_score,mean_neighbor_overlap_share,median_neighbor_overlap_share,p10_neighbor_overlap_share,mean_knn_distance,distance_concentration_ratio
0,bus,pca_to_umap,pca_80pct,14,NaN,0.987354,0.318022,0.333333,0.133333,0.243062,0.688815
1,bus,raw_to_umap,raw_scaled,40,NaN,0.981803,0.299916,0.266667,0.066667,0.218477,0.697991
2,fhvhv,pca_to_umap,pca_80pct,13,NaN,0.987996,0.338387,0.333333,0.133333,0.226195,0.637768
3,fhvhv,raw_to_umap,raw_scaled,37,NaN,0.978410,0.303018,0.266667,0.066667,0.221423,0.681999
4,multimodal,pca_to_umap,pca_80pct,46,NaN,0.966497,0.291591,0.266667,0.066667,0.208087,0.559024
5,multimodal,raw_to_umap,raw_scaled,142,NaN,0.955377,0.288551,0.266667,0.066667,0.192226,0.573111
6,subway,pca_to_umap,pca_80pct,11,NaN,0.993573,0.478409,0.466667,0.200000,0.155323,0.373846
7,subway,raw_to_umap,raw_scaled,21,NaN,0.988624,0.445973,0.466667,0.200000,0.154992,0.447946
8,taxi,pca_to_umap,pca_80pct,15,NaN,0.987323,0.364084,0.333333,0.133333,0.259495,0.681267
9,taxi,raw_to_umap,raw_scaled,38,NaN,0.976379,0.308662,0.266667,0.066667,0.249084,0.733211


Findings\. The workflow summary table shows a consistent quality advantage for pca\_to\_umap across the three representation metrics\. Trustworthiness improves for all five feature sets, neighbor overlap improves for all five, and distance concentration improves for all five\. The size of the gain varies by modality: Multimodal and Taxi benefit most clearly, Subway and FHVHV also improve meaningfully, and Bus improves more modestly\. The tradeoff is runtime\. PCA preprocessing is slower for Bus, Taxi, Subway, and especially Multimodal, while FHVHV is the only modality where pca\_to\_umap is also faster\. Taken together, the table suggests that the workflow choice is no longer about whether PCA helps quality at all; it is about whether the quality gains are large enough to justify the extra preprocessing step\.

Let's visualize our scorecardd with a heatmap\.

In [49]:
# ---------------------------------------------------------------------
# Visualize UMAP workflow scorecard
# ---------------------------------------------------------------------

workflow_scorecard_df = umap_workflow_summary_df.copy()

workflow_metric_specs = {
    "trustworthiness_score": {
        "ascending": False,
        "label": "Trustworthiness",
        "round_digits": 4,
    },
    "mean_neighbor_overlap_share": {
        "ascending": False,
        "label": "Neighbor overlap",
        "round_digits": 4,
    },
    "distance_concentration_ratio": {
        "ascending": True,
        "label": "Distance concentration",
        "round_digits": 4,
    },
    "elapsed_seconds": {
        "ascending": True,
        "label": "Runtime (sec)",
        "round_digits": 1,
    },
}

workflow_scorecard_records = []

for feature_set_name, feature_df in workflow_scorecard_df.groupby("feature_set"):
    feature_df = feature_df.sort_values("umap_workflow").copy()

    for metric_name, metric_spec in workflow_metric_specs.items():
        metric_values = feature_df[metric_name].astype(float)

        if metric_values.nunique() == 1:
            normalized_values = pd.Series([0.5] * len(metric_values), index=metric_values.index)
        else:
            min_val = metric_values.min()
            max_val = metric_values.max()
            normalized_values = (metric_values - min_val) / (max_val - min_val)

            if metric_spec["ascending"]:
                normalized_values = 1 - normalized_values

        for row_idx, normalized_value in normalized_values.items():
            raw_value = float(feature_df.loc[row_idx, metric_name])
            workflow_scorecard_records.append(
                {
                    "feature_set": feature_set_name,
                    "umap_workflow": feature_df.loc[row_idx, "umap_workflow"],
                    "metric": metric_spec["label"],
                    "normalized_score": float(normalized_value),
                    "raw_value": round(raw_value, metric_spec["round_digits"]),
                }
            )

workflow_scorecard_long_df = pd.DataFrame(workflow_scorecard_records)
workflow_scorecard_long_df["feature_set_workflow"] = (
    workflow_scorecard_long_df["feature_set"]
    + " | "
    + workflow_scorecard_long_df["umap_workflow"]
)

row_order = []
for feature_set_name in MODEL_FEATURE_SET_NAMES:
    for workflow_name in ["raw_to_umap", "pca_to_umap"]:
        row_order.append(f"{feature_set_name} | {workflow_name}")

metric_order = [
    "Trustworthiness",
    "Neighbor overlap",
    "Distance concentration",
    "Runtime (sec)",
]

workflow_scorecard_matrix_df = (
    workflow_scorecard_long_df
    .pivot(index="feature_set_workflow", columns="metric", values="normalized_score")
    .reindex(index=row_order, columns=metric_order)
)

workflow_scorecard_text_df = (
    workflow_scorecard_long_df
    .pivot(index="feature_set_workflow", columns="metric", values="raw_value")
    .reindex(index=row_order, columns=metric_order)
)

workflow_scorecard_fig = px.imshow(
    workflow_scorecard_matrix_df,
    color_continuous_scale="RdYlGn",
    zmin=0,
    zmax=1,
    aspect="auto",
    labels={
        "x": "Metric",
        "y": "Feature set and workflow",
        "color": "Relative score",
    },
    title="UMAP Workflow Scorecard",
)

workflow_scorecard_fig.update_traces(
    text=workflow_scorecard_text_df.values,
    texttemplate="%{text}",
    hovertemplate=(
        "Candidate: %{y}<br>"
        "Metric: %{x}<br>"
        "Relative score: %{z:.3f}<br>"
        "Raw value: %{text}"
        "<extra></extra>"
    ),
)

workflow_scorecard_fig.update_layout(
    width=920,
    height=520,
    template="plotly_white",
)

workflow_scorecard_fig.show()

Findings\. The scorecard makes that tradeoff visually decisive\. For every feature set, the pca\_to\_umap row is stronger on trustworthiness, neighbor overlap, and distance concentration, while raw\_to\_umap wins mainly on runtime\. The most compelling cases are Taxi, FHVHV, and Multimodal, where the quality improvements are broad rather than marginal\. Bus is the closest call because its quality gains are smaller and its runtime penalty is clearer, but even there the PCA\-preprocessed workflow still wins on all three representation\-quality metrics\. Across the full set of modalities, the heatmap reads less like a mixed result and more like a consistent pattern: PCA preprocessing buys better UMAP structure, and the cost is additional computation\.

> Decision\. Use pca\_to\_umap as the preferred UMAP construction workflow for downstream modeling\. It improves neighborhood faithfulness, literal neighbor preservation, and embedding geometry across all five feature sets, with especially meaningful gains for Taxi, FHVHV, Subway, and Multimodal\. Although it is sometimes slower than raw\_to\_umap, the quality improvements are consistent enough to justify the extra PCA preprocessing step as the default workflow\.

### Record the preferred UMAP construction workflow

This final block records the workflow decision so later sections can inherit it directly\. The comparison above has already established the tradeoff; here we turn that result into the canonical UMAP construction policy for the rest of the notebook\.

In [50]:
# ---------------------------------------------------------------------
# Record preferred UMAP construction workflow
# ---------------------------------------------------------------------

CANONICAL_UMAP_WORKFLOW = "pca_to_umap"

canonical_umap_workflow_decision_df = pd.DataFrame(
    [
        {
            "decision_scope": "shared_umap_modeling_workflow",
            "canonical_umap_workflow": CANONICAL_UMAP_WORKFLOW,
            "umap_model_dimensions": CANONICAL_UMAP_MODEL_DIMENSIONS,
            "n_neighbors": UMAP_N_NEIGHBORS,
            "min_dist": UMAP_MIN_DIST,
            "metric": UMAP_METRIC,
            "workflow_description": (
                "Fit UMAP on canonical PCA modeling inputs defined by the 80% "
                "cumulative explained-variance rule from 3.2.1"
            ),
            "decision_basis": (
                "Consistent gains in trustworthiness, neighbor retention, and "
                "distance-concentration behavior across all five feature sets"
            ),
        }
    ]
)

display(canonical_umap_workflow_decision_df)

,decision_scope,canonical_umap_workflow,umap_model_dimensions,n_neighbors,min_dist,metric,workflow_description,decision_basis
0,shared_umap_modeling_workflow,pca_to_umap,10,50,0.1,euclidean,Fit UMAP on canonical PCA modeling inputs defi...,"Consistent gains in trustworthiness, neighbor ..."


This decision applies to the modeling UMAP branch used downstream\. The notebook’s canonical nonlinear representation is now pca\_to\_umap with 10D output dimensionality, while the 2D UMAP from 3\.2\.3 remains the visualization\-oriented embedding\.

### Section Summary

> This section compared two ways of constructing the canonical 10D UMAP modeling representation: fitting UMAP directly on the scaled feature space or fitting it on the canonical PCA modeling inputs exported from 3\.2\.1\. Across all five feature sets, pca\_to\_umap outperformed raw\_to\_umap on trustworthiness, neighbor retention, and distance\-concentration behavior, while raw\_to\_umap was usually faster\. The gains were smallest for Bus and largest for Taxi, FHVHV, Subway, and Multimodal, but the direction of the result was consistent across the full modality set\. With that tradeoff now clear, the notebook adopts pca\_to\_umap as the preferred UMAP construction workflow for downstream representation comparison and export\.

## 3\.2\.4\.5 Compare Candidate Representation Spaces

This section compares the three candidate representation families that may advance into downstream anomaly\-detection workflows: the raw scaled feature space, the canonical PCA score space, and the selected UMAP representation\. The goal is not to force a single winner, but to understand the structural tradeoffs each representation makes in neighborhood preservation, geometry, runtime, and interpretability\. These results will determine whether any representation should be excluded as clearly unsuitable and will provide the context needed for the downstream bakeoff in later notebooks\.

### Define the final comparison set

We'll start by defining the final candidate representation set for each modality\. This keeps the representation comparison explicit and prevents later metric blocks from quietly mixing in stale or intermediate variants\.

In [51]:
# ---------------------------------------------------------------------
# Define final candidate representation set
# ---------------------------------------------------------------------

FINAL_COMPARISON_REPRESENTATIONS = [
    "raw_scaled",
    "pca_80pct",
    "umap_pca_to_umap_10d",
]

final_comparison_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    final_comparison_records.append(
        {
            "feature_set": feature_set_name,
            "representation_name": "raw_scaled",
            "representation_family": "raw",
            "input_dimensions": len(REPRESENTATION_COLUMNS_BY_SET[feature_set_name]),
            "comparison_role": "source_baseline",
        }
    )

    pca_input_components = int(
        pca_to_umap_component_df.loc[
            pca_to_umap_component_df["feature_set"].eq(feature_set_name),
            "pca_input_components",
        ].iloc[0]
    )

    final_comparison_records.append(
        {
            "feature_set": feature_set_name,
            "representation_name": "pca_80pct",
            "representation_family": "pca",
            "input_dimensions": pca_input_components,
            "comparison_role": "linear_reduced_candidate",
        }
    )

    final_comparison_records.append(
        {
            "feature_set": feature_set_name,
            "representation_name": "umap_pca_to_umap_10d",
            "representation_family": "umap",
            "input_dimensions": CANONICAL_UMAP_MODEL_DIMENSIONS,
            "comparison_role": "nonlinear_reduced_candidate",
        }
    )

final_comparison_set_df = (
    pd.DataFrame(final_comparison_records)
    .sort_values(["feature_set", "representation_family"])
    .reset_index(drop=True)
)

display(final_comparison_set_df)

,feature_set,representation_name,representation_family,input_dimensions,comparison_role
0,bus,pca_80pct,pca,14,linear_reduced_candidate
1,bus,raw_scaled,raw,40,source_baseline
2,bus,umap_pca_to_umap_10d,umap,10,nonlinear_reduced_candidate
3,fhvhv,pca_80pct,pca,13,linear_reduced_candidate
4,fhvhv,raw_scaled,raw,37,source_baseline
5,fhvhv,umap_pca_to_umap_10d,umap,10,nonlinear_reduced_candidate
6,multimodal,pca_80pct,pca,46,linear_reduced_candidate
7,multimodal,raw_scaled,raw,142,source_baseline
8,multimodal,umap_pca_to_umap_10d,umap,10,nonlinear_reduced_candidate
9,subway,pca_80pct,pca,11,linear_reduced_candidate


Findings\. The final comparison set is now explicit for every modality: the raw scaled feature space remains the unreduced source baseline, PCA provides the canonical linear reduction under the 80% variance rule, and UMAP provides the canonical nonlinear reduction at 10 dimensions\. This also makes the dimensionality tradeoff concrete\. Bus moves from 40 raw features to 14 PCA components and 10 UMAP dimensions; Multimodal moves from 142 raw features to 46 PCA components and then 10 UMAP dimensions\. From here on, the notebook is no longer comparing candidate settings within PCA or UMAP; it is comparing the three settled representation families that may advance into downstream anomaly detection\.

Next, define the asset paths that correspond to those three final representations\. This gives later comparison blocks one stable dictionary per representation family instead of forcing them to reconstruct path logic on the fly\.

In [52]:
# ---------------------------------------------------------------------
# Define final comparison asset paths
# ---------------------------------------------------------------------

FINAL_COMPARISON_PATHS = {
    "raw_scaled": EVALUATION_SAMPLE_SCALED_PATHS_BY_SET,
    "pca_80pct": PCA_MODELING_EVAL_SAMPLE_PATHS_BY_SET,
    "umap_pca_to_umap_10d": {
        feature_set_name: UMAP_WORKFLOW_OUTPUT_PATHS[(feature_set_name, "pca_to_umap")]
        for feature_set_name in MODEL_FEATURE_SET_NAMES
    },
}

final_comparison_path_records = []

for representation_name, paths_by_set in FINAL_COMPARISON_PATHS.items():
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        asset_path = paths_by_set[feature_set_name]

        final_comparison_path_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": representation_name,
                "asset_path": str(asset_path),
                "path_exists": Path(asset_path).exists(),
            }
        )

final_comparison_path_df = pd.DataFrame(final_comparison_path_records)
final_comparison_path_df["status"] = np.where(
    final_comparison_path_df["path_exists"],
    "pass",
    "review",
)

display(final_comparison_path_df)

assert final_comparison_path_df["status"].eq("pass").all(), (
    "One or more final comparison representation paths is missing."
)

,feature_set,representation_name,asset_path,path_exists,status
0,bus,raw_scaled,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,True,pass
1,subway,raw_scaled,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,True,pass
2,taxi,raw_scaled,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,True,pass
3,fhvhv,raw_scaled,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,True,pass
4,multimodal,raw_scaled,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,True,pass
5,bus,pca_80pct,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,True,pass
6,subway,pca_80pct,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,True,pass
7,taxi,pca_80pct,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,True,pass
8,fhvhv,pca_80pct,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,True,pass
9,multimodal,pca_80pct,/datasets/_deepnote_work/pipeline_data/3.2.4.i...,True,pass


Findings\. The comparison set is now fixed and fully path\-backed\. Raw scaled space serves as the unreduced source baseline, PCA carries the canonical linear reduction, and UMAP carries the canonical nonlinear reduction selected in the previous sections\.

Finally, validate the dimensionality of each final comparison asset\. This is a quick guardrail before we start scoring representation quality\.

In [53]:
# ---------------------------------------------------------------------
# Validate final comparison representation dimensions
# ---------------------------------------------------------------------

final_comparison_dimension_records = []

for _, row in final_comparison_set_df.iterrows():
    feature_set_name = row["feature_set"]
    representation_name = row["representation_name"]
    expected_dimensions = int(row["input_dimensions"])

    asset_path = FINAL_COMPARISON_PATHS[representation_name][feature_set_name]
    asset_columns = get_parquet_columns(asset_path)

    if representation_name == "raw_scaled":
        representation_columns = [
            column_name
            for column_name in asset_columns
            if column_name in REPRESENTATION_COLUMNS_BY_SET[feature_set_name]
        ]
    elif representation_name == "pca_80pct":
        representation_columns = [
            column_name
            for column_name in asset_columns
            if str(column_name).startswith("PC")
        ]
    elif representation_name == "umap_pca_to_umap_10d":
        representation_columns = [
            column_name
            for column_name in asset_columns
            if str(column_name).startswith("umap_")
            and str(column_name)[5:].isdigit()
        ]
    else:
        raise ValueError(f"Unknown representation: {representation_name}")

    final_comparison_dimension_records.append(
        {
            "feature_set": feature_set_name,
            "representation_name": representation_name,
            "expected_dimensions": expected_dimensions,
            "observed_dimensions": len(representation_columns),
            "status": "pass" if len(representation_columns) == expected_dimensions else "review",
        }
    )

final_comparison_dimension_df = (
    pd.DataFrame(final_comparison_dimension_records)
    .sort_values(["feature_set", "representation_name"])
    .reset_index(drop=True)
)

display(final_comparison_dimension_df)

assert final_comparison_dimension_df["status"].eq("pass").all(), (
    "One or more final comparison assets has unexpected dimensionality."
)

,feature_set,representation_name,expected_dimensions,observed_dimensions,status
0,bus,pca_80pct,14,14,pass
1,bus,raw_scaled,40,40,pass
2,bus,umap_pca_to_umap_10d,10,10,pass
3,fhvhv,pca_80pct,13,13,pass
4,fhvhv,raw_scaled,37,37,pass
5,fhvhv,umap_pca_to_umap_10d,10,10,pass
6,multimodal,pca_80pct,46,46,pass
7,multimodal,raw_scaled,142,142,pass
8,multimodal,umap_pca_to_umap_10d,10,10,pass
9,subway,pca_80pct,11,11,pass


Findings\. With the comparison set now fixed, path\-backed, and dimension\-checked, the next section can focus directly on how much raw neighborhood structure PCA and UMAP preserve rather than on bookkeeping\.

### Materialize aligned comparison tables

Before we compare representation quality, we need one aligned evaluation view per modality that places the raw, PCA, UMAP, and metadata assets on the same sampled observations\. This keeps every downstream metric row\-consistent and makes it possible to compare representations without hidden sampling drift or join ambiguity\.

Start by defining the output paths for the aligned comparison tables\. These tables will be the canonical row\-matched inputs for the representation\-comparison metrics that follow\.

In [54]:
# ---------------------------------------------------------------------
# Define aligned comparison-table output paths
# ---------------------------------------------------------------------

REBUILD_COMPARISON_TABLES = False

COMPARISON_TABLE_PATHS_BY_SET = {
    feature_set_name: (
        INTERMEDIATE_OUTPUT_DIR
        / f"{feature_set_name}_representation_comparison_table.parquet"
    )
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

comparison_table_path_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "comparison_table_path": str(COMPARISON_TABLE_PATHS_BY_SET[feature_set_name]),
        }
        for feature_set_name in MODEL_FEATURE_SET_NAMES
    ]
)

display(comparison_table_path_df)

,feature_set,comparison_table_path
0,bus,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
1,subway,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
2,taxi,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
3,fhvhv,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
4,multimodal,/datasets/_deepnote_work/pipeline_data/3.2.4.i...


Now build the aligned comparison tables themselves\. Each table keeps one shared row universe and carries the raw scaled columns, PCA columns, UMAP columns, and key metadata fields needed for downstream evaluation\.

In [55]:
# ---------------------------------------------------------------------
# Materialize aligned comparison tables
# ---------------------------------------------------------------------

comparison_table_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    output_path = COMPARISON_TABLE_PATHS_BY_SET[feature_set_name]
    should_rebuild = REBUILD_COMPARISON_TABLES or (not output_path.exists())

    if should_rebuild:
        metadata_df = pd.read_parquet(
            EVALUATION_SAMPLE_METADATA_PATHS_BY_SET[feature_set_name]
        )

        raw_df = pd.read_parquet(
            EVALUATION_SAMPLE_SCALED_PATHS_BY_SET[feature_set_name]
        )
        raw_columns = ["modeling_row_id"] + REPRESENTATION_COLUMNS_BY_SET[feature_set_name]
        raw_df = raw_df.loc[:, raw_columns].copy()
        raw_df = raw_df.rename(
            columns={
                column_name: f"raw__{column_name}"
                for column_name in raw_df.columns
                if column_name != "modeling_row_id"
            }
        )

        pca_df = pd.read_parquet(
            PCA_MODELING_EVAL_SAMPLE_PATHS_BY_SET[feature_set_name]
        )
        pca_columns = [
            "modeling_row_id",
            *[
                column_name
                for column_name in pca_df.columns
                if str(column_name).startswith("PC")
            ],
        ]
        pca_df = pca_df.loc[:, pca_columns].copy()
        pca_df = pca_df.rename(
            columns={
                column_name: f"pca__{column_name}"
                for column_name in pca_df.columns
                if column_name != "modeling_row_id"
            }
        )

        umap_df = pd.read_parquet(
            UMAP_WORKFLOW_OUTPUT_PATHS[(feature_set_name, "pca_to_umap")]
        )
        umap_columns = [
            "modeling_row_id",
            *[
                column_name
                for column_name in umap_df.columns
                if str(column_name).startswith("umap_")
                and str(column_name)[5:].isdigit()
            ],
        ]
        umap_df = umap_df.loc[:, umap_columns].copy()
        umap_df = umap_df.rename(
            columns={
                column_name: f"umap__{column_name}"
                for column_name in umap_df.columns
                if column_name != "modeling_row_id"
            }
        )

        comparison_df = (
            metadata_df
            .merge(raw_df, on="modeling_row_id", how="left", validate="1:1")
            .merge(pca_df, on="modeling_row_id", how="left", validate="1:1")
            .merge(umap_df, on="modeling_row_id", how="left", validate="1:1")
        )

        comparison_df.to_parquet(output_path, index=False)
        output_action = "written"
    else:
        comparison_df = pd.read_parquet(output_path)
        output_action = "loaded_existing"

    raw_prefixed_columns = [
        column_name for column_name in comparison_df.columns
        if str(column_name).startswith("raw__")
    ]
    pca_prefixed_columns = [
        column_name for column_name in comparison_df.columns
        if str(column_name).startswith("pca__")
    ]
    umap_prefixed_columns = [
        column_name for column_name in comparison_df.columns
        if str(column_name).startswith("umap__")
    ]

    total_missing_cells = int(
        comparison_df[
            raw_prefixed_columns + pca_prefixed_columns + umap_prefixed_columns
        ].isna().sum().sum()
    )

    comparison_table_records.append(
        {
            "feature_set": feature_set_name,
            "sample_rows": len(comparison_df),
            "raw_columns": len(raw_prefixed_columns),
            "pca_columns": len(pca_prefixed_columns),
            "umap_columns": len(umap_prefixed_columns),
            "missing_representation_cells": total_missing_cells,
            "output_action": output_action,
            "status": "pass"
            if (
                len(comparison_df) == REPRESENTATION_EVALUATION_SAMPLE_ROWS
                and total_missing_cells == 0
            )
            else "review",
        }
    )

comparison_table_summary_df = pd.DataFrame(comparison_table_records)

display(comparison_table_summary_df)

assert comparison_table_summary_df["status"].eq("pass").all(), (
    "One or more aligned comparison tables failed validation."
)

,feature_set,sample_rows,raw_columns,pca_columns,umap_columns,missing_representation_cells,output_action,status
0,bus,50000,40,14,10,0,loaded_existing,pass
1,subway,50000,21,11,10,0,loaded_existing,pass
2,taxi,50000,38,15,10,0,loaded_existing,pass
3,fhvhv,50000,37,13,10,0,loaded_existing,pass
4,multimodal,50000,142,46,10,0,loaded_existing,pass


Findings\. These aligned comparison tables now serve as the common evaluation surface for the rest of the chapter\. Each row corresponds to the same sampled mobility\-state observation across raw, PCA, and UMAP representations, which means the next metric sections can compare representations directly rather than reconstructing joins on the fly\.

Finish with a quick schema check so we can confirm that the three representation families were materialized with the expected prefixed column groups\.

In [56]:
# ---------------------------------------------------------------------
# Validate aligned comparison-table schema groups
# ---------------------------------------------------------------------

comparison_schema_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    comparison_path = COMPARISON_TABLE_PATHS_BY_SET[feature_set_name]
    comparison_columns = get_parquet_columns(comparison_path)

    raw_columns = [
        column_name for column_name in comparison_columns
        if str(column_name).startswith("raw__")
    ]
    pca_columns = [
        column_name for column_name in comparison_columns
        if str(column_name).startswith("pca__")
    ]
    umap_columns = [
        column_name for column_name in comparison_columns
        if str(column_name).startswith("umap__")
    ]

    expected_raw_columns = len(REPRESENTATION_COLUMNS_BY_SET[feature_set_name])
    expected_pca_columns = int(
        pca_to_umap_component_df.loc[
            pca_to_umap_component_df["feature_set"].eq(feature_set_name),
            "pca_input_components",
        ].iloc[0]
    )
    expected_umap_columns = CANONICAL_UMAP_MODEL_DIMENSIONS

    comparison_schema_records.append(
        {
            "feature_set": feature_set_name,
            "expected_raw_columns": expected_raw_columns,
            "observed_raw_columns": len(raw_columns),
            "expected_pca_columns": expected_pca_columns,
            "observed_pca_columns": len(pca_columns),
            "expected_umap_columns": expected_umap_columns,
            "observed_umap_columns": len(umap_columns),
            "status": "pass"
            if (
                len(raw_columns) == expected_raw_columns
                and len(pca_columns) == expected_pca_columns
                and len(umap_columns) == expected_umap_columns
            )
            else "review",
        }
    )

comparison_schema_df = pd.DataFrame(comparison_schema_records)

display(comparison_schema_df)

assert comparison_schema_df["status"].eq("pass").all(), (
    "One or more comparison tables has unexpected representation-column counts."
)

,feature_set,expected_raw_columns,observed_raw_columns,expected_pca_columns,observed_pca_columns,expected_umap_columns,observed_umap_columns,status
0,bus,40,40,14,14,10,10,pass
1,subway,21,21,11,11,10,10,pass
2,taxi,38,38,15,15,10,10,pass
3,fhvhv,37,37,13,13,10,10,pass
4,multimodal,142,142,46,46,10,10,pass


Findings\. The prefixed schema groups make the later metric code much easier to read: raw\_\_ columns define the source baseline, pca\_\_ columns define the linear reduced space, and umap\_\_ columns define the nonlinear reduced space\. From here on, the chapter can focus purely on what these representations preserve and what they simplify\.

### Compare PCA and UMAP against raw neighborhood structure

This section treats the raw scaled feature space as the neighborhood reference and asks a narrow question: when we reduce that space with PCA or UMAP, which representation better preserves the local observation structure that was present in the original data? The comparison focuses on neighborhood fidelity rather than global distance preservation, because that is the behavior UMAP is designed to optimize and the behavior most relevant for downstream local\-structure and anomaly workflows\.

We will compare the two reduced candidates, pca\_80pct and umap\_pca\_to\_umap\_10d, against the raw baseline using two metrics: trustworthiness and neighbor overlap\. Trustworthiness checks whether points that become neighbors in the reduced space were also genuinely nearby in the raw space, while neighbor overlap checks how much of each observation’s original local neighborhood survives the reduction\.

Let's start by defining the comparison settings and output paths for the two neighborhood\-fidelity evaluations\. This keeps the expensive metric blocks below short and reusable\.

In [57]:
# ---------------------------------------------------------------------
# Define raw-neighborhood comparison settings and output paths
# ---------------------------------------------------------------------

REPRESENTATION_NEIGHBOR_K = 15
REPRESENTATION_EVAL_SAMPLE_ROWS = 15_000
REPRESENTATION_COMPARISON_REPRESENTATIONS = [
    "raw_scaled",
    "pca_80pct",
    "umap_pca_to_umap_10d",
]

REBUILD_REPRESENTATION_NEIGHBOR_METRICS = False

REPRESENTATION_TRUSTWORTHINESS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "representation_trustworthiness_vs_raw.csv"
)
REPRESENTATION_NEIGHBOR_OVERLAP_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "representation_neighbor_overlap_vs_raw.csv"
)

representation_metric_settings_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "raw_reference": "raw_scaled",
            "comparison_representations": ", ".join(
                rep for rep in REPRESENTATION_COMPARISON_REPRESENTATIONS if rep != "raw_scaled"
            ),
            "sample_rows": REPRESENTATION_EVAL_SAMPLE_ROWS,
            "neighbor_k": REPRESENTATION_NEIGHBOR_K,
        }
        for feature_set_name in MODEL_FEATURE_SET_NAMES
    ]
)

display(representation_metric_settings_df)

print("Representation neighborhood-comparison settings ready.")

,feature_set,raw_reference,comparison_representations,sample_rows,neighbor_k
0,bus,raw_scaled,"pca_80pct, umap_pca_to_umap_10d",15000,15
1,subway,raw_scaled,"pca_80pct, umap_pca_to_umap_10d",15000,15
2,taxi,raw_scaled,"pca_80pct, umap_pca_to_umap_10d",15000,15
3,fhvhv,raw_scaled,"pca_80pct, umap_pca_to_umap_10d",15000,15
4,multimodal,raw_scaled,"pca_80pct, umap_pca_to_umap_10d",15000,15


Representation neighborhood-comparison settings ready.


Findings\. This comparison uses the same setup across all five feature sets: raw scaled space is the neighborhood reference, pca\_80pct and umap\_pca\_to\_umap\_10d are the two reduced candidates, and trustworthiness is evaluated on the same 15,000\-row sample with k = 15\. That makes the representation comparison directly comparable across modalities\.

Compute trustworthiness for PCA and UMAP relative to the raw scaled feature space\. This asks whether neighborhoods created in the reduced representation are locally faithful to the source space\.

In [58]:
# ---------------------------------------------------------------------
# Evaluate trustworthiness of PCA and UMAP against raw neighborhoods
# ---------------------------------------------------------------------

from sklearn.manifold import trustworthiness

if REBUILD_REPRESENTATION_NEIGHBOR_METRICS or not REPRESENTATION_TRUSTWORTHINESS_PATH.exists():
    trustworthiness_records = []
    total_jobs = len(MODEL_FEATURE_SET_NAMES) * 2
    job_counter = 0

    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        comparison_path = COMPARISON_TABLE_PATHS_BY_SET[feature_set_name]
        comparison_df = pd.read_parquet(comparison_path)

        sample_n = min(REPRESENTATION_EVAL_SAMPLE_ROWS, len(comparison_df))
        eval_df = comparison_df.sample(
            n=sample_n,
            random_state=REPRESENTATION_EVALUATION_RANDOM_STATE,
        ).reset_index(drop=True)

        raw_columns = sorted(
            column_name
            for column_name in eval_df.columns
            if str(column_name).startswith("raw__")
        )
        pca_columns = sorted(
            column_name
            for column_name in eval_df.columns
            if str(column_name).startswith("pca__")
        )
        umap_columns = sorted(
            column_name
            for column_name in eval_df.columns
            if str(column_name).startswith("umap__umap_")
        )

        assert raw_columns, f"{feature_set_name}: no raw columns found."
        assert pca_columns, f"{feature_set_name}: no PCA columns found."
        assert umap_columns, f"{feature_set_name}: no UMAP columns found."

        X_raw = eval_df[raw_columns].to_numpy(dtype="float32", copy=False)
        comparison_column_map = {
            "pca_80pct": pca_columns,
            "umap_pca_to_umap_10d": umap_columns,
        }

        for representation_name, representation_columns in comparison_column_map.items():
            job_counter += 1
            print(
                f"[{job_counter}/{total_jobs}] Computing trustworthiness for "
                f"{feature_set_name} | {representation_name}."
            )

            X_reduced = eval_df[representation_columns].to_numpy(dtype="float32", copy=False)

            trustworthiness_score = trustworthiness(
                X_raw,
                X_reduced,
                n_neighbors=REPRESENTATION_NEIGHBOR_K,
            )

            trustworthiness_records.append(
                {
                    "feature_set": feature_set_name,
                    "representation_name": representation_name,
                    "sample_rows": len(eval_df),
                    "input_dimensions": X_reduced.shape[1],
                    "trustworthiness_k": REPRESENTATION_NEIGHBOR_K,
                    "trustworthiness_score": trustworthiness_score,
                    "status": "pass",
                }
            )

    representation_trustworthiness_df = pd.DataFrame(trustworthiness_records)
    representation_trustworthiness_df.to_csv(REPRESENTATION_TRUSTWORTHINESS_PATH, index=False)
    output_action = "written"
else:
    representation_trustworthiness_df = pd.read_csv(REPRESENTATION_TRUSTWORTHINESS_PATH)
    output_action = "loaded"

display(
    representation_trustworthiness_df.assign(
        trustworthiness_score=lambda df: df["trustworthiness_score"].round(4)
    ).sort_values(["feature_set", "representation_name"]).reset_index(drop=True)
)

print(f"Representation trustworthiness results {output_action}.")

,feature_set,representation_name,sample_rows,input_dimensions,trustworthiness_k,trustworthiness_score,status
0,bus,pca_80pct,15000,14,15,0.9985,pass
1,bus,umap_pca_to_umap_10d,15000,10,15,0.9788,pass
2,fhvhv,pca_80pct,15000,13,15,0.9967,pass
3,fhvhv,umap_pca_to_umap_10d,15000,10,15,0.9734,pass
4,multimodal,pca_80pct,15000,46,15,0.9992,pass
5,multimodal,umap_pca_to_umap_10d,15000,10,15,0.9534,pass
6,subway,pca_80pct,15000,11,15,0.9979,pass
7,subway,umap_pca_to_umap_10d,15000,10,15,0.9851,pass
8,taxi,pca_80pct,15000,15,15,0.9940,pass
9,taxi,umap_pca_to_umap_10d,15000,10,15,0.9674,pass


Representation trustworthiness results loaded.


> 📌 Note\. Trustworthiness is the stricter of the two neighborhood metrics here\. A high value means that points treated as neighbors in the reduced representation were usually already close in the raw feature space, so the reduction is not inventing too many false local relationships\.

Findings\. PCA preserves raw local neighborhoods much more faithfully than 10D UMAP in every feature set\. All PCA trustworthiness scores are extremely high, ranging from 0\.994 for Taxi to 0\.999 for Multimodal, while UMAP falls noticeably lower in every case: 0\.985 for Subway, 0\.979 for Bus, 0\.973 for FHVHV, 0\.967 for Taxi, and 0\.953 for Multimodal\. The biggest gap appears in Multimodal, where UMAP loses the most local fidelity relative to the raw feature space\. So by this metric, PCA is clearly the stronger “stay close to raw structure” reduction\.

The trustworthiness table gives the exact scores; this slope chart makes the direction of the comparison easier to see\. An upward slope means the representation becomes more locally faithful to the raw feature space when we move from pca\_80pct to umap\_pca\_to\_umap\_10d, while a downward slope means PCA is preserving more of the raw local structure\.

In [59]:
# ---------------------------------------------------------------------
# Plot trustworthiness comparison as a slope chart
# ---------------------------------------------------------------------

import plotly.graph_objects as go

trust_plot_df = (
    representation_trustworthiness_df.copy()
    .assign(
        workflow_order=lambda df: df["representation_name"].map(
            {
                "pca_80pct": 0,
                "umap_pca_to_umap_10d": 1,
            }
        ),
        workflow_label=lambda df: df["representation_name"].map(
            {
                "pca_80pct": "PCA",
                "umap_pca_to_umap_10d": "UMAP",
            }
        ),
    )
    .sort_values(["feature_set", "workflow_order"])
)

trust_slope_fig = go.Figure()

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_df = trust_plot_df.loc[
        trust_plot_df["feature_set"].eq(feature_set_name)
    ].sort_values("workflow_order")

    trust_slope_fig.add_trace(
        go.Scatter(
            x=feature_df["workflow_label"],
            y=feature_df["trustworthiness_score"],
            mode="lines+markers+text",
            name=feature_set_name,
            text=feature_df["feature_set"].where(feature_df["workflow_order"].eq(0), ""),
            textposition="middle left",
            marker={"size": 9},
            line={"width": 3},
            hovertemplate=(
                "Feature set: %{customdata[0]}<br>"
                "Representation: %{x}<br>"
                "Trustworthiness: %{y:.4f}"
                "<extra></extra>"
            ),
            customdata=np.column_stack([feature_df["feature_set"]]),
        )
    )

trust_slope_fig.update_layout(
    title="Representation Trustworthiness Slope Chart",
    width=860,
    height=520,
    template="plotly_white",
    showlegend=False,
    margin={"t": 70, "b": 60, "l": 70, "r": 40},
)

trust_slope_fig.update_xaxes(title_text="Representation")
trust_slope_fig.update_yaxes(title_text="Trustworthiness")

trust_slope_fig.show()

Findings\. The slope chart makes the pattern impossible to miss: every feature set moves downward from PCA to UMAP, meaning trustworthiness declines consistently when we switch from the linear PCA representation to the nonlinear UMAP representation\. Subway remains the least damaged by the move to UMAP, while Multimodal is affected the most\. So yes, if the question is strictly “which reduced representation preserves original raw neighborhoods better,” UMAP does substantially worse here\. That does not make UMAP useless, but it does mean its value would need to come from revealing alternative local organization rather than from faithfully reproducing the raw neighborhood geometry\.

Now compare neighborhood retention more directly\. Instead of asking whether reduced\-space neighbors are legitimate, this block asks how much of each row’s original raw nearest\-neighbor set survives into PCA or UMAP\.

In [60]:
# ---------------------------------------------------------------------
# Evaluate PCA and UMAP neighbor overlap against raw neighborhoods
# ---------------------------------------------------------------------sk

if REBUILD_REPRESENTATION_NEIGHBOR_METRICS or not REPRESENTATION_NEIGHBOR_OVERLAP_PATH.exists():
    overlap_records = []
    total_jobs = len(MODEL_FEATURE_SET_NAMES) * 2
    job_counter = 0

    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        comparison_path = COMPARISON_TABLE_PATHS_BY_SET[feature_set_name]
        comparison_df = pd.read_parquet(comparison_path)

        sample_n = min(REPRESENTATION_EVAL_SAMPLE_ROWS, len(comparison_df))
        eval_df = comparison_df.sample(
            n=sample_n,
            random_state=REPRESENTATION_EVALUATION_RANDOM_STATE,
        ).reset_index(drop=True)

        raw_columns = sorted(
            column_name
            for column_name in eval_df.columns
            if str(column_name).startswith("raw__")
        )
        pca_columns = sorted(
            column_name
            for column_name in eval_df.columns
            if str(column_name).startswith("pca__")
        )
        umap_columns = sorted(
            column_name
            for column_name in eval_df.columns
            if str(column_name).startswith("umap__umap_")
        )

        assert raw_columns, f"{feature_set_name}: no raw columns found."
        assert pca_columns, f"{feature_set_name}: no PCA columns found."
        assert umap_columns, f"{feature_set_name}: no UMAP columns found."

        X_raw = eval_df[raw_columns].to_numpy(dtype="float32", copy=False)
        comparison_column_map = {
            "pca_80pct": pca_columns,
            "umap_pca_to_umap_10d": umap_columns,
        }

        raw_nn = NearestNeighbors(
            n_neighbors=REPRESENTATION_NEIGHBOR_K + 1,
            metric="euclidean",
        )
        raw_nn.fit(X_raw)
        raw_neighbor_idx = raw_nn.kneighbors(return_distance=False)[:, 1:]
        raw_neighbor_sets = [set(row.tolist()) for row in raw_neighbor_idx]

        for representation_name, representation_columns in comparison_column_map.items():
            job_counter += 1
            print(
                f"[{job_counter}/{total_jobs}] Computing neighbor overlap for "
                f"{feature_set_name} | {representation_name}."
            )

            X_reduced = eval_df[representation_columns].to_numpy(dtype="float32", copy=False)

            reduced_nn = NearestNeighbors(
                n_neighbors=REPRESENTATION_NEIGHBOR_K + 1,
                metric="euclidean",
            )
            reduced_nn.fit(X_reduced)
            reduced_neighbor_idx = reduced_nn.kneighbors(return_distance=False)[:, 1:]

            overlap_shares = []
            for row_index, reduced_neighbors in enumerate(reduced_neighbor_idx):
                overlap_count = len(raw_neighbor_sets[row_index] & set(reduced_neighbors.tolist()))
                overlap_shares.append(overlap_count / REPRESENTATION_NEIGHBOR_K)

            overlap_records.append(
                {
                    "feature_set": feature_set_name,
                    "representation_name": representation_name,
                    "sample_rows": len(eval_df),
                    "neighbor_k": REPRESENTATION_NEIGHBOR_K,
                    "mean_neighbor_overlap_share": float(np.mean(overlap_shares)),
                    "median_neighbor_overlap_share": float(np.median(overlap_shares)),
                    "p10_neighbor_overlap_share": float(np.quantile(overlap_shares, 0.10)),
                    "status": "pass",
                }
            )

    representation_neighbor_overlap_df = pd.DataFrame(overlap_records)
    representation_neighbor_overlap_df.to_csv(REPRESENTATION_NEIGHBOR_OVERLAP_PATH, index=False)
    output_action = "written"
else:
    representation_neighbor_overlap_df = pd.read_csv(REPRESENTATION_NEIGHBOR_OVERLAP_PATH)
    output_action = "loaded"

display(
    representation_neighbor_overlap_df.assign(
        mean_neighbor_overlap_share=lambda df: df["mean_neighbor_overlap_share"].round(4),
        median_neighbor_overlap_share=lambda df: df["median_neighbor_overlap_share"].round(4),
        p10_neighbor_overlap_share=lambda df: df["p10_neighbor_overlap_share"].round(4),
    ).sort_values(["feature_set", "representation_name"]).reset_index(drop=True)
)

print(f"Representation neighbor-overlap results {output_action}.")

,feature_set,representation_name,sample_rows,neighbor_k,mean_neighbor_overlap_share,median_neighbor_overlap_share,p10_neighbor_overlap_share,status
0,bus,pca_80pct,15000,15,0.6096,0.6000,0.4000,pass
1,bus,umap_pca_to_umap_10d,15000,15,0.2733,0.2667,0.0667,pass
2,fhvhv,pca_80pct,15000,15,0.5236,0.5333,0.3333,pass
3,fhvhv,umap_pca_to_umap_10d,15000,15,0.2630,0.2667,0.0667,pass
4,multimodal,pca_80pct,15000,15,0.6665,0.6667,0.4667,pass
5,multimodal,umap_pca_to_umap_10d,15000,15,0.2775,0.2667,0.0667,pass
6,subway,pca_80pct,15000,15,0.6193,0.6000,0.4000,pass
7,subway,umap_pca_to_umap_10d,15000,15,0.3781,0.4000,0.1333,pass
8,taxi,pca_80pct,15000,15,0.4924,0.4667,0.2667,pass
9,taxi,umap_pca_to_umap_10d,15000,15,0.2509,0.2000,0.0667,pass


Representation neighbor-overlap results loaded.


> 📌 Note\. Neighbor overlap is easier to read literally: if the mean overlap is 0\.30 at k=15, then the reduced representation is preserving about 4\.5 of each row’s original 15 nearest neighbors on average\.

Findings\. Neighbor retention tells the same story as trustworthiness, but in a much more concrete way: pca\_80pct keeps far more of each row’s original raw nearest neighbors than umap\_pca\_to\_umap\_10d\. On average, PCA preserves about 61% of the raw 15\-neighbor set for Bus, 62% for Subway, 49% for Taxi, 52% for FHVHV, and 67% for Multimodal\. UMAP is much lower across the board, mostly clustering near 25% to 28%, with Subway as the only relative standout at 38%\. The lower\-tail results point in the same direction: PCA keeps at least 40% of raw neighbors for Bus and Subway and nearly 47% for Multimodal at the 10th percentile, while UMAP drops to just 6\.7% for four of the five feature sets\. So this metric says not only that PCA is better on average, but that UMAP is also much less reliable observation\-by\-observation\.

Now let's apply the same slope plot idea to neighbor retention\. The table shows the exact overlap rates, while the slope chart shows whether local raw neighborhoods survive better under PCA or UMAP for each modality\.

In [61]:
# ---------------------------------------------------------------------
# Plot neighbor-overlap comparison as a slope chart
# ---------------------------------------------------------------------

overlap_plot_df = (
    representation_neighbor_overlap_df.copy()
    .assign(
        workflow_order=lambda df: df["representation_name"].map(
            {
                "pca_80pct": 0,
                "umap_pca_to_umap_10d": 1,
            }
        ),
        workflow_label=lambda df: df["representation_name"].map(
            {
                "pca_80pct": "PCA",
                "umap_pca_to_umap_10d": "UMAP",
            }
        ),
    )
    .sort_values(["feature_set", "workflow_order"])
)

overlap_slope_fig = go.Figure()

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_df = overlap_plot_df.loc[
        overlap_plot_df["feature_set"].eq(feature_set_name)
    ].sort_values("workflow_order")

    overlap_slope_fig.add_trace(
        go.Scatter(
            x=feature_df["workflow_label"],
            y=feature_df["mean_neighbor_overlap_share"],
            mode="lines+markers+text",
            name=feature_set_name,
            text=feature_df["feature_set"].where(feature_df["workflow_order"].eq(0), ""),
            textposition="middle left",
            marker={"size": 9},
            line={"width": 3},
            hovertemplate=(
                "Feature set: %{customdata[0]}<br>"
                "Representation: %{x}<br>"
                "Mean neighbor overlap: %{y:.4f}"
                "<extra></extra>"
            ),
            customdata=np.column_stack([feature_df["feature_set"]]),
        )
    )

overlap_slope_fig.update_layout(
    title="Representation Neighbor-Retention Slope Chart",
    width=860,
    height=520,
    template="plotly_white",
    showlegend=False,
    margin={"t": 70, "b": 60, "l": 70, "r": 40},
)

overlap_slope_fig.update_xaxes(title_text="Representation")
overlap_slope_fig.update_yaxes(title_text="Mean neighbor overlap")

overlap_slope_fig.show()

Findings\. The slope chart shows a large downward move from PCA to UMAP for every feature set, and the drop is materially bigger than what we saw in the trustworthiness comparison\. Multimodal, Bus, and Subway start especially high under PCA and then fall sharply under UMAP, while Taxi and FHVHV also decline to the same low\-overlap band\. Subway remains the strongest UMAP case, but even there PCA still preserves substantially more raw local structure\. Taken together, the visual reinforces that when the target is raw\-neighborhood preservation, PCA is not just slightly better than UMAP; it is operating in a different performance tier\.

Put the two metrics side by side in a compact visual comparison\. The goal here is to make it obvious, by feature set, whether PCA or UMAP is preserving local raw structure better\.

In [62]:
# ---------------------------------------------------------------------
# Visualize PCA vs UMAP neighborhood fidelity against raw
# ---------------------------------------------------------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots

representation_metric_plot_df = (
    representation_trustworthiness_df.merge(
        representation_neighbor_overlap_df[
            [
                "feature_set",
                "representation_name",
                "mean_neighbor_overlap_share",
            ]
        ],
        on=["feature_set", "representation_name"],
        how="inner",
    )
)

representation_label_map = {
    "pca_80pct": "PCA",
    "umap_pca_to_umap_10d": "UMAP",
}
representation_color_map = {
    "pca_80pct": "#1f77b4",
    "umap_pca_to_umap_10d": "#d62728",
}

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Trustworthiness vs Raw", "Neighbor Overlap vs Raw"],
    horizontal_spacing=0.12,
)

for representation_name in ["pca_80pct", "umap_pca_to_umap_10d"]:
    plot_df = (
        representation_metric_plot_df.loc[
            representation_metric_plot_df["representation_name"].eq(representation_name)
        ]
        .sort_values("feature_set")
    )

    fig.add_trace(
        go.Bar(
            x=plot_df["feature_set"],
            y=plot_df["trustworthiness_score"],
            name=representation_label_map[representation_name],
            marker_color=representation_color_map[representation_name],
            legendgroup=representation_name,
            showlegend=True,
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Bar(
            x=plot_df["feature_set"],
            y=plot_df["mean_neighbor_overlap_share"],
            name=representation_label_map[representation_name],
            marker_color=representation_color_map[representation_name],
            legendgroup=representation_name,
            showlegend=False,
        ),
        row=1,
        col=2,
    )

fig.update_layout(
    title="PCA vs UMAP Neighborhood Fidelity Relative to Raw",
    barmode="group",
    width=980,
    height=460,
    template="plotly_white",
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.08,
        "xanchor": "center",
        "x": 0.5,
    },
    margin={"t": 90, "b": 60, "l": 50, "r": 30},
)

fig.update_yaxes(title_text="Trustworthiness", row=1, col=1)
fig.update_yaxes(title_text="Mean neighbor overlap", row=1, col=2)

fig.show()

Findings\. Viewed side by side, the two neighborhood\-fidelity metrics tell a consistent story\. On the left, trustworthiness shows that both reduced representations remain fairly locally sensible, but PCA stays closer to the raw space in every feature set, with the gap widest for Multimodal and Taxi\. On the right, the practical difference becomes much larger: PCA preserves far more of each observation’s original raw neighbors than UMAP, especially for Bus, Multimodal, and Taxi\. Subway is the strongest UMAP case on both panels, but PCA still leads there too\. So the combined view sharpens the interpretation: UMAP can still produce locally coherent neighborhoods, but it does not preserve the original raw neighborhood structure nearly as well as PCA\.

### Compare geometry and practicality across all three

At this point, we already know that PCA preserves raw local neighborhoods better than UMAP\. This last comparison is a little different\. Rather than treating the three representations as competitors in a single winner\-take\-all contest, we use the same aligned observations to describe the geometric regime each one creates: the original raw scaled space, the fidelity\-preserving PCA reduction, and the nonlinear UMAP reduction\. 

That helps clarify what each representation is likely to contribute once we move into downstream anomaly workflows\.The focus here is descriptive\. We profile local spacing, broad distance concentration, and dimensional footprint across all three representations\. Raw is the source baseline, PCA is the compact linear approximation to that baseline, and UMAP is the compact nonlinear alternative\. The point is to understand the tradeoff landscape cleanly before we leave the representation notebook\.

The same 15,000\-row evaluation sample and 15\-neighbor local definition used above are reused here so that the geometry profiles stay comparable across feature sets and representation families\.

The first block computes the geometry profile for all three aligned representations\. This is the expensive step in the subsection, so it writes the results to disk and reloads them on reruns unless the rebuild toggle is turned on\.

In [63]:
# ---------------------------------------------------------------------
# Profile geometry across raw, PCA, and UMAP representation spaces
# ---------------------------------------------------------------------

REBUILD_REPRESENTATION_GEOMETRY_PROFILE = False
REPRESENTATION_GEOMETRY_PROFILE_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "representation_geometry_profile.csv"
)
PAIRWISE_DISTANCE_SAMPLE_ROWS = 2_000

if REBUILD_REPRESENTATION_GEOMETRY_PROFILE or not REPRESENTATION_GEOMETRY_PROFILE_PATH.exists():
    geometry_records = []
    total_jobs = len(MODEL_FEATURE_SET_NAMES) * len(FINAL_COMPARISON_REPRESENTATIONS)
    job_counter = 0

    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        comparison_path = COMPARISON_TABLE_PATHS_BY_SET[feature_set_name]
        comparison_df = pd.read_parquet(comparison_path)

        sample_n = min(REPRESENTATION_EVAL_SAMPLE_ROWS, len(comparison_df))
        eval_df = comparison_df.sample(
            n=sample_n,
            random_state=REPRESENTATION_EVALUATION_RANDOM_STATE,
        ).reset_index(drop=True)

        raw_columns = sorted(
            column_name
            for column_name in eval_df.columns
            if str(column_name).startswith("raw__")
        )
        pca_columns = sorted(
            column_name
            for column_name in eval_df.columns
            if str(column_name).startswith("pca__")
        )
        umap_columns = sorted(
            column_name
            for column_name in eval_df.columns
            if str(column_name).startswith("umap__umap_")
        )

        representation_column_map = {
            "raw_scaled": raw_columns,
            "pca_80pct": pca_columns,
            "umap_pca_to_umap_10d": umap_columns,
        }

        raw_dimension_count = len(raw_columns)

        for representation_name in FINAL_COMPARISON_REPRESENTATIONS:
            job_counter += 1
            print(
                f"[{job_counter}/{total_jobs}] Profiling geometry for "
                f"{feature_set_name} | {representation_name}."
            )

            representation_columns = representation_column_map[representation_name]
            assert representation_columns, (
                f"{feature_set_name} | {representation_name}: no columns found."
            )

            X = eval_df[representation_columns].to_numpy(dtype="float32", copy=False)

            nn_model = NearestNeighbors(
                n_neighbors=REPRESENTATION_NEIGHBOR_K + 1,
                metric="euclidean",
            )
            nn_model.fit(X)
            knn_distances = nn_model.kneighbors(return_distance=True)[0][:, 1:]

            pairwise_sample_n = min(PAIRWISE_DISTANCE_SAMPLE_ROWS, len(eval_df))
            pairwise_idx = eval_df.sample(
                n=pairwise_sample_n,
                random_state=REPRESENTATION_EVALUATION_RANDOM_STATE,
            ).index
            X_pairwise = X[pairwise_idx]
            pairwise_distances = pdist(X_pairwise, metric="euclidean")

            pairwise_distance_mean = float(np.mean(pairwise_distances))
            pairwise_distance_std = float(np.std(pairwise_distances))
            distance_concentration_ratio = (
                pairwise_distance_std / pairwise_distance_mean
                if pairwise_distance_mean > 0
                else np.nan
            )

            geometry_records.append(
                {
                    "feature_set": feature_set_name,
                    "representation_name": representation_name,
                    "sample_rows": len(eval_df),
                    "input_dimensions": X.shape[1],
                    "dimension_share_of_raw": X.shape[1] / raw_dimension_count,
                    "neighbor_k": REPRESENTATION_NEIGHBOR_K,
                    "mean_knn_distance": float(np.mean(knn_distances)),
                    "median_knn_distance": float(np.median(knn_distances)),
                    "p95_knn_distance": float(np.quantile(knn_distances, 0.95)),
                    "pairwise_distance_mean": pairwise_distance_mean,
                    "pairwise_distance_std": pairwise_distance_std,
                    "distance_concentration_ratio": float(distance_concentration_ratio),
                    "status": "pass",
                }
            )

    representation_geometry_profile_df = pd.DataFrame(geometry_records)
    representation_geometry_profile_df.to_csv(
        REPRESENTATION_GEOMETRY_PROFILE_PATH,
        index=False,
    )
    output_action = "written"
else:
    representation_geometry_profile_df = pd.read_csv(REPRESENTATION_GEOMETRY_PROFILE_PATH)
    output_action = "loaded"

display(
    representation_geometry_profile_df.assign(
        dimension_share_of_raw=lambda df: df["dimension_share_of_raw"].round(3),
        mean_knn_distance=lambda df: df["mean_knn_distance"].round(4),
        pairwise_distance_mean=lambda df: df["pairwise_distance_mean"].round(4),
        distance_concentration_ratio=lambda df: df["distance_concentration_ratio"].round(4),
    ).sort_values(["feature_set", "representation_name"]).reset_index(drop=True)
)

print(f"Representation geometry profiles {output_action}.")

,feature_set,representation_name,sample_rows,input_dimensions,dimension_share_of_raw,neighbor_k,mean_knn_distance,median_knn_distance,p95_knn_distance,pairwise_distance_mean,pairwise_distance_std,distance_concentration_ratio,status
0,bus,pca_80pct,15000,14,0.350,15,2.4256,2.179521,4.057523,7.3842,3.546483,0.4803,pass
1,bus,raw_scaled,15000,40,1.000,15,3.2080,2.886393,5.263895,8.2132,3.948101,0.4807,pass
2,bus,umap_pca_to_umap_10d,15000,10,0.250,15,0.2431,0.218979,0.478790,6.5851,4.529706,0.6879,pass
3,fhvhv,pca_80pct,15000,13,0.351,15,1.9743,1.637936,4.003725,7.1893,3.750107,0.5216,pass
4,fhvhv,raw_scaled,15000,37,1.000,15,2.8493,2.393938,5.587315,8.0366,4.025873,0.5009,pass
5,fhvhv,umap_pca_to_umap_10d,15000,10,0.270,15,0.2262,0.201395,0.481443,6.7766,4.300796,0.6347,pass
6,multimodal,pca_80pct,15000,46,0.324,15,6.0253,5.311795,10.900395,13.7610,5.875496,0.4270,pass
7,multimodal,raw_scaled,15000,142,1.000,15,7.7696,6.860068,13.826090,15.4869,6.308244,0.4073,pass
8,multimodal,umap_pca_to_umap_10d,15000,10,0.070,15,0.2081,0.188884,0.428453,5.6235,3.108634,0.5528,pass
9,subway,pca_80pct,15000,11,0.524,15,0.8500,0.533663,2.460366,4.6745,3.165809,0.6773,pass


Representation geometry profiles loaded.


Findings\. The geometry profile shows that pca\_80pct behaves like a compact approximation of the raw space, while umap\_pca\_to\_umap\_10d creates a very different geometric regime\. PCA reduces dimensionality substantially but keeps local and global distance scales in the same general band as raw\. UMAP compresses dimensionality even further and collapses mean kNN distances dramatically across all five feature sets, while also changing distance concentration much more sharply\. The clearest example is Multimodal: raw drops from 142 dimensions to 46 under PCA and to 10 under UMAP, but UMAP also shifts its distance concentration ratio from 0\.41 to 0\.55, showing that it is not just compressing the space but reorganizing its geometry\.

Next, we compress the raw profile into a more readable tradeoff table\. This pulls the most useful columns forward and makes the dimensionality reduction explicit\.

In [64]:
# ---------------------------------------------------------------------
# Summarize geometry and dimensionality tradeoffs across all three
# ---------------------------------------------------------------------

representation_tradeoff_summary_df = (
    representation_geometry_profile_df[
        [
            "feature_set",
            "representation_name",
            "input_dimensions",
            "dimension_share_of_raw",
            "mean_knn_distance",
            "pairwise_distance_mean",
            "distance_concentration_ratio",
        ]
    ]
    .copy()
)

representation_tradeoff_summary_df["representation_family"] = (
    representation_tradeoff_summary_df["representation_name"].map(
        {
            "raw_scaled": "raw",
            "pca_80pct": "pca",
            "umap_pca_to_umap_10d": "umap",
        }
    )
)

representation_tradeoff_summary_display_df = (
    representation_tradeoff_summary_df.assign(
        dimension_share_of_raw=lambda df: df["dimension_share_of_raw"].round(3),
        mean_knn_distance=lambda df: df["mean_knn_distance"].round(4),
        pairwise_distance_mean=lambda df: df["pairwise_distance_mean"].round(4),
        distance_concentration_ratio=lambda df: df["distance_concentration_ratio"].round(4),
    )
    .sort_values(["feature_set", "input_dimensions"], ascending=[True, False])
    .reset_index(drop=True)
)

display(representation_tradeoff_summary_display_df)

print("Representation geometry tradeoffs summarized.")

,feature_set,representation_name,input_dimensions,dimension_share_of_raw,mean_knn_distance,pairwise_distance_mean,distance_concentration_ratio,representation_family
0,bus,raw_scaled,40,1.000,3.2080,8.2132,0.4807,raw
1,bus,pca_80pct,14,0.350,2.4256,7.3842,0.4803,pca
2,bus,umap_pca_to_umap_10d,10,0.250,0.2431,6.5851,0.6879,umap
3,fhvhv,raw_scaled,37,1.000,2.8493,8.0366,0.5009,raw
4,fhvhv,pca_80pct,13,0.351,1.9743,7.1893,0.5216,pca
5,fhvhv,umap_pca_to_umap_10d,10,0.270,0.2262,6.7766,0.6347,umap
6,multimodal,raw_scaled,142,1.000,7.7696,15.4869,0.4073,raw
7,multimodal,pca_80pct,46,0.324,6.0253,13.7610,0.4270,pca
8,multimodal,umap_pca_to_umap_10d,10,0.070,0.2081,5.6235,0.5528,umap
9,subway,raw_scaled,21,1.000,1.2251,5.2040,0.6498,raw


Representation geometry tradeoffs summarized.


Findings\. The tradeoff table makes the three representation roles easier to read\. Raw is the highest\-dimensional baseline by definition\. PCA keeps only about 32% to 52% of raw dimensions for four of the five feature sets, and just 32% for Multimodal, while preserving a geometry that still looks raw\-like\. UMAP is the most aggressive compression in every case, dropping Multimodal to just 7% of its original dimension count and the others to roughly 25% to 48%, but that compression comes with a much larger geometric shift\. So this section is not showing “three versions of the same space”; it is showing one baseline, one fidelity\-preserving reduction, and one much more transformed nonlinear space\.

Finally, turn the summary into a compact visual comparison\. This is not a winner chart; it is a quick way to see how the three representation families move between higher\-dimensional, more concentrated geometry and lower\-dimensional, more spread\-out geometry\.

In [65]:
# ---------------------------------------------------------------------
# Visualize geometry and practicality across all three representations
# ---------------------------------------------------------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots

plot_df = representation_tradeoff_summary_df.copy()
plot_df["representation_label"] = plot_df["representation_name"].map(
    {
        "raw_scaled": "Raw",
        "pca_80pct": "PCA",
        "umap_pca_to_umap_10d": "UMAP",
    }
)

representation_order = ["Raw", "PCA", "UMAP"]
feature_set_colors = {
    "bus": "#1f77b4",
    "subway": "#ff7f0e",
    "taxi": "#2ca02c",
    "fhvhv": "#d62728",
    "multimodal": "#9467bd",
}

geometry_fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Input dimensions",
        "Mean kNN distance",
        "Distance concentration ratio",
    ],
    horizontal_spacing=0.10,
)

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_df = (
        plot_df.loc[plot_df["feature_set"].eq(feature_set_name)]
        .copy()
    )
    feature_df["representation_label"] = pd.Categorical(
        feature_df["representation_label"],
        categories=representation_order,
        ordered=True,
    )
    feature_df = feature_df.sort_values("representation_label")

    common_style = dict(
        mode="lines+markers",
        name=feature_set_name,
        legendgroup=feature_set_name,
        marker={"size": 8},
        line={"width": 3, "color": feature_set_colors[feature_set_name]},
        showlegend=True,
    )

    geometry_fig.add_trace(
        go.Scatter(
            x=feature_df["representation_label"],
            y=feature_df["input_dimensions"],
            hovertemplate=(
                "Feature set: %{customdata[0]}<br>"
                "Representation: %{x}<br>"
                "Dimensions: %{y}"
                "<extra></extra>"
            ),
            customdata=np.column_stack([feature_df["feature_set"]]),
            **common_style,
        ),
        row=1,
        col=1,
    )

    geometry_fig.add_trace(
        go.Scatter(
            x=feature_df["representation_label"],
            y=feature_df["mean_knn_distance"],
            hovertemplate=(
                "Feature set: %{customdata[0]}<br>"
                "Representation: %{x}<br>"
                "Mean kNN distance: %{y:.4f}"
                "<extra></extra>"
            ),
            customdata=np.column_stack([feature_df["feature_set"]]),
            showlegend=False,
            **{k: v for k, v in common_style.items() if k != "showlegend"},
        ),
        row=1,
        col=2,
    )

    geometry_fig.add_trace(
        go.Scatter(
            x=feature_df["representation_label"],
            y=feature_df["distance_concentration_ratio"],
            hovertemplate=(
                "Feature set: %{customdata[0]}<br>"
                "Representation: %{x}<br>"
                "Distance concentration ratio: %{y:.4f}"
                "<extra></extra>"
            ),
            customdata=np.column_stack([feature_df["feature_set"]]),
            showlegend=False,
            **{k: v for k, v in common_style.items() if k != "showlegend"},
        ),
        row=1,
        col=3,
    )

geometry_fig.update_layout(
    title="Geometry and Practicality Across Raw, PCA, and UMAP",
    width=950,
    height=460,
    template="plotly_white",
    legend={
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
    },
    margin={"t": 70, "b": 95, "l": 55, "r": 30},
)

geometry_fig.update_yaxes(title_text="Dimensions", row=1, col=1)
geometry_fig.update_yaxes(title_text="Mean kNN distance", row=1, col=2)
geometry_fig.update_yaxes(title_text="Concentration ratio", row=1, col=3)

geometry_fig.show()

Findings\. The plot makes the tradeoff intuitive\. In the left panel, UMAP is consistently the smallest representation, with PCA sitting between raw and UMAP as the middle\-ground compression option\. In the center panel, raw and PCA remain relatively close to one another, while UMAP pushes mean local distances into a completely different scale\. In the right panel, the contrast sharpens further: Bus, Taxi, FHVHV, and Multimodal all become more distance\-concentrated under UMAP than under either raw or PCA, while Subway is the one exception where UMAP reduces concentration sharply\. Taken together, the figure reinforces the basic interpretation of this notebook: PCA is the compressed representation that stays closest to raw geometry, while UMAP is the compact nonlinear representation that buys stronger transformation at the cost of geometric continuity\.

### Summarize tradeoffs

The previous blocks answered narrower questions: how well each reduced representation preserves raw local neighborhoods, and what kind of geometry each representation creates\. This subsection pulls those results together so the tradeoffs are visible in one place, and to make the strengths and weaknesses of raw\_scaled, pca\_80pct, and umap\_pca\_to\_umap\_10d easy to compare before we move into downstream anomaly workflows\.

The summary should keep two ideas in view at the same time\. First, raw\_scaled is the source baseline, so it is not expected to “beat” the reduced representations on compression\. Second, PCA and UMAP are doing different jobs: PCA is the fidelity\-preserving linear reduction, while UMAP is the more aggressive nonlinear restructuring\. The useful question is therefore not just which representation scores highest on one metric, but what each one buys us in exchange for its distortions and dimensional footprint\.

The first block assembles a single comparison table across all three representations\. It combines neighborhood fidelity, geometry behavior, and dimensionality into one row per feature set and representation\.

In [66]:
# ---------------------------------------------------------------------
# Assemble the final representation tradeoff summary
# ---------------------------------------------------------------------

representation_tradeoff_summary_df = (
    representation_geometry_profile_df[
        [
            "feature_set",
            "representation_name",
            "input_dimensions",
            "dimension_share_of_raw",
            "mean_knn_distance",
            "pairwise_distance_mean",
            "distance_concentration_ratio",
        ]
    ]
    .merge(
        representation_trustworthiness_df[
            [
                "feature_set",
                "representation_name",
                "trustworthiness_score",
            ]
        ],
        on=["feature_set", "representation_name"],
        how="left",
    )
    .merge(
        representation_neighbor_overlap_df[
            [
                "feature_set",
                "representation_name",
                "mean_neighbor_overlap_share",
                "median_neighbor_overlap_share",
                "p10_neighbor_overlap_share",
            ]
        ],
        on=["feature_set", "representation_name"],
        how="left",
    )
)

representation_tradeoff_summary_df["representation_family"] = (
    representation_tradeoff_summary_df["representation_name"].map(
        {
            "raw_scaled": "raw",
            "pca_80pct": "pca",
            "umap_pca_to_umap_10d": "umap",
        }
    )
)

representation_tradeoff_summary_display_df = (
    representation_tradeoff_summary_df.assign(
        dimension_share_of_raw=lambda df: df["dimension_share_of_raw"].round(3),
        trustworthiness_score=lambda df: df["trustworthiness_score"].round(4),
        mean_neighbor_overlap_share=lambda df: df["mean_neighbor_overlap_share"].round(4),
        median_neighbor_overlap_share=lambda df: df["median_neighbor_overlap_share"].round(4),
        p10_neighbor_overlap_share=lambda df: df["p10_neighbor_overlap_share"].round(4),
        mean_knn_distance=lambda df: df["mean_knn_distance"].round(4),
        pairwise_distance_mean=lambda df: df["pairwise_distance_mean"].round(4),
        distance_concentration_ratio=lambda df: df["distance_concentration_ratio"].round(4),
    )
    .sort_values(["feature_set", "representation_family"])
    .reset_index(drop=True)
)

display(representation_tradeoff_summary_display_df)

print("Representation tradeoff summary assembled.")

,feature_set,representation_name,input_dimensions,dimension_share_of_raw,mean_knn_distance,pairwise_distance_mean,distance_concentration_ratio,trustworthiness_score,mean_neighbor_overlap_share,median_neighbor_overlap_share,p10_neighbor_overlap_share,representation_family
0,bus,pca_80pct,14,0.350,2.4256,7.3842,0.4803,0.9985,0.6096,0.6000,0.4000,pca
1,bus,raw_scaled,40,1.000,3.2080,8.2132,0.4807,NaN,NaN,NaN,NaN,raw
2,bus,umap_pca_to_umap_10d,10,0.250,0.2431,6.5851,0.6879,0.9788,0.2733,0.2667,0.0667,umap
3,fhvhv,pca_80pct,13,0.351,1.9743,7.1893,0.5216,0.9967,0.5236,0.5333,0.3333,pca
4,fhvhv,raw_scaled,37,1.000,2.8493,8.0366,0.5009,NaN,NaN,NaN,NaN,raw
5,fhvhv,umap_pca_to_umap_10d,10,0.270,0.2262,6.7766,0.6347,0.9734,0.2630,0.2667,0.0667,umap
6,multimodal,pca_80pct,46,0.324,6.0253,13.7610,0.4270,0.9992,0.6665,0.6667,0.4667,pca
7,multimodal,raw_scaled,142,1.000,7.7696,15.4869,0.4073,NaN,NaN,NaN,NaN,raw
8,multimodal,umap_pca_to_umap_10d,10,0.070,0.2081,5.6235,0.5528,0.9534,0.2775,0.2667,0.0667,umap
9,subway,pca_80pct,11,0.524,0.8500,4.6745,0.6773,0.9979,0.6193,0.6000,0.4000,pca


Representation tradeoff summary assembled.


Findings\. The combined tradeoff table makes the representation roles very clear\. raw\_scaled remains the full geometric baseline, pca\_80pct is the compressed representation that stays closest to raw neighborhood structure, and umap\_pca\_to\_umap\_10d is the most aggressive reduction\. Across all five feature sets, PCA keeps much higher trustworthiness and neighbor overlap than UMAP while still cutting dimensionality materially: for example, Bus falls from 40 raw dimensions to 14 in PCA, while preserving trustworthiness of 0\.999 and mean neighbor overlap of 0\.610\. UMAP compresses further to 10 dimensions, but that comes with much weaker neighborhood fidelity, especially for Bus, Taxi, FHVHV, and Multimodal\. Subway is the one case where UMAP produces a notably different geometry in a potentially favorable way, with a much lower distance\-concentration ratio than either raw or PCA, but even there PCA still preserves raw neighborhoods better\.

Now turn that summary into a compact scorecard\. This is a visual aid for seeing which representations dominate on fidelity, which dominate on compression, and where the tradeoffs are uneven across modalities\.

In [67]:
# ---------------------------------------------------------------------
# Visualize the final representation tradeoff scorecard
# ---------------------------------------------------------------------

scorecard_df = representation_tradeoff_summary_df.copy()

scorecard_df["row_label"] = (
    scorecard_df["feature_set"] + " | " + scorecard_df["representation_family"]
)

scorecard_df["compression_gain"] = 1 - scorecard_df["dimension_share_of_raw"]

scorecard_metrics_df = scorecard_df[
    [
        "row_label",
        "trustworthiness_score",
        "mean_neighbor_overlap_share",
        "compression_gain",
        "distance_concentration_ratio",
    ]
].copy()

# Higher is better for the first three; lower is better for concentration.
scorecard_normalized_df = scorecard_metrics_df.copy()
scorecard_normalized_df["trustworthiness_score"] = (
    scorecard_metrics_df["trustworthiness_score"]
    / scorecard_metrics_df["trustworthiness_score"].max()
)
scorecard_normalized_df["mean_neighbor_overlap_share"] = (
    scorecard_metrics_df["mean_neighbor_overlap_share"]
    / scorecard_metrics_df["mean_neighbor_overlap_share"].max()
)
scorecard_normalized_df["compression_gain"] = (
    scorecard_metrics_df["compression_gain"]
    / scorecard_metrics_df["compression_gain"].max()
)
scorecard_normalized_df["distance_concentration_ratio"] = 1 - (
    scorecard_metrics_df["distance_concentration_ratio"]
    / scorecard_metrics_df["distance_concentration_ratio"].max()
)

scorecard_plot_df = scorecard_normalized_df.melt(
    id_vars="row_label",
    var_name="metric",
    value_name="relative_score",
)

scorecard_fig = px.imshow(
    scorecard_plot_df.pivot(
        index="row_label",
        columns="metric",
        values="relative_score",
    ),
    color_continuous_scale="RdYlGn",
    aspect="auto",
    origin="lower",
)

scorecard_fig.update_yaxes(
    automargin=True,
    ticklabelstandoff=14,
)

scorecard_fig.update_layout(
    title="Representation Tradeoff Scorecard",
    width=920,
    height=620,
    template="plotly_white",
    margin={"t": 70, "b": 50, "l": 170, "r": 30},
    coloraxis_colorbar_title="Relative score",
)

scorecard_fig.update_xaxes(title_text=None)
scorecard_fig.update_yaxes(title_text=None)

scorecard_fig.show()

Findings\. The scorecard reinforces the same broad pattern but also reveals why the representation choice is not one\-dimensional\. PCA dominates UMAP on the two raw\-fidelity metrics for every feature set, while UMAP predictably wins compression gain because it is the smallest representation in every case\. Distance concentration is more mixed: UMAP is best only for Subway, while raw or PCA remain better for the other feature sets\. So the heatmap does not point to a universal winner across all criteria; it shows a stable tradeoff structure in which PCA is the fidelity\-preserving reduction and UMAP is the more aggressively compressed alternative\.

### Record downstream handoff set

This last step records the candidate representation set that will move forward into the anomaly notebooks\. By this point, the notebook has already done the substantive work: it established the canonical UMAP construction workflow, compared PCA and UMAP against raw neighborhood structure, and summarized the tradeoffs across the three candidate spaces\. What remains is simply to write down the approved handoff set in one reusable table\.
The handoff should stay practical\. For each feature set, we keep the raw scaled baseline, the pca\_80pct linear reduction, and the canonical umap\_pca\_to\_umap\_10d nonlinear reduction\. The next notebooks can then load these assets directly without rebuilding the comparison logic\.
The block below writes one compact handoff table for downstream use\. It records the candidate representation name, its role, dimensionality, and the path that the anomaly notebooks should load\.

In [68]:
# ---------------------------------------------------------------------
# Record downstream candidate representation handoff set
# ---------------------------------------------------------------------

REPRESENTATION_HANDOFF_PATH = FINAL_OUTPUT_DIR / "candidate_representation_handoff.csv"

representation_handoff_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    comparison_path = COMPARISON_TABLE_PATHS_BY_SET[feature_set_name]
    comparison_columns = pd.read_parquet(comparison_path).columns.tolist()

    raw_columns = [
        column_name
        for column_name in comparison_columns
        if str(column_name).startswith("raw__")
    ]
    pca_columns = [
        column_name
        for column_name in comparison_columns
        if str(column_name).startswith("pca__")
    ]
    umap_columns = [
        column_name
        for column_name in comparison_columns
        if str(column_name).startswith("umap__umap_")
    ]

    umap_output_path = (
        INTERMEDIATE_OUTPUT_DIR / f"{feature_set_name}_umap_pca_to_umap_10d.parquet"
    )

    representation_handoff_records.extend(
        [
            {
                "feature_set": feature_set_name,
                "representation_name": "raw_scaled",
                "representation_role": "source_baseline",
                "input_dimensions": len(raw_columns),
                "path": str(EVALUATION_SAMPLE_SCALED_PATHS_BY_SET[feature_set_name]),
            },
            {
                "feature_set": feature_set_name,
                "representation_name": "pca_80pct",
                "representation_role": "linear_reduced_candidate",
                "input_dimensions": len(pca_columns),
                "path": str(PCA_MODELING_EVAL_SAMPLE_PATHS_BY_SET[feature_set_name]),
            },
            {
                "feature_set": feature_set_name,
                "representation_name": "umap_pca_to_umap_10d",
                "representation_role": "nonlinear_reduced_candidate",
                "input_dimensions": len(umap_columns),
                "path": str(umap_output_path),
            },
        ]
    )

representation_handoff_df = pd.DataFrame(representation_handoff_records)
representation_handoff_df.to_csv(REPRESENTATION_HANDOFF_PATH, index=False)

display(representation_handoff_df)

print(f"Candidate representation handoff written to {REPRESENTATION_HANDOFF_PATH}.")

,feature_set,representation_name,representation_role,input_dimensions,path
0,bus,raw_scaled,source_baseline,40,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
1,bus,pca_80pct,linear_reduced_candidate,14,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
2,bus,umap_pca_to_umap_10d,nonlinear_reduced_candidate,10,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
3,subway,raw_scaled,source_baseline,21,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
4,subway,pca_80pct,linear_reduced_candidate,11,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
5,subway,umap_pca_to_umap_10d,nonlinear_reduced_candidate,10,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
6,taxi,raw_scaled,source_baseline,38,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
7,taxi,pca_80pct,linear_reduced_candidate,15,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
8,taxi,umap_pca_to_umap_10d,nonlinear_reduced_candidate,10,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
9,fhvhv,raw_scaled,source_baseline,37,/datasets/_deepnote_work/pipeline_data/3.2.4.i...


Candidate representation handoff written to /datasets/_deepnote_work/pipeline_data/3.2.4.final_tables/candidate_representation_handoff.csv.


Findings\. The approved downstream handoff set is now written to disk for all five feature sets\. Each modality carries forward the same three candidate spaces: raw\_scaled, pca\_80pct, and umap\_pca\_to\_umap\_10d, along with their roles, dimensions, and load paths\. This gives the anomaly notebooks a clean, reusable representation inventory instead of forcing them to reconstruct the selection logic\.

> ▣ Decision\. The downstream candidate representation set will retain all three approved spaces: raw\_scaled as the source baseline, pca\_80pct as the primary linear reduced representation, and umap\_pca\_to\_umap\_10d as the nonlinear reduced representation\. PCA is the stronger fidelity\-preserving reduction, while UMAP remains useful as an alternative nonlinear space rather than the default replacement for PCA or raw geometry\.

### Section Summary

> This section completed the core representation comparison across raw, PCA, and UMAP candidate spaces\. The results were consistent across the notebook: pca\_80pct preserved raw neighborhood structure much better than umap\_pca\_to\_umap\_10d, while still reducing dimensionality substantially\. UMAP provided the most aggressive compression and the strongest geometric transformation, but that transformation usually came with meaningful loss of raw local fidelity\. Subway remained the main partial exception, where UMAP produced a more favorable distance\-concentration profile, though PCA still preserved raw neighborhoods better\.

The practical outcome is not a single winner\-take\-all representation\. Instead, the notebook records a three\-part downstream handoff set: raw scaled space as the source baseline, pca\_80pct as the fidelity\-preserving linear reduction, and umap\_pca\_to\_umap\_10d as the nonlinear reduced candidate\. Those are the representation spaces that should move forward into the anomaly and clustering notebooks\.

## 3\.2\.4\.6 Fit and Export Full Selected UMAP Representations

This section turns the selected nonlinear representation into a full downstream asset, but in a way that fits the notebook’s memory budget\. Instead of refitting UMAP on the full panel, we recreate the selected pca\_to\_umap 10D map from the shared 50,000\-row evaluation sample and then apply that learned map to the full dataset in chunks\.

That gives us a full modeling\-ready UMAP table for each feature set while staying consistent with the representation recipe already selected earlier in the notebook\. The last steps then attach row metadata, validate the exported assets, and leave a clean nonlinear handoff for downstream clustering and anomaly workflows\.

### Fit full selected UMAP embeddings

We begin by fitting the selected UMAP recipe on the shared evaluation sample for each feature set and caching the fitted model objects to disk\. These fitted sample models are the reusable maps that the next block will apply to the full PCA tables\.

In [69]:
# ---------------------------------------------------------------------
# Fit sample UMAP models for full-panel transform
# ---------------------------------------------------------------------

REBUILD_FULL_SELECTED_UMAP = False

SELECTED_UMAP_WORKFLOW = "pca_to_umap"
SELECTED_UMAP_DIMENSIONS = 10
FULL_SELECTED_UMAP_FIT_SAMPLE_ROWS = REPRESENTATION_EVALUATION_SAMPLE_ROWS

FULL_SELECTED_UMAP_MODEL_PROGRESS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "full_selected_umap_model_progress.parquet"
)

FULL_SELECTED_UMAP_MODEL_PATHS_BY_SET = {
    feature_set: INTERMEDIATE_OUTPUT_DIR / f"{feature_set}_umap_{SELECTED_UMAP_WORKFLOW}_{SELECTED_UMAP_DIMENSIONS}d_sample_model.pkl"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

if "PCA_MODELING_EVAL_SAMPLE_PATHS_BY_SET" not in globals():
    PCA_MODELING_EVAL_SAMPLE_PATHS_BY_SET = {
        feature_set: INTERMEDIATE_OUTPUT_DIR / f"{feature_set}_pca_modeling_eval_sample.parquet"
        for feature_set in MODEL_FEATURE_SET_NAMES
    }

full_selected_umap_model_progress_df = (
    pd.read_parquet(FULL_SELECTED_UMAP_MODEL_PROGRESS_PATH)
    if FULL_SELECTED_UMAP_MODEL_PROGRESS_PATH.exists()
    else pd.DataFrame()
)

completed_model_paths = set()
if not full_selected_umap_model_progress_df.empty:
    completed_model_paths = set(
        full_selected_umap_model_progress_df.loc[
            full_selected_umap_model_progress_df["status"].eq("pass"),
            "model_path",
        ].tolist()
    )

full_selected_umap_model_records = []
total_jobs = len(MODEL_FEATURE_SET_NAMES)

for job_index, feature_set_name in enumerate(MODEL_FEATURE_SET_NAMES, start=1):
    model_path = FULL_SELECTED_UMAP_MODEL_PATHS_BY_SET[feature_set_name]
    sample_input_path = PCA_MODELING_EVAL_SAMPLE_PATHS_BY_SET[feature_set_name]

    if (
        not REBUILD_FULL_SELECTED_UMAP
        and model_path.exists()
        and str(model_path) in completed_model_paths
    ):
        sample_row_count = read_parquet_row_count(sample_input_path)
        sample_input_columns = [
            column_name
            for column_name in pd.read_parquet(sample_input_path, columns=None).columns
            if column_name != "modeling_row_id"
        ]

        full_selected_umap_model_records.append(
            {
                "feature_set": feature_set_name,
                "sample_rows": sample_row_count,
                "input_columns": len(sample_input_columns),
                "model_action": "loaded_existing",
                "elapsed_seconds": np.nan,
                "status": "pass",
                "model_path": str(model_path),
            }
        )
        print(f"[{job_index}/{total_jobs}] Reused sample model for {feature_set_name}.")
        continue

    sample_input_df = pd.read_parquet(sample_input_path)
    input_columns = [
        column_name
        for column_name in sample_input_df.columns
        if column_name != "modeling_row_id"
    ]
    sample_X = sample_input_df[input_columns].to_numpy(dtype="float32", copy=False)

    print(
        f"[{job_index}/{total_jobs}] Fitting sample UMAP model for {feature_set_name} "
        f"({len(sample_input_df):,} rows x {len(input_columns)} inputs)."
    )

    start_time = time.time()

    umap_model = umap.UMAP(
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        n_components=SELECTED_UMAP_DIMENSIONS,
        metric=UMAP_METRIC,
        random_state=UMAP_RANDOM_STATE,
    )
    umap_model.fit(sample_X)

    elapsed_seconds = round(time.time() - start_time, 1)

    with open(model_path, "wb") as model_file:
        pickle.dump(umap_model, model_file)

    model_record = {
        "feature_set": feature_set_name,
        "sample_rows": len(sample_input_df),
        "input_columns": len(input_columns),
        "model_action": "fit_sample",
        "elapsed_seconds": elapsed_seconds,
        "status": "pass" if model_path.exists() else "review",
        "model_path": str(model_path),
    }
    full_selected_umap_model_records.append(model_record)

    pd.DataFrame(full_selected_umap_model_records).to_parquet(
        FULL_SELECTED_UMAP_MODEL_PROGRESS_PATH,
        index=False,
    )

    print(
        f"[{job_index}/{total_jobs}] Finished sample model for {feature_set_name} "
        f"in {elapsed_seconds:.1f}s."
    )

    del sample_input_df, sample_X, umap_model

full_selected_umap_model_summary_df = pd.DataFrame(full_selected_umap_model_records)

if full_selected_umap_model_summary_df.empty and FULL_SELECTED_UMAP_MODEL_PROGRESS_PATH.exists():
    full_selected_umap_model_summary_df = pd.read_parquet(
        FULL_SELECTED_UMAP_MODEL_PROGRESS_PATH
    )

display(full_selected_umap_model_summary_df)

assert not full_selected_umap_model_summary_df.empty, (
    "No sample UMAP models were recorded."
)
assert full_selected_umap_model_summary_df["status"].eq("pass").all(), (
    "One or more sample UMAP models did not complete successfully."
)

print("Sample UMAP models are ready.")

[1/5] Reused sample model for bus.
[2/5] Reused sample model for subway.
[3/5] Reused sample model for taxi.
[4/5] Reused sample model for fhvhv.
[5/5] Reused sample model for multimodal.


,feature_set,sample_rows,input_columns,model_action,elapsed_seconds,status,model_path
0,bus,50000,14,loaded_existing,NaN,pass,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
1,subway,50000,11,loaded_existing,NaN,pass,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
2,taxi,50000,15,loaded_existing,NaN,pass,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
3,fhvhv,50000,13,loaded_existing,NaN,pass,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
4,multimodal,50000,46,loaded_existing,NaN,pass,/datasets/_deepnote_work/pipeline_data/3.2.4.i...


Sample UMAP models are ready.


Findings\. The sample UMAP models were fit successfully for all five feature sets using the shared 50,000\-row evaluation sample\. Fit times stayed within a fairly tight band, from about 82 seconds for FHVHV to about 108 seconds for Subway, even though the PCA input widths ranged from 11 columns for Subway to 46 for Multimodal\. This confirms that the notebook can recreate the selected pca\_to\_umap 10D mapping in transformable form without full\-panel fitting, and it leaves one cached sample model per modality ready for chunked application to the full dataset\.

### Transform full PCA tables into 10D UMAP coordinates

With the sample models cached, the next step is to map the full PCA tables into the selected 10D UMAP space\. The transform is written in chunks so the notebook can move through the full panel without trying to hold an entire feature set in memory at once\.

In [70]:
# ---------------------------------------------------------------------
# Transform full PCA tables into 10D UMAP coordinates
# ---------------------------------------------------------------------

FULL_SELECTED_UMAP_TRANSFORM_CHUNK_ROWS = 50_000

FULL_SELECTED_UMAP_PROGRESS_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "full_selected_umap_fit_progress.parquet"
)

FULL_SELECTED_UMAP_OUTPUT_PATHS_BY_SET = {
    feature_set: INTERMEDIATE_OUTPUT_DIR / f"{feature_set}_umap_{SELECTED_UMAP_WORKFLOW}_{SELECTED_UMAP_DIMENSIONS}d_full.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

if "PCA_MODELING_SCORE_PATHS_BY_SET" not in globals():
    PCA_MODELING_SCORE_PATHS_BY_SET = {
        feature_set: PCA_ASSET_DIR / f"{feature_set}_pca_modeling_scores.parquet"
        for feature_set in MODEL_FEATURE_SET_NAMES
    }

full_selected_umap_progress_df = (
    pd.read_parquet(FULL_SELECTED_UMAP_PROGRESS_PATH)
    if FULL_SELECTED_UMAP_PROGRESS_PATH.exists()
    else pd.DataFrame()
)

completed_output_paths = set()
if not full_selected_umap_progress_df.empty:
    completed_output_paths = set(
        full_selected_umap_progress_df.loc[
            full_selected_umap_progress_df["status"].eq("pass"),
            "output_path",
        ].tolist()
    )

full_selected_umap_fit_records = []
total_jobs = len(MODEL_FEATURE_SET_NAMES)

for job_index, feature_set_name in enumerate(MODEL_FEATURE_SET_NAMES, start=1):
    model_path = FULL_SELECTED_UMAP_MODEL_PATHS_BY_SET[feature_set_name]
    input_path = PCA_MODELING_SCORE_PATHS_BY_SET[feature_set_name]
    output_path = FULL_SELECTED_UMAP_OUTPUT_PATHS_BY_SET[feature_set_name]

    if (
        not REBUILD_FULL_SELECTED_UMAP
        and output_path.exists()
        and str(output_path) in completed_output_paths
    ):
        full_selected_umap_fit_records.append(
            {
                "feature_set": feature_set_name,
                "umap_workflow": SELECTED_UMAP_WORKFLOW,
                "umap_dimensions": SELECTED_UMAP_DIMENSIONS,
                "transform_rows": read_parquet_row_count(output_path),
                "output_action": "loaded_existing",
                "elapsed_seconds": np.nan,
                "status": "pass",
                "output_path": str(output_path),
            }
        )
        print(f"[{job_index}/{total_jobs}] Reused transformed full UMAP for {feature_set_name}.")
        continue

    with open(model_path, "rb") as model_file:
        umap_model = pickle.load(model_file)

    input_schema_columns = get_parquet_columns(input_path)
    input_columns = [
        column_name
        for column_name in input_schema_columns
        if column_name != "modeling_row_id"
    ]

    total_full_rows = read_parquet_row_count(input_path)
    embedding_columns = [f"umap_{i + 1}" for i in range(SELECTED_UMAP_DIMENSIONS)]

    print(
        f"[{job_index}/{total_jobs}] Transforming full PCA table for {feature_set_name} "
        f"({total_full_rows:,} rows x {len(input_columns)} inputs)."
    )

    if output_path.exists():
        output_path.unlink()

    input_path_sql = sql_path(input_path)
    input_columns_sql = ", ".join(
        [duckdb_identifier("modeling_row_id")]
        + [duckdb_identifier(column_name) for column_name in input_columns]
    )

    start_time = time.time()
    written_rows = 0
    writer = None

    with duckdb.connect() as con:
        for offset in range(0, total_full_rows, FULL_SELECTED_UMAP_TRANSFORM_CHUNK_ROWS):
            batch_number = offset // FULL_SELECTED_UMAP_TRANSFORM_CHUNK_ROWS + 1

            batch_df = con.execute(
                f"""
                SELECT {input_columns_sql}
                FROM read_parquet('{input_path_sql}')
                ORDER BY modeling_row_id
                LIMIT {FULL_SELECTED_UMAP_TRANSFORM_CHUNK_ROWS}
                OFFSET {offset}
                """
            ).fetchdf()

            if batch_df.empty:
                break

            batch_X = batch_df[input_columns].to_numpy(dtype="float32", copy=False)
            batch_embedding = umap_model.transform(batch_X)

            batch_output_df = pd.DataFrame(
                batch_embedding,
                columns=embedding_columns,
            )
            batch_output_df.insert(
                0,
                "modeling_row_id",
                batch_df["modeling_row_id"].to_numpy(),
            )

            batch_table = pa.Table.from_pandas(batch_output_df, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(output_path, batch_table.schema)

            writer.write_table(batch_table)
            written_rows += len(batch_output_df)

            print(
                f"[{job_index}/{total_jobs}] {feature_set_name}: transformed "
                f"{written_rows:,}/{total_full_rows:,} rows "
                f"(batch {batch_number})."
            )

            del batch_df, batch_X, batch_embedding, batch_output_df, batch_table

    if writer is not None:
        writer.close()

    elapsed_seconds = round(time.time() - start_time, 1)

    fit_record = {
        "feature_set": feature_set_name,
        "umap_workflow": SELECTED_UMAP_WORKFLOW,
        "umap_dimensions": SELECTED_UMAP_DIMENSIONS,
        "transform_rows": written_rows,
        "output_action": "sample_fit_transform_full",
        "elapsed_seconds": elapsed_seconds,
        "status": "pass" if written_rows == total_full_rows else "review",
        "output_path": str(output_path),
    }
    full_selected_umap_fit_records.append(fit_record)

    pd.DataFrame(full_selected_umap_fit_records).to_parquet(
        FULL_SELECTED_UMAP_PROGRESS_PATH,
        index=False,
    )

    print(
        f"[{job_index}/{total_jobs}] Finished full transform for {feature_set_name} "
        f"in {elapsed_seconds:.1f}s."
    )

    del umap_model

full_selected_umap_fit_summary_df = pd.DataFrame(full_selected_umap_fit_records)

if full_selected_umap_fit_summary_df.empty and FULL_SELECTED_UMAP_PROGRESS_PATH.exists():
    full_selected_umap_fit_summary_df = pd.read_parquet(FULL_SELECTED_UMAP_PROGRESS_PATH)

display(full_selected_umap_fit_summary_df)

assert not full_selected_umap_fit_summary_df.empty, (
    "No full selected UMAP outputs were recorded."
)
assert full_selected_umap_fit_summary_df["status"].eq("pass").all(), (
    "One or more full selected UMAP transforms did not complete successfully."
)

print("Full selected UMAP coordinate tables are ready.")

[1/5] Reused transformed full UMAP for bus.
[2/5] Reused transformed full UMAP for subway.
[3/5] Reused transformed full UMAP for taxi.
[4/5] Reused transformed full UMAP for fhvhv.
[5/5] Reused transformed full UMAP for multimodal.


,feature_set,umap_workflow,umap_dimensions,transform_rows,output_action,elapsed_seconds,status,output_path
0,bus,pca_to_umap,10,1472947,loaded_existing,NaN,pass,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
1,subway,pca_to_umap,10,911455,loaded_existing,NaN,pass,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
2,taxi,pca_to_umap,10,1541800,loaded_existing,NaN,pass,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
3,fhvhv,pca_to_umap,10,1541800,loaded_existing,NaN,pass,/datasets/_deepnote_work/pipeline_data/3.2.4.i...
4,multimodal,pca_to_umap,10,1541800,loaded_existing,NaN,pass,/datasets/_deepnote_work/pipeline_data/3.2.4.i...


Full selected UMAP coordinate tables are ready.


Findings\. The full nonlinear modeling assets were created successfully for all five feature sets using the sample\-fit, full\-transform workflow\. Each modality was transformed into the selected pca\_to\_umap 10D space at full row count: 1\.47M Bus rows, 0\.91M Subway rows, and 1\.54M rows each for Taxi, FHVHV, and Multimodal\. Runtime was substantial but manageable, ranging from about 591 seconds for Bus to about 1,183 seconds for Subway, which confirms that chunked full\-panel transform is operationally feasible even though full\-panel UMAP refitting was not\.

### Export modeling\-ready UMAP tables

Now that the full\-panel UMAP coordinates exist, the remaining work is packaging rather than fitting\. This block writes the final nonlinear representation tables in the same structural style used for Raw in 3\.1\.1 and PCA in 3\.2\.1: modeling\_row\_id plus the UMAP coordinate columns only\. Row metadata remains available from the shared 3\.1\.1 metadata assets and does not need to be duplicated inside the UMAP representation tables\.

In [71]:
# ---------------------------------------------------------------------
# Export modeling-ready full selected UMAP tables
# ---------------------------------------------------------------------

FULL_SELECTED_UMAP_FINAL_PATHS_BY_SET = {
    feature_set: FINAL_OUTPUT_DIR / f"{feature_set}_umap_pca_to_umap_10d.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

full_selected_umap_export_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    embedding_path = FULL_SELECTED_UMAP_OUTPUT_PATHS_BY_SET[feature_set_name]
    output_path = FULL_SELECTED_UMAP_FINAL_PATHS_BY_SET[feature_set_name]

    embedding_df = pd.read_parquet(embedding_path).copy()

    umap_coordinate_columns = [
        column_name
        for column_name in embedding_df.columns
        if str(column_name).startswith("umap_")
    ]

    export_columns = ["modeling_row_id", *umap_coordinate_columns]
    export_df = embedding_df.loc[:, export_columns].copy()

    output_path.parent.mkdir(parents=True, exist_ok=True)
    export_df.to_parquet(output_path, index=False)

    written_columns = list(export_df.columns)

    full_selected_umap_export_records.append(
        {
            "feature_set": feature_set_name,
            "export_rows": len(export_df),
            "umap_coordinate_columns": len(umap_coordinate_columns),
            "export_column_count": len(written_columns),
            "contains_only_model_id_and_umap": (
                written_columns == ["modeling_row_id", *umap_coordinate_columns]
            ),
            "output_action": "written",
            "status": "pass" if output_path.exists() else "review",
        }
    )

full_selected_umap_export_df = pd.DataFrame(full_selected_umap_export_records)

display(full_selected_umap_export_df)

assert full_selected_umap_export_df["status"].eq("pass").all(), (
    "One or more full selected UMAP exports was not written successfully."
)
assert full_selected_umap_export_df["contains_only_model_id_and_umap"].all(), (
    "One or more exported UMAP tables includes unexpected non-UMAP columns."
)

print("Modeling-ready UMAP representation tables exported.")

,feature_set,export_rows,umap_coordinate_columns,export_column_count,contains_only_model_id_and_umap,output_action,status
0,bus,1472947,10,11,True,written,pass
1,subway,911455,10,11,True,written,pass
2,taxi,1541800,10,11,True,written,pass
3,fhvhv,1541800,10,11,True,written,pass
4,multimodal,1541800,10,11,True,written,pass


Modeling-ready UMAP representation tables exported.


Findings\. The final UMAP representation tables were written successfully for all five feature sets\. Each export now follows the same structural contract as the other representation assets: modeling\_row\_id plus the 10 UMAP coordinate columns only\. This keeps the nonlinear representation tables clean and modeling\-ready, while leaving row metadata in the shared 3\.1\.1 metadata assets\.

### Validate exported UMAP assets

Before closing the section, let’s confirm that the exported UMAP tables are complete and aligned\. Before closing the section, let’s confirm that the exported UMAP tables are complete and aligned\. The final check verifies row counts, coordinate columns, and modeling\_row\_id integrity, while also confirming that the exported tables do not carry duplicated metadata fields\.

In [72]:
# ---------------------------------------------------------------------
# Validate exported full selected UMAP assets
# ---------------------------------------------------------------------

full_selected_umap_validation_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    output_path = FULL_SELECTED_UMAP_FINAL_PATHS_BY_SET[feature_set_name]
    export_df = pd.read_parquet(output_path)

    export_columns = list(export_df.columns)

    umap_coordinate_columns = [
        column_name
        for column_name in export_columns
        if str(column_name).startswith("umap_")
    ]

    duplicate_modeling_rows = int(
        export_df["modeling_row_id"].duplicated().sum()
    )

    null_coordinate_rows = int(
        export_df[umap_coordinate_columns].isnull().any(axis=1).sum()
    )

    expected_rows = read_parquet_row_count(ROW_METADATA_PATHS_BY_SET[feature_set_name])

    unexpected_non_umap_columns = [
        column_name
        for column_name in export_columns
        if column_name != "modeling_row_id"
        and not str(column_name).startswith("umap_")
    ]

    status = (
        "pass"
        if (
            len(export_df) == expected_rows
            and len(umap_coordinate_columns) == SELECTED_UMAP_DIMENSIONS
            and duplicate_modeling_rows == 0
            and null_coordinate_rows == 0
            and len(unexpected_non_umap_columns) == 0
        )
        else "review"
    )

    full_selected_umap_validation_records.append(
        {
            "feature_set": feature_set_name,
            "export_rows": len(export_df),
            "expected_rows": expected_rows,
            "umap_dimensions": len(umap_coordinate_columns),
            "unexpected_non_umap_columns": (
                "none" if not unexpected_non_umap_columns else ", ".join(unexpected_non_umap_columns)
            ),
            "duplicate_modeling_rows": duplicate_modeling_rows,
            "null_coordinate_rows": null_coordinate_rows,
            "status": status,
        }
    )

full_selected_umap_validation_df = pd.DataFrame(full_selected_umap_validation_records)

display(full_selected_umap_validation_df)

assert full_selected_umap_validation_df["status"].eq("pass").all(), (
    "One or more exported full selected UMAP assets failed validation."
)

print("Exported UMAP representation assets validated.")

,feature_set,export_rows,expected_rows,umap_dimensions,unexpected_non_umap_columns,duplicate_modeling_rows,null_coordinate_rows,status
0,bus,1472947,1472947,10,none,0,0,pass
1,subway,911455,911455,10,none,0,0,pass
2,taxi,1541800,1541800,10,none,0,0,pass
3,fhvhv,1541800,1541800,10,none,0,0,pass
4,multimodal,1541800,1541800,10,none,0,0,pass


Exported UMAP representation assets validated.


Findings\. The exported full UMAP assets pass the final handoff checks\. Row counts match the expected metadata tables for every feature set, all 10 UMAP dimensions are present, there are no duplicate modeling\_row\_id values, no null coordinate rows, and no unexpected metadata columns embedded in the representation tables\. The 10D pca\_to\_umap representation is therefore fully materialized as a clean downstream modeling asset\.

## 3\.2\.4\.7 Export Evaluation Summaries

This final section packages the representation\-evaluation results into a compact set of reusable outputs\. By this point, the notebook has already done the substantive work: it evaluated UMAP dimensionality, compared raw\-to\-UMAP against PCA\-to\-UMAP construction workflows, compared raw, PCA, and UMAP representation behavior, and exported the selected full UMAP assets\. What remains is to write the summary tables and configuration metadata that later notebooks and final reporting will need\.

The exports here should stay focused and lightweight\. They should capture the main evaluation results, runtime summaries, and selected representation settings, without drifting into downstream modeling outputs such as clusters, anomaly scores, or labels\. The goal is to leave a clean trail of what was tested, what was selected, and what should be loaded next\.

The first block assembles the summary tables and configuration metadata into a single export plan\. This keeps the actual write step below short and explicit\.

In [73]:
# ---------------------------------------------------------------------
# Assemble evaluation summary export tables
# ---------------------------------------------------------------------

SELECTED_REPRESENTATION_CONFIGURATION_DF = pd.DataFrame(
    [
        {
            "setting_name": "selected_umap_workflow",
            "setting_value": "pca_to_umap",
        },
        {
            "setting_name": "selected_umap_dimensions",
            "setting_value": "10",
        },
        {
            "setting_name": "selected_pca_input_rule",
            "setting_value": "pca_80pct",
        },
        {
            "setting_name": "retained_representation_set",
            "setting_value": "raw_scaled, pca_80pct, umap_pca_to_umap_10d",
        },
    ]
)

REPRESENTATION_RUNTIME_SUMMARY_DF = pd.concat(
    [
        umap_dimensionality_fit_df.assign(
            experiment_family="umap_dimensionality"
        )[
            [
                "experiment_family",
                "feature_set",
                "input_space",
                "umap_dimensions",
                "elapsed_seconds",
            ]
        ],
        umap_workflow_fit_summary_df.assign(
            experiment_family="umap_workflow"
        )[
            [
                "experiment_family",
                "feature_set",
                "umap_workflow",
                "input_space",
                "input_dimensions",
                "elapsed_seconds",
            ]
        ],
    ],
    ignore_index=True,
    sort=False,
)

EVALUATION_SUMMARY_EXPORTS = {
    "umap_dimensionality_trustworthiness.csv": umap_trustworthiness_df,
    "umap_dimensionality_neighbor_retention.csv": umap_neighbor_retention_df,
    "umap_dimensionality_distance_behavior.csv": umap_distance_behavior_df,
    "umap_workflow_trustworthiness.csv": workflow_trustworthiness_df,
    "umap_workflow_neighbor_retention.csv": workflow_neighbor_retention_df,
    "umap_workflow_distance_behavior.csv": workflow_distance_behavior_df,
    "representation_tradeoff_summary.csv": representation_tradeoff_summary_df,
    "candidate_representation_handoff.csv": representation_handoff_df,
    "representation_runtime_summary.csv": REPRESENTATION_RUNTIME_SUMMARY_DF,
    "selected_representation_configuration.csv": SELECTED_REPRESENTATION_CONFIGURATION_DF,
}

evaluation_summary_export_plan_df = pd.DataFrame(
    [
        {
            "file_name": file_name,
            "rows": len(df),
            "columns": len(df.columns),
        }
        for file_name, df in EVALUATION_SUMMARY_EXPORTS.items()
    ]
).sort_values("file_name").reset_index(drop=True)

display(evaluation_summary_export_plan_df)

print("Evaluation summary export plan assembled.")

,file_name,rows,columns
0,candidate_representation_handoff.csv,15,5
1,representation_runtime_summary.csv,30,7
2,representation_tradeoff_summary.csv,15,12
3,selected_representation_configuration.csv,4,2
4,umap_dimensionality_distance_behavior.csv,20,12
5,umap_dimensionality_neighbor_retention.csv,20,9
6,umap_dimensionality_trustworthiness.csv,20,9
7,umap_workflow_distance_behavior.csv,10,12
8,umap_workflow_neighbor_retention.csv,10,9
9,umap_workflow_trustworthiness.csv,10,9


Evaluation summary export plan assembled.


Findings\. The export plan covers the full representation\-evaluation handoff set in a compact form\. It includes the dimensionality comparison tables, workflow comparison tables, the final representation tradeoff summary, runtime summary, retained\-representation handoff table, and the selected configuration record\. In other words, the notebook is exporting the decision trail as well as the final representation choices\.

This next block writes the summary exports to the final output directory\. It is intentionally simple: one file per summary table, all saved in flat CSV form for easy reuse in later notebooks and reporting workflows\.

In [74]:
# ---------------------------------------------------------------------
# Write evaluation summary exports
# ---------------------------------------------------------------------

evaluation_summary_export_records = []

for file_name, export_df in EVALUATION_SUMMARY_EXPORTS.items():
    output_path = FINAL_OUTPUT_DIR / file_name
    export_df.to_csv(output_path, index=False)

    evaluation_summary_export_records.append(
        {
            "file_name": file_name,
            "output_path": str(output_path),
            "rows": len(export_df),
            "columns": len(export_df.columns),
            "status": "pass" if output_path.exists() else "review",
        }
    )

evaluation_summary_export_df = (
    pd.DataFrame(evaluation_summary_export_records)
    .sort_values("file_name")
    .reset_index(drop=True)
)

display(evaluation_summary_export_df)

assert evaluation_summary_export_df["status"].eq("pass").all(), (
    "One or more evaluation summary exports was not written successfully."
)

print("Evaluation summary exports written.")

,file_name,output_path,rows,columns,status
0,candidate_representation_handoff.csv,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,15,5,pass
1,representation_runtime_summary.csv,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,30,7,pass
2,representation_tradeoff_summary.csv,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,15,12,pass
3,selected_representation_configuration.csv,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,4,2,pass
4,umap_dimensionality_distance_behavior.csv,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,20,12,pass
5,umap_dimensionality_neighbor_retention.csv,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,20,9,pass
6,umap_dimensionality_trustworthiness.csv,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,20,9,pass
7,umap_workflow_distance_behavior.csv,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,10,12,pass
8,umap_workflow_neighbor_retention.csv,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,10,9,pass
9,umap_workflow_trustworthiness.csv,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,10,9,pass


Evaluation summary exports written.


Findings\. All planned evaluation summary files were written successfully to the 3\.2\.4\.final\_tables directory\. The outputs include 10 summary tables ranging from compact configuration metadata to the full 30\-row runtime summary, and every file reports passing write status\. This gives downstream notebooks and final reporting a stable, reusable set of representation\-evaluation artifacts\.

Finally, validate that the written files are present and non\-empty\. This gives the notebook one last lightweight handoff check before closing\.

In [75]:
# ---------------------------------------------------------------------
# Validate evaluation summary exports
# ---------------------------------------------------------------------

evaluation_summary_validation_records = []

for file_name in sorted(EVALUATION_SUMMARY_EXPORTS.keys()):
    output_path = FINAL_OUTPUT_DIR / file_name
    exported_df = pd.read_csv(output_path)

    evaluation_summary_validation_records.append(
        {
            "file_name": file_name,
            "path_exists": output_path.exists(),
            "rows": len(exported_df),
            "columns": len(exported_df.columns),
            "status": "pass" if output_path.exists() and len(exported_df) > 0 else "review",
        }
    )

evaluation_summary_validation_df = (
    pd.DataFrame(evaluation_summary_validation_records)
    .sort_values("file_name")
    .reset_index(drop=True)
)

display(evaluation_summary_validation_df)

assert evaluation_summary_validation_df["status"].eq("pass").all(), (
    "One or more evaluation summary exports failed validation."
)

print("Evaluation summary exports validated.")

,file_name,path_exists,rows,columns,status
0,candidate_representation_handoff.csv,True,15,5,pass
1,representation_runtime_summary.csv,True,30,7,pass
2,representation_tradeoff_summary.csv,True,15,12,pass
3,selected_representation_configuration.csv,True,4,2,pass
4,umap_dimensionality_distance_behavior.csv,True,20,12,pass
5,umap_dimensionality_neighbor_retention.csv,True,20,9,pass
6,umap_dimensionality_trustworthiness.csv,True,20,9,pass
7,umap_workflow_distance_behavior.csv,True,10,12,pass
8,umap_workflow_neighbor_retention.csv,True,10,9,pass
9,umap_workflow_trustworthiness.csv,True,10,9,pass


Evaluation summary exports validated.


Findings\. The written summary exports pass the final handoff check\. Every expected file exists, every file is non\-empty, and all 10 exported tables report passing validation\. That closes the loop on 3\.2\.4: both the selected UMAP assets and the evaluation summaries are now fully materialized for downstream use\.

### 3\.2\.4 Final Output Inventory

The final outputs from 3\.2\.4 are the representation\-evaluation handoff assets\. These files record the selected nonlinear modeling representation, the retained downstream candidate spaces, and the compact evaluation summaries needed by later anomaly and clustering notebooks\.

Full selected UMAP modeling tables\. These are the final nonlinear modeling assets\. Each file contains modeling\_row\_id and the full selected 10D UMAP coordinates only\. Interpretation metadata is loaded separately from the shared 3\.1\.1 row metadata assets\.
\* bus\_umap\_pca\_to\_umap\_10d\.parquet
\* subway\_umap\_pca\_to\_umap\_10d\.parquet
\* taxi\_umap\_pca\_to\_umap\_10d\.parquet
\* fhvhv\_umap\_pca\_to\_umap\_10d\.parquet
\* multimodal\_umap\_pca\_to\_umap\_10d\.parquet

candidate\_representation\_handoff\.csv
Downstream representation inventory\. Records the retained candidate spaces for each feature set: raw\_scaled, pca\_80pct, and umap\_pca\_to\_umap\_10d, along with their roles, dimensions, and load paths\.

selected\_representation\_configuration\.csv
Compact configuration record for the final nonlinear choice\. Documents the selected UMAP workflow, selected UMAP dimensionality, PCA input rule, and the retained downstream representation set\.

representation\_tradeoff\_summary\.csv
Compact comparison table across raw, PCA, and UMAP candidate spaces\. Summarizes the core tradeoff metrics used in the notebook\.

representation\_runtime\_summary\.csv
Runtime\-oriented summary for dimensionality experiments and workflow experiments\. Useful for practical downstream planning and final reporting\.

UMAP dimensionality evaluation summaries
These capture the results used to choose the canonical UMAP modeling dimensionality\.
\* umap\_dimensionality\_trustworthiness\.csv
\* umap\_dimensionality\_neighbor\_retention\.csv
\* umap\_dimensionality\_distance\_behavior\.csv

UMAP workflow evaluation summaries
These capture the results used to choose between raw\_to\_umap and pca\_to\_umap\.
\* umap\_workflow\_trustworthiness\.csv
\* umap\_workflow\_neighbor\_retention\.csv
\* umap\_workflow\_distance\_behavior\.csv

## Close

Summarize the selected UMAP dimensionality, selected UMAP construction workflow, representation\-quality findings, and downstream representation selections\.

Explicitly document that representation learning has been completed and that subsequent notebooks will begin clustering, density estimation, and anomaly\-detection workflows\. Also note that future anomaly notebooks should evaluate both global and temporal\-bucket\-aware scoring approaches, with temporal\-bucket\-aware scoring treated as the preferred default unless evidence suggests otherwise\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>